In [ ]:
# @title ⚙️ STEP 1 — Install Packages & Set API Key { display-mode: "form" }
# @markdown Paste your SerpAPI key below, then run this cell. Only needs to run once per session.

# Key intentionally not stored in the notebook (notebooks get shared/uploaded;
# a live key in source leaks). ONE key for the whole run -- the old per-account
# rotation is gone. serpapi_client.get_key() reads, in order:
#   1. the SERPAPI_KEY environment variable
#   2. serpapi_key.txt next to this notebook (local only, never shared)
import os, sys

_here = globals().get("PROJECT_DIR") or r"C:/Users/SAgno/Documents/AIO_Workspace/LocalRun"
if _here not in sys.path:
    sys.path.insert(0, _here)
import serpapi_client as _serp
SERPAPI_KEY = _serp.get_key()

import os, subprocess, sys

# Dependency install is a no-op locally: the .venv already has everything and
# Playwright's Chromium is installed. Kept as a check so a missing package
# fails here with a clear message instead of deep inside a scrape.
_required = ["requests", "pandas", "trafilatura", "bs4", "tenacity",
             "playwright", "pdfplumber", "openpyxl", "lxml"]
_missing = []
for _p in _required:
    try:
        __import__(_p)
    except ImportError:
        _missing.append(_p)
print("Environment check:")
if _missing:
    print(f"  MISSING: {_missing}")
    print(f"  Install with: pip install {' '.join(_missing)}")
else:
    print(f"  All {len(_required)} packages present.")

if not SERPAPI_KEY:
    print("\n⚠️  WARNING: no SerpAPI key (set SERPAPI_KEY or create serpapi_key.txt).")
else:
    print(f"\n✓ SerpAPI key loaded "
          f"({SERPAPI_KEY[:6]}{'*' * max(0, len(SERPAPI_KEY)-6)}); "
          f"hourly budget {_serp.budget()} searches")

print("\n✅ Step 1 complete.")

In [ ]:
# @title 📁 STEP 2 — Load questions.xlsx (auto from Drive) { display-mode: "form" }
# @markdown Reads `questions.xlsx` straight from your Drive `PROJECT_DIR` — no
# @markdown manual upload needed. If it isn't found there, it falls back to a
# @markdown manual upload prompt. Set the question column, or leave blank to
# @markdown auto-detect the first text column.

PROJECT_DIR     = r"C:/Users/SAgno/Documents/AIO_Workspace/LocalRun"  # @param {type:"string"}
QUESTIONS_FILE  = "questions.xlsx"  # @param {type:"string"}
QUESTION_COLUMN = "question"        # @param {type:"string"}
# interleave_risk: alternate risk levels in proportion (file order kept within
# each level) so that wherever a run stops -- e.g. when the SerpAPI plan runs
# out -- the finished questions are a representative sample of the whole file.
# file: plain row order (the file is sorted High -> Medium -> Low).
# complexity_first (user decision 2026-09-16, mid-run): questions.xlsx is ALSO
# sorted by rubric_complexity inside each risk level (High rows first), so plain
# interleave_risk reaches only High-complexity questions before the SerpAPI plan
# runs out. complexity_first queues every Medium/Low-complexity question first
# (interleaved by risk), then the High ones (interleaved by risk). Already-scraped
# questions are skipped by Step 2.5 whatever the order, so switching is safe.
QUESTION_ORDER  = "complexity_first"  # @param ["complexity_first", "interleave_risk", "file"]

import os, io
import pandas as pd

df_raw = None
xlsx_path = os.path.join(PROJECT_DIR, QUESTIONS_FILE)
if os.path.exists(xlsx_path):
    df_raw = pd.read_excel(xlsx_path)
    print(f"✓ Loaded from Drive: {xlsx_path}")
else:
    print(f"⚠️  '{xlsx_path}' not found. Put questions.xlsx in PROJECT_DIR.")

if df_raw is None:
    print("⚠️  No questions file loaded.")
    QUESTIONS = []
else:
    print(f"  Columns: {list(df_raw.columns)}")
    print(f"  Rows:    {len(df_raw)}")

    col = QUESTION_COLUMN.strip() if QUESTION_COLUMN.strip() else None
    # Match the column case-insensitively: the 408-question file said
    # "Question", the 3,852-question file says "question".
    if col and col not in df_raw.columns:
        _ci = {str(c).strip().lower(): c for c in df_raw.columns}
        if col.lower() in _ci:
            col = _ci[col.lower()]

    # Optional inclusion flag ("Include? Y/N"): keep only rows marked Y.
    _inc_col = next((c for c in df_raw.columns
                     if str(c).strip().lower().startswith("include")), None)
    if _inc_col is not None:
        _keep = df_raw[_inc_col].astype(str).str.strip().str.upper().str.startswith("Y")
        print(f"  Include column '{_inc_col}': keeping {int(_keep.sum())} of {len(df_raw)} rows")
        df_raw = df_raw[_keep].reset_index(drop=True)

    if col:
        if col not in df_raw.columns:
            print(f"\n⚠️  Column '{col}' not found. Available: {list(df_raw.columns)}")
            QUESTIONS = []
        else:
            QUESTIONS = df_raw[col].dropna().astype(str).tolist()
    else:
        QUESTIONS = []
        for c in df_raw.columns:
            sample = df_raw[c].dropna().astype(str)
            if sample.str.len().mean() > 15:
                col = c
                QUESTIONS = sample.tolist()
                print(f"  Auto-detected question column: '{col}'")
                break
        if not QUESTIONS:
            print("⚠️  Could not auto-detect column. Set QUESTION_COLUMN manually.")

    # ── Per-question metadata ────────────────────────────────────────────────
    # These ride alongside the question text through the whole pipeline:
    # research_output.csv -> aio/organic_dataset.csv -> comparison workbook -> R.
    QUESTION_META_COLS = ["risk_level", "rubric_complexity", "requires_hedging"]
    QUESTION_META = {}
    if QUESTIONS and col:
        _present = [c for c in QUESTION_META_COLS if c in df_raw.columns]
        _absent  = [c for c in QUESTION_META_COLS if c not in df_raw.columns]
        for _, _r in df_raw.dropna(subset=[col]).iterrows():
            QUESTION_META[str(_r[col])] = {
                c: ("" if pd.isna(_r[c]) else _r[c]) for c in _present}
        print(f"\n  Metadata columns found: {_present if _present else '(none)'}")
        if _absent:
            print(f"  ⚠️  Not in {QUESTIONS_FILE}: {_absent} — blank downstream.")
        for c in _present:
            _counts = df_raw[c].value_counts(dropna=False).to_dict()
            print(f"    {c}: {_counts}")
        _dupes = len(QUESTIONS) - len(QUESTION_META)
        if _dupes > 0:
            print(f"  ⚠️  {_dupes} duplicate question text(s) — metadata keeps the LAST of each.")

    # ── Run order ────────────────────────────────────────────────────────────
    if QUESTIONS and QUESTION_ORDER in ("interleave_risk", "complexity_first") \
            and "risk_level" in df_raw.columns:
        # Proportional interleave: each question gets the fractional position
        # (i + 0.5) / n_level within its own level; sorting on that spreads every
        # level evenly through the run while keeping file order inside a level.
        _order_rank = {"High": 0, "Medium": 1, "Low": 2}
        _src = df_raw.dropna(subset=[col]).copy()
        if QUESTION_ORDER == "complexity_first" and "rubric_complexity" in _src.columns:
            _src["_tier"] = (_src["rubric_complexity"].astype(str).str.strip() == "High").astype(int)
        else:
            _src["_tier"] = 0
        _pos = {}
        for (_tier, _lvl), _grp in _src.groupby(["_tier", "risk_level"], sort=False):
            _qs = _grp[col].astype(str).tolist()
            for _i, _q in enumerate(_qs):
                _pos[_q] = (int(_tier), (_i + 0.5) / len(_qs), _order_rank.get(str(_lvl), 9))
        QUESTIONS = sorted(QUESTIONS, key=lambda q: _pos.get(q, (9, 2.0, 9)))
        print(f"\n  Run order: {QUESTION_ORDER} (proportional interleave by risk_level"
              f"{' within complexity tier' if QUESTION_ORDER == 'complexity_first' else ''}).")
        for _n in (100, 1000, 2270):
            if len(QUESTIONS) >= _n:
                _mix = pd.Series([f"{QUESTION_META.get(q, {}).get('risk_level', '')}/"
                                  f"{QUESTION_META.get(q, {}).get('rubric_complexity', '')}"
                                  for q in QUESTIONS[:_n]]).value_counts().to_dict()
                print(f"    first {_n}: {_mix}")
    elif QUESTIONS:
        print(f"\n  Run order: file order.")

    if QUESTIONS:
        print(f"\n  Total questions: {len(QUESTIONS)}")
        print("  Preview (first 5):")
        for i, q in enumerate(QUESTIONS[:5]):
            print(f"    [{i}] {q[:160]}{'…' if len(q) > 160 else ''}")
        print("\n✅ Step 2 complete.")


In [ ]:
# @title 🧮 STEP 2.5 — Batch Plan (per-question resume) { display-mode: "form" }
# @markdown Picks the next BATCH_SIZE questions that still need scraping, by
# @markdown reading every `research_output_batchNN.csv` already on disk.
# @markdown Re-run it (after Step 2) every time you come back -- nothing to
# @markdown track by hand.
# @markdown
# @markdown Resume is PER QUESTION, not per batch number. The old grid mapped
# @markdown batch N to questions (N-1)*BATCH_SIZE.., so changing BATCH_SIZE
# @markdown re-pointed every existing file at the wrong questions, and one
# @markdown permanently failing question kept a whole batch "incomplete"
# @markdown forever. Now:
# @markdown - a question is DONE once any batch file holds a row for it that is
# @markdown   not a retryable SerpAPI failure (a real "no_aio" counts as done);
# @markdown - retryable failures (quota, rate limit, network, crashed question)
# @markdown   are re-queued; quota failures never count as an attempt, others
# @markdown   give up after MAX_ATTEMPTS;
# @markdown - each run writes the NEXT unused batch number, so BATCH_SIZE can
# @markdown   change freely between runs.

BATCH_SIZE   = 50  # @param {type:"slider", min:10, max:200, step:10}
MAX_ATTEMPTS = 2   # @param {type:"integer"}  -> one retry, then give up

import glob, json, math, re, os
import pandas as pd

# Errors that mean "no usable data -- try again later".
RETRYABLE_ERROR_RE = (r"serpapi_error|question_exception|run out of searches|"
                      r"429|rate limit|hourly|exceeded|invalid api key|unauthor|"
                      r"aio_prefetch_failed")
# Subset that is about the ACCOUNT, not the question: never counts as an attempt.
QUOTA_ERROR_RE = (r"serpapi quota|run out of searches|invalid api key|unauthor|"
                  r"hourly|rate limit|429|skipped_quota")

_n_total = len(QUESTIONS) if 'QUESTIONS' in dir() else 0


def _qkey(q):
    """The question text exactly as Step 8 writes it (_csv_clean): newlines and
    tabs to spaces, double quotes to single, runs of spaces collapsed. Batch
    files hold THIS form, so comparing raw text would never match a question
    containing a newline or a quote and would re-scrape it every batch."""
    q = re.sub(r"[\r\n]+", " ", str(q)).replace("\t", " ").replace('"', "'")
    return re.sub(r"  +", " ", q).strip()

if _n_total == 0:
    print("⚠️  Run Step 2 first to load questions.")
    CURRENT_BATCH_NUM = None
    CURRENT_BATCH_QUESTIONS = []
    _n_batches = 0
else:
    _done, _attempts = set(), {}
    _batch_nums = []
    for _f in glob.glob(os.path.join(PROJECT_DIR, "research_output_batch*.csv")):
        _m = re.search(r"research_output_batch(\d+)\.csv$", _f)
        if not _m:
            continue
        _batch_nums.append(int(_m.group(1)))
        try:
            _d = pd.read_csv(_f, encoding="utf-8-sig", low_memory=False,
                             usecols=lambda c: c in ("question", "error"))
        except Exception as _e:
            print(f"  ⚠️  could not read {os.path.basename(_f)}: {_e}")
            continue
        if "error" not in _d.columns:
            _d["error"] = ""
        _err = _d["error"].fillna("").astype(str)
        _retry = _err.str.contains(RETRYABLE_ERROR_RE, case=False, regex=True)
        _quota = _err.str.contains(QUOTA_ERROR_RE, case=False, regex=True)
        for _q, _r, _qu in zip(_d["question"].map(_qkey), _retry, _quota):
            if not _r:
                _done.add(_q)
            elif not _qu:
                _attempts[_q] = _attempts.get(_q, 0) + 1

    _gave_up = [q for q in QUESTIONS
                if _qkey(q) not in _done and _attempts.get(_qkey(q), 0) >= MAX_ATTEMPTS]
    _gave_up_set = set(_gave_up)
    _pending = [q for q in QUESTIONS if _qkey(q) not in _done and q not in _gave_up_set]
    _n_done = sum(1 for q in QUESTIONS if _qkey(q) in _done)

    print(f"{_n_total} questions: {_n_done} done, {len(_pending)} pending, "
          f"{len(_gave_up)} gave up after {MAX_ATTEMPTS} failed attempts.")
    print(f"Batch files on disk: {len(_batch_nums)}")

    _n_batches = len(_batch_nums) + math.ceil(len(_pending) / BATCH_SIZE)

    try:  # machine-readable snapshot for progress.py / status.py
        os.makedirs(os.path.join(PROJECT_DIR, "logs"), exist_ok=True)
        with open(os.path.join(PROJECT_DIR, "logs", "scrape_plan.json"), "w",
                  encoding="utf-8") as _fh:
            json.dump({"total": _n_total, "done": _n_done, "pending": len(_pending),
                       "gave_up": len(_gave_up), "batch_files": len(_batch_nums),
                       "batch_size": BATCH_SIZE}, _fh)
    except Exception:
        pass

    if not _pending:
        print(f"\n🎉 Nothing left to scrape.")
        print("   Proceed to Step 8.5 (Merge Batch Files), then Step 9 as usual.")
        CURRENT_BATCH_NUM = None
        CURRENT_BATCH_QUESTIONS = []
    else:
        CURRENT_BATCH_NUM = (max(_batch_nums) + 1) if _batch_nums else 1
        CURRENT_BATCH_QUESTIONS = _pending[:BATCH_SIZE]
        print(f"\n▶ This run will process BATCH {CURRENT_BATCH_NUM}: "
              f"{len(CURRENT_BATCH_QUESTIONS)} question(s) "
              f"({sum(1 for q in CURRENT_BATCH_QUESTIONS if _attempts.get(_qkey(q)))} retries).")
        print("  Run Steps 3-7 (if not already defined this session), then Step 8.")


In [ ]:
# @title 🔬 STEP 3 — Test One SerpAPI Call & Inspect Output { display-mode: "form" }
# @markdown Runs SerpAPI on ONE question so you can inspect the raw structure.
# @markdown Change the index to test a different question.

TEST_INDEX = 0  # @param {type:"integer"}

import json
import re
import threading as _threading
import time as _time_mod
import requests as _requests
from serpapi import GoogleSearch

# Single-key SerpAPI plumbing: hourly rate limiter + error classification.
# (Step 1 already imported it; re-import so this cell also works on its own.)
import sys as _sys
_here = globals().get("PROJECT_DIR") or r"C:/Users/SAgno/Documents/AIO_Workspace/LocalRun"
if _here not in _sys.path:
    _sys.path.insert(0, _here)
import serpapi_client as _serp

# ── URL type helpers — used by Steps 3, 4, 7, and 8 ─────────────────────────

VIDEO_DOMAINS = [
    "youtube.com", "youtu.be", "vimeo.com",
    "dailymotion.com", "tiktok.com", "instagram.com/reel",
]

def is_video_url(url: str) -> bool:
    return any(d in url for d in VIDEO_DOMAINS)

# ── Google citation redirect wrappers ───────────────────────────────────────
# Google serves a share of AI Overview citations (and the discussions/forums
# link) as https://www.google.com/goto?url=<blob> click-tracking redirects
# rather than the publisher URL. The blob is an opaque protobuf-ish token, NOT
# a percent-encoded URL, so the destination cannot be parsed out of the query
# string -- it has to be resolved over HTTP.
#
# Leaving them unresolved corrupts everything keyed on the URL: the domain
# reads as "google.com", HON_Authority scores the wrong publisher, and
# AIO-vs-organic URL overlap misses genuine matches. On the 2026-08-26 run this
# hit 428 of 2,387 AIO sources (17.9%).
#
# Measured on that corpus: HEAD never resolves these (Google does not emit
# Location for a HEAD), so go straight to a streamed GET -- stream=True reads
# the final URL off the connection without pulling the body, and a destination
# that answers 403 still yields its URL, which is all we need.

from urllib.parse import urlparse as _urlparse, parse_qs as _parse_qs, unquote as _unquote

_GOOGLE_WRAPPER_HOSTS = {"google.com", "www.google.com"}
_GOOGLE_WRAPPER_PATHS = ("/goto", "/url")
# /searchviewer is Google's in-SERP reader, not a redirect to an external page.
# It has no publisher destination, so it is detected but never resolved.
_GOOGLE_VIEWER_PATHS  = ("/searchviewer",)

_redirect_cache = {}
_redirect_lock = _threading.Lock()
_redirect_tls = _threading.local()
# Set when Google answers a /goto with its /sorry/ bot check. Every resolver
# thread waits it out instead of hammering Google into a longer block. At
# 3,800+ questions this is thousands of /goto GETs, far more than the 30-URL
# validation sample ever exercised.
_redirect_block_until = [0.0]


def _is_google_host(url: str) -> bool:
    try:
        host = _urlparse(url or "").netloc.lower()
    except Exception:
        return False
    host = host.split(":")[0]
    return host == "google.com" or host.endswith(".google.com")


def _redirect_session():
    s = getattr(_redirect_tls, "session", None)
    if s is None:
        s = _requests.Session()
        _redirect_tls.session = s
    return s


def is_google_redirect(url: str) -> bool:
    """True for a Google click-tracking wrapper we can try to resolve."""
    try:
        p = _urlparse(url or "")
    except Exception:
        return False
    return (p.netloc.lower() in _GOOGLE_WRAPPER_HOSTS
            and p.path.startswith(_GOOGLE_WRAPPER_PATHS))


def is_google_viewer(url: str) -> bool:
    """True for /searchviewer -- a wrapper with no external destination."""
    try:
        p = _urlparse(url or "")
    except Exception:
        return False
    return (p.netloc.lower() in _GOOGLE_WRAPPER_HOSTS
            and p.path.startswith(_GOOGLE_VIEWER_PATHS))


def resolve_google_redirect(url: str, timeout: int = 15) -> str:
    """Resolve a Google redirect wrapper to the publisher URL.

    Returns the input unchanged when the URL is not a wrapper, or when
    resolution fails -- never raises, so a dead redirect degrades to the old
    behaviour instead of losing the citation entirely.
    """
    if not url or not is_google_redirect(url):
        return url
    if url in _redirect_cache:
        return _redirect_cache[url]

    resolved = url
    # Cheap path first: /url?q=... carries a plain URL.
    try:
        q = _parse_qs(_urlparse(url).query)
        for key in ("q", "url"):
            for v in q.get(key, []):
                v = _unquote(v)
                if v.startswith(("http://", "https://")):
                    resolved = v
                    break
            if resolved != url:
                break
    except Exception:
        pass

    if resolved == url:
        # get_headers() lives in Step 5. This function is only ever CALLED
        # after Step 5 has run, but Step 3's own preview block can fire
        # earlier in a straight top-to-bottom interactive run, so fall back
        # to a plain UA rather than dying on a NameError.
        _hdrs = get_headers() if "get_headers" in globals() else {
            "User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) "
                           "Chrome/124.0 Safari/537.36")}
        for _try in range(3):
            _wait = _redirect_block_until[0] - _time_mod.time()
            if _wait > 0:
                _time_mod.sleep(_wait)
            try:
                with _redirect_session().get(url, headers=_hdrs, timeout=timeout,
                                             allow_redirects=True, stream=True) as r:
                    final = r.url or ""
                if final and not _is_google_host(final):
                    resolved = final
                    break
                # Landed on Google itself: /sorry/ bot check, consent page, or
                # the wrapper again. Never record a google.com URL as the
                # publisher -- that is the exact bug this code exists to fix.
                if "/sorry" in final or r.status_code == 429:
                    with _redirect_lock:
                        _redirect_block_until[0] = max(
                            _redirect_block_until[0],
                            _time_mod.time() + 60 * (2 ** _try))
                    print(f"    ⚠ Google bot-check on redirect resolution — "
                          f"backing off {60 * (2 ** _try)}s")
                    continue
                break
            except Exception as e:
                if _try == 2:
                    print(f"    ⚠ Could not resolve Google redirect ({type(e).__name__}): "
                          f"{url[:70]}")
                _time_mod.sleep(2 * (_try + 1))

    # Only cache successes: a failure caused by a temporary block should get
    # another chance when the same wrapper shows up for a later question.
    if resolved != url:
        with _redirect_lock:
            _redirect_cache[url] = resolved
    return resolved


def _resolve_raw_result_in_place(result):
    """Resolve Google /goto wrappers in the RAW SerpAPI response, BEFORE parsing.

    Resolving only after parsing (the 2026-08-26 fix) was too late: the video
    filter and the statement -> citation index map (citation_indexes = 1-based
    position in the video-filtered source list) had already been computed on
    the wrapper URLs. A wrapper that resolves to YouTube / an Instagram reel is
    then skipped by Step 8's source loop, every later source shifts down one
    aio_src slot, and statements point at the wrong citation. Rewriting the
    raw links first means filtering, de-duplication and indexing all see the
    real publisher URLs.

    Only the parts of the response the pipeline reads are walked: the AI
    Overview subtree, organic_results, and the first discussions_and_forums
    entry.
    """
    if not isinstance(result, dict):
        return result
    targets = []
    if isinstance(result.get("ai_overview"), dict):
        targets.append(result["ai_overview"])
    targets.extend(r for r in (result.get("organic_results") or []) if isinstance(r, dict))
    _disc = result.get("discussions_and_forums") or []
    if _disc and isinstance(_disc[0], dict):
        targets.append(_disc[0])

    slots = []          # (dict, key) holding a wrapper URL

    def _walk(node):
        if isinstance(node, dict):
            for k, v in node.items():
                if k in ("link", "url") and isinstance(v, str) and is_google_redirect(v):
                    slots.append((node, k))
                elif isinstance(v, (dict, list)):
                    _walk(v)
        elif isinstance(node, list):
            for v in node:
                _walk(v)

    for t in targets:
        _walk(t)
    wrappers = list(dict.fromkeys(d[k] for d, k in slots))
    resolved = {}
    if wrappers:
        from concurrent.futures import ThreadPoolExecutor as _TPE
        with _TPE(max_workers=min(4, len(wrappers))) as _ex:
            resolved = dict(zip(wrappers, _ex.map(resolve_google_redirect, wrappers)))
        for d, k in slots:
            d[k] = resolved.get(d[k], d[k])
    n_unres = sum(1 for u, r in resolved.items() if r == u)
    result["_google_redirects"] = {"resolved": len(wrappers) - n_unres, "unresolved": n_unres}
    if wrappers:
        print(f"  ↪ Resolved {len(wrappers) - n_unres}/{len(wrappers)} Google redirect "
              f"wrapper(s) before parsing")
    return result


def _resolve_urls_in_place(output):
    """Rewrite every captured URL that is a Google redirect wrapper.

    Applied once, at the end of parse_serpapi_result, so EVERY downstream
    consumer -- scraping, domain extraction, JAMA/HON scoring, AIO-vs-organic
    overlap -- sees the real publisher URL.
    """
    # Resolve every distinct wrapper once, a few at a time, then rewrite.
    _wrappers = set()
    for s in output.get("aio_sources") or []:
        if isinstance(s, dict):
            for key in ("link", "url"):
                if s.get(key) and is_google_redirect(s[key]):
                    _wrappers.add(s[key])
    for st in output.get("aio_statements") or []:
        if isinstance(st, dict) and st.get("source_url") and is_google_redirect(st["source_url"]):
            _wrappers.add(st["source_url"])
    for u in output.get("organic_urls") or []:
        if u and is_google_redirect(u):
            _wrappers.add(u)
    if output.get("forum_url") and is_google_redirect(output["forum_url"]):
        _wrappers.add(output["forum_url"])
    _resolved = {}
    if _wrappers:
        from concurrent.futures import ThreadPoolExecutor as _TPE
        with _TPE(max_workers=min(4, len(_wrappers))) as _ex:
            for u, r in zip(list(_wrappers), _ex.map(resolve_google_redirect, list(_wrappers))):
                _resolved[u] = r

    n = 0
    for s in output.get("aio_sources") or []:
        if not isinstance(s, dict):
            continue
        for key in ("link", "url"):
            u = s.get(key)
            if u and is_google_redirect(u):
                r = _resolved.get(u, u)
                if r != u:
                    s[key] = r
                    n += 1

    # Legacy per-statement source URLs (aio_statement_N_source_url columns).
    for st in output.get("aio_statements") or []:
        if isinstance(st, dict):
            u = st.get("source_url")
            if u and is_google_redirect(u) and _resolved.get(u, u) != u:
                st["source_url"] = _resolved[u]

    # snippet_by_url is keyed on the URL for AIO sources too -- move those keys.
    snips = output.get("snippet_by_url")
    if isinstance(snips, dict):
        for u in [k for k in snips if is_google_redirect(k)]:
            r = _resolved.get(u, u)
            if r != u:
                if not snips.get(r):
                    snips[r] = snips[u]
                snips.pop(u, None)

    org = output.get("organic_urls") or []
    for i, u in enumerate(org):
        if u and is_google_redirect(u):
            r = _resolved.get(u, u)
            if r != u:
                # snippet_by_url is keyed on the URL -- move the key with it.
                snips = output.get("snippet_by_url")
                if isinstance(snips, dict) and u in snips:
                    snips[r] = snips.pop(u)
                org[i] = r
                n += 1

    fu = output.get("forum_url")
    if fu and is_google_redirect(fu):
        r = _resolved.get(fu, fu)
        if r != fu:
            output["forum_url"] = r
            n += 1

    _unresolved = sorted(u for u, r in _resolved.items() if r == u)
    # Nearly always zero here: the raw response was already resolved before
    # parsing. Anything left is a wrapper that failed both passes.
    _raw = output.pop("_raw_redirects", {}) or {}
    output["google_redirects_resolved"] = _raw.get("resolved", 0) + len(_wrappers) - len(_unresolved)
    output["google_redirects_unresolved"] = len(_unresolved)
    if n:
        print(f"  ↪ Resolved {n} Google redirect wrapper(s) to publisher URLs")
    if _unresolved:
        print(f"  ⚠ {len(_unresolved)} Google redirect wrapper(s) could NOT be resolved")
    return output


# ── SerpAPI search — resolves AIO page_token immediately ─────────────────────

def search_google(query, api_key, max_retries=3):
    params = {
        "q":             query,
        "api_key":       api_key,
        "engine":        "google",
        "num":           10,
        "hl":            "en",
        "gl":            "us",
        "google_domain": "google.com",
    }
    import time as _time
    if _serp.quota_exhausted():
        raise _serp.SerpApiQuotaError("serpapi quota: skipped_quota (key already out of searches)")

    # SerpAPI reports failures as a JSON body with an "error" key and the
    # client library does NOT raise on it. Unchecked, "run out of searches" or
    # the hourly limit parsed as an empty SERP -> recorded as no_aio -> counted
    # as a finished question. Every error is classified here instead.
    last_exc = None
    result = None
    _attempt = 0
    _rate_waits = 0
    while _attempt < max_retries:
        _serp.acquire(reserve=2)   # this search + its page_token follow-up
        try:
            result = GoogleSearch(params).get_dict()
        except Exception as _e:
            _attempt += 1
            last_exc = _e
            print(f"  ⚠ SerpAPI attempt {_attempt}/{max_retries} failed: {_e}")
            if _attempt < max_retries:
                _time.sleep(2 ** _attempt)
            continue
        _err = result.get("error") if isinstance(result, dict) else "non-dict response"
        if not _err:
            break
        _kind = _serp.classify_error(_err)
        if _kind == "no_results":
            break                        # Google genuinely has nothing: a real no_aio
        if _kind == "quota":
            _serp.mark_quota_exhausted()
            raise _serp.SerpApiQuotaError(f"serpapi quota: {_err}")
        if _kind == "rate_limited" and _rate_waits < 6:
            _rate_waits += 1
            _serp.wait_after_rate_limit(_rate_waits)
            continue
        _attempt += 1
        last_exc = RuntimeError(f"serpapi returned error: {_err}")
        print(f"  ⚠ SerpAPI attempt {_attempt}/{max_retries} returned error: {_err}")
        if _attempt < max_retries:
            _time.sleep(2 ** _attempt)
    else:
        raise last_exc or RuntimeError("serpapi search failed")

    aio = result.get("ai_overview", {})
    if aio.get("page_token") and aio.get("serpapi_link"):
        print("  ℹ AIO page_token detected — pre-fetching full content...")
        sep      = "&" if "?" in aio["serpapi_link"] else "?"
        full_url = f"{aio['serpapi_link']}{sep}api_key={api_key}"
        aio_data, _last, _token_empty = None, "", False
        for _p in range(4):
            if _p == 0:
                _serp.record_reserved()
            else:
                _serp.acquire(reserve=1)
            try:
                resp = _requests.get(full_url, timeout=60)
                try:
                    _body = resp.json()
                except ValueError:
                    _body = {"error": f"HTTP {resp.status_code} non-JSON body"}
                _err = _body.get("error") if isinstance(_body, dict) else None
                if resp.status_code == 429 and not _err:
                    _err = "HTTP 429 too many requests"
                if not _err and resp.ok:
                    aio_data = _body
                    break
                _last = _err or f"HTTP {resp.status_code}"
                _kind = _serp.classify_error(_last)
                if _kind == "no_results":
                    # Google showed an AIO placeholder but its content comes back
                    # empty. Deterministic -- the same question failed identically
                    # on retry -- so record it as final instead of re-spending
                    # ~2 searches per attempt.
                    _token_empty = True
                    break
                if _kind == "quota":
                    _serp.mark_quota_exhausted()
                    raise _serp.SerpApiQuotaError(f"serpapi quota: {_last}")
                if _kind == "rate_limited":
                    _serp.wait_after_rate_limit(_p + 1)
                    continue
                print(f"  ⚠ AIO pre-fetch attempt {_p + 1} failed: {_last}")
                break   # an expired/invalid page_token will not fix itself
            except _serp.SerpApiQuotaError:
                raise
            except Exception as e:
                _last = f"{type(e).__name__}: {e}"
                print(f"  ⚠ AIO pre-fetch attempt {_p + 1} failed: {_last}")
                _time.sleep(2 * (_p + 1))

        if _token_empty:
            print("  ⚠ AIO page_token returned no content — recording aio_token_empty")
            result["ai_overview"].pop("page_token", None)
            result["ai_overview"].pop("serpapi_link", None)
            result["_aio_token_empty"] = True
            return result

        if aio_data is None:
            # Without the follow-up the overview is a stub; recording it would
            # look like "no AI Overview". Fail the question so Step 2.5 retries
            # it with a fresh search (and a fresh page_token).
            raise RuntimeError(f"aio_prefetch_failed: {_last}")

        text_blocks = (
            aio_data.get("text_blocks") or
            aio_data.get("ai_overview", {}).get("text_blocks", [])
        )
        if text_blocks:
            result["ai_overview"]["text_blocks"] = text_blocks
            print(f"  ✓ AIO text_blocks injected ({len(text_blocks)} blocks)")

        references = (
            aio_data.get("references") or
            aio_data.get("ai_overview", {}).get("references", [])
        )
        if references and not aio.get("references"):
            result["ai_overview"]["references"] = references

    return result

# ── AIO text block parser ─────────────────────────────────────────────────────

# Citation-chip artifacts leaked into AIO text, e.g. "Mayo Clinic +4",
# "www.heart.org +1", "(.gov) +3" — Google's "source pill" ("Mayo Clinic and
# 4 more") rendered as plain text.
_AIO_CHIP_NAME_RE = re.compile(
    r"\b[A-Z][\w&.'-]*(?:\s+(?:[A-Z&][\w&.'-]*|of|for|and|the)){0,5}\s*\+\d+\b\s*"
)
# Domain-style / parenthesized-TLD chips, and bare "+N" remnants
_AIO_CHIP_DOMAIN_RE = re.compile(
    r"(?:\(\.?[\w.-]{2,30}\)|(?:www\.)?[\w-]+(?:\.[a-z]{2,6})+)?\s*\+\d+\b\s*"
)

def _clean_aio_artifacts(text):
    """Strip citation-chip fragments like 'Mayo Clinic +4' from AIO text."""
    if not text:
        return text
    cleaned = _AIO_CHIP_NAME_RE.sub(" ", text)
    cleaned = _AIO_CHIP_DOMAIN_RE.sub(" ", cleaned)
    return re.sub(r"  +", " ", cleaned).strip()

def _parse_text_blocks(text_blocks):
    """
    Assembles SerpAPI text_blocks into a clean readable string.
    Handles paragraph, heading, list, and expandable block types.
    """
    parts = []
    for block in text_blocks:
        btype = block.get("type", "")

        if btype == "heading":
            snippet = block.get("snippet", "")
            if snippet:
                parts.append(f"\n{snippet}\n")

        elif btype in ("paragraph", "expandable"):
            snippet = block.get("snippet", "")
            if snippet:
                parts.append(snippet)
            nested = block.get("text_blocks", [])
            if nested:
                parts.append(_parse_text_blocks(nested))

        elif btype == "list":
            for item in block.get("list", []):
                sub = item.get("snippet", "")
                if sub:
                    parts.append(f"• {sub}")

    return _clean_aio_artifacts(" ".join(p.strip() for p in parts if p.strip()))

# ── NEW: AIO statement + source extractor ─────────────────────────────────────

def _parse_text_blocks_with_sources(text_blocks, all_sources):
    """
    Extract individual statements from AIO text_blocks that carry source references.
    Returns a list of dicts:
        {"statement": <str>, "source_title": <str>, "source_url": <str>}
    One entry per (statement, source) pair — a statement cited by two sources
    produces two entries.

    Handles:
      • references[].source_index  → look up in all_sources list
      • references[].link / .title → direct inline reference data
    """
    results = []

    def _lookup_source(ref, all_sources):
        """Resolve a single reference dict to (title, url)."""
        # Case A: numeric source_index into the main sources array
        src_idx = ref.get("source_index")
        if src_idx is not None and all_sources:
            try:
                src = all_sources[int(src_idx)]
                return (
                    src.get("title", "") or src.get("name", ""),
                    src.get("link", "") or src.get("url", ""),
                )
            except (IndexError, ValueError, TypeError):
                pass
        # Case B: inline link / title directly on the reference object
        if ref.get("link") or ref.get("url"):
            return (
                ref.get("title", "") or ref.get("name", ""),
                ref.get("link", "") or ref.get("url", ""),
            )
        return None

    def _process_snippet(snippet, refs):
        """Add (statement, source) entries for a snippet+refs pair."""
        if not snippet or not refs:
            return
        added = 0
        for ref in refs:
            resolved = _lookup_source(ref, all_sources)
            if resolved:
                title, url = resolved
                if url and not is_video_url(url):
                    results.append({
                        "statement":    snippet.strip(),
                        "source_title": title,
                        "source_url":   url,
                    })
                    added += 1
        # If the block has refs but none resolved to a URL, still record it
        # with blank source so the statement isn't silently dropped
        if not added:
            results.append({
                "statement":    snippet.strip(),
                "source_title": "",
                "source_url":   "",
            })

    for block in text_blocks:
        btype = block.get("type", "")
        refs  = block.get("references", [])

        if btype == "heading":
            continue  # headings are structural, not cited statements

        elif btype in ("paragraph", "expandable"):
            snippet = block.get("snippet", "")
            if snippet and refs:
                _process_snippet(snippet, refs)
            # Recurse into nested text_blocks (expandable sections)
            nested = block.get("text_blocks", [])
            if nested:
                results.extend(_parse_text_blocks_with_sources(nested, all_sources))

        elif btype == "list":
            for item in block.get("list", []):
                sub      = item.get("snippet", "")
                item_refs = item.get("references", [])
                if sub and item_refs:
                    _process_snippet(sub, item_refs)

    return results


# ── NEW: Grouped statement → citation-index mapping ───────────────────────────
# _parse_text_blocks_with_sources emits one (statement, source) entry per source,
# duplicating the statement. Grouped variant returns one entry per statement
# with citation_indexes pointing to positions in aio_sources (matching aio_src1..N).

def _format_citation_range(idxs):
    """Format [1,2,3,4] -> '1:4'; [1,3,5] -> '1,3,5'; [1,2,4,5] -> '1:2,4:5'."""
    if not idxs:
        return ""
    idxs = sorted(set(int(x) for x in idxs))
    groups, start, prev = [], idxs[0], idxs[0]
    for x in idxs[1:]:
        if x == prev + 1:
            prev = x
        else:
            groups.append(f"{start}:{prev}" if start != prev else str(start))
            start = prev = x
    groups.append(f"{start}:{prev}" if start != prev else str(start))
    return ",".join(groups)

def _parse_text_blocks_grouped(text_blocks, all_sources):
    """Return [{statement, citation_indexes:[int,...]}, ...] — ONE entry per
    block unit (paragraph / list item) in document order, INCLUDING uncited
    units (citation_indexes=[]). citation_indexes are 1-based positions in the
    final aio_sources list (matching aio_src1, aio_src2, ... columns). Filters
    out video sources to stay aligned with aio_sources construction."""
    # Build URL -> 1-based filtered-index lookup. Filter step MUST match aio_sources.
    filtered = [s for s in all_sources
                if s.get("link") and not is_video_url(s.get("link", ""))]
    url_to_idx = {}
    for i, s in enumerate(filtered, start=1):
        u = s.get("link") or s.get("url") or ""
        if u:
            url_to_idx[u] = i

    grouped = []

    def _resolve_ref_url(ref):
        si = ref.get("source_index")
        if si is not None:
            try:
                idx = int(si)
                if 0 <= idx < len(all_sources):
                    s = all_sources[idx]
                    return s.get("link") or s.get("url") or ""
            except Exception:
                pass
        return ref.get("link", "")

    def _collect_block_refs(block):
        """R5 - defensive: handle multiple SerpAPI shapes for inline references."""
        out = []
        for ref in block.get("references") or []:
            out.append(ref)
        # Alternate shapes: arrays of source indices
        for idx in block.get("reference_indexes") or []:
            out.append({"source_index": idx})
        for idx in block.get("source_indexes") or []:
            out.append({"source_index": idx})
        # Alternate shape: singular source_index on the block itself
        si = block.get("source_index")
        if si is not None:
            out.append({"source_index": si})
        return out

    def _refs_to_idxs(refs):
        idxs = []
        for ref in refs:
            url = _resolve_ref_url(ref)
            if url and url in url_to_idx:
                idxs.append(url_to_idx[url])
        return sorted(set(idxs))

    def _emit(text, idxs):
        text = _clean_aio_artifacts(text.strip())
        if text:
            grouped.append({"statement": text,
                            "citation_indexes": sorted(set(idxs))})

    # v3 — emit statements at GOOGLE'S RENDERED granularity, cited or not:
    #   • paragraph                     -> one statement (its own refs)
    #   • heading + list (section)     -> when the citations sit on the LIST
    #     block (Google's usual shape: "2. Move Your Body" heading, then a
    #     list with reference_indexes), the WHOLE section — heading + all
    #     bullets — is ONE statement carrying the section's citation group.
    #   • list whose ITEMS carry refs  -> each item is its own statement.
    # Units with no references anywhere are emitted with citation_indexes=[]
    # (uncited) so the AIO is fully accounted for.
    def _item_text(item):
        title    = (item.get("title") or "").strip()
        sub_snip = (item.get("snippet") or "").strip()
        if title and sub_snip and not sub_snip.lower().startswith(title.lower()[:20]):
            return f"{title}: {sub_snip}"
        return sub_snip or title

    def _walk(blocks, inherited=()):
        pending_heading = ""
        for b in blocks:
            btype = b.get("type", "")
            if btype == "heading":
                pending_heading = (b.get("snippet") or "").strip()
                continue
            own = _refs_to_idxs(_collect_block_refs(b))
            eff = own or list(inherited)

            snippet = (b.get("snippet") or "").strip()
            items = b.get("list", []) or []

            if items:
                item_data = [(_item_text(it),
                              _refs_to_idxs(_collect_block_refs(it)), it)
                             for it in items]
                if any(idxs for _, idxs, _ in item_data):
                    # Items are individually cited -> per-item statements
                    for text, idxs, it in item_data:
                        if text:
                            _emit(text, idxs or eff)
                        _walk(it.get("text_blocks", []) or [], idxs or eff)
                else:
                    # Section-cited (refs on the list block) or uncited:
                    # heading + bullets = ONE statement
                    joined = " • ".join(t for t, _, _ in item_data if t)
                    section_text = (f"{pending_heading}: {joined}"
                                    if pending_heading else joined)
                    if section_text:
                        _emit(section_text, eff)
                    for _, _, it in item_data:
                        _walk(it.get("text_blocks", []) or [], eff)
                pending_heading = ""
            elif snippet:
                _emit(snippet, eff)

            _walk(b.get("text_blocks", []) or [], eff)

    _walk(text_blocks)
    return grouped

# ── SerpAPI result parser ─────────────────────────────────────────────────────

def parse_serpapi_result(result: dict) -> dict:
    # Real publisher URLs first -- see _resolve_raw_result_in_place.
    _resolve_raw_result_in_place(result)
    output = {
        "aio_text":       "",
        "aio_sources":    [],
        "aio_statements": [],   # NEW — list of {statement, source_title, source_url}
        "aio_statements_grouped": [],  # NEW — [{statement, citation_indexes}, ...]
        "organic_urls":   [],
        "forum_url":      None,
        "snippet_by_url": {},
    }

    # ── AI Overview ───────────────────────────────────────────────────────────
    aio = result.get("ai_overview", {})
    if aio:
        text_blocks = aio.get("text_blocks", [])
        if text_blocks:
            output["aio_text"] = _parse_text_blocks(text_blocks)

        elif aio.get("page_token"):
            print("  ℹ AIO page_token still present — attempting follow-up...")
            try:
                serpapi_link = aio.get("serpapi_link", "")
                if serpapi_link:
                    sep      = "&" if "?" in serpapi_link else "?"
                    full_url = f"{serpapi_link}{sep}api_key={SERPAPI_KEY}"
                    _serp.acquire(reserve=1)
                    resp     = _requests.get(full_url, timeout=60)
                    aio_data = resp.json()
                    if aio_data.get("error"):
                        if _serp.classify_error(aio_data["error"]) == "quota":
                            _serp.mark_quota_exhausted()
                        raise RuntimeError(f"serpapi returned error: {aio_data['error']}")
                    resp.raise_for_status()
                    text_blocks = (
                        aio_data.get("text_blocks") or
                        aio_data.get("ai_overview", {}).get("text_blocks", [])
                    )
                    if text_blocks:
                        output["aio_text"] = _parse_text_blocks(text_blocks)
                    if not aio.get("references"):
                        aio["references"] = (
                            aio_data.get("references") or
                            aio_data.get("ai_overview", {}).get("references", [])
                        )
                    if output["aio_text"]:
                        print(f"  ✓ AIO text retrieved ({len(output['aio_text'])} chars)")
                    else:
                        print(f"  ⚠ AIO follow-up returned no text_blocks")
                        print(f"    Response keys: {list(aio_data.keys())}")
            except Exception as e:
                print(f"  ⚠ AIO follow-up failed: {e}")

        if not output["aio_text"]:
            output["aio_text"] = _clean_aio_artifacts(
                aio.get("text") or aio.get("snippet") or ""
            )

        # Sources — filter out videos, try both "sources" and "references" keys
        raw_sources = list(aio.get("sources") or aio.get("references") or [])

        # NEW — capture EVERY AIO citation: union in inline reference URLs from
        # text_blocks that Google left out of the main sources array. Appended
        # AFTER the main list so existing source indexes stay stable.
        if text_blocks:
            _known_urls = {s.get("link") or s.get("url") for s in raw_sources}

            def _walk_inline_refs(blocks):
                for b in blocks or []:
                    for ref in b.get("references") or []:
                        if isinstance(ref, dict):
                            u = ref.get("link") or ref.get("url")
                            if u and u not in _known_urls:
                                _known_urls.add(u)
                                raw_sources.append({
                                    "title":   ref.get("title", "") or ref.get("name", ""),
                                    "link":    u,
                                    "snippet": ref.get("snippet", ""),
                                })
                    for item in b.get("list", []) or []:
                        _walk_inline_refs([item])
                    _walk_inline_refs(b.get("text_blocks"))

            _walk_inline_refs(text_blocks)

        output["aio_sources"] = [
            {"title": s.get("title", ""), "url": s.get("link", ""),
             "snippet": s.get("snippet", "")}        # NEW — for snippet fallback
            for s in raw_sources
            if s.get("link") and not is_video_url(s.get("link", ""))
        ]
        # NEW — populate snippet_by_url for AIO sources
        for s in raw_sources:
            link = s.get("link", "")
            if link and not is_video_url(link):
                output["snippet_by_url"][link] = s.get("snippet", "")
        # NEW — extract per-statement citations from text_blocks
        if text_blocks:
            output["aio_statements"] = _parse_text_blocks_with_sources(
                text_blocks, raw_sources
            )
            # NEW — grouped statement→citation-index extraction
            output["aio_statements_grouped"] = _parse_text_blocks_grouped(
                text_blocks, raw_sources
            )

    # ── Discussions & Forums — first only ────────────────────────────────────
    discussions = result.get("discussions_and_forums", [])
    if discussions:
        output["forum_url"] = discussions[0].get("link", None)

    # ── Organic — exactly 5 non-video results ────────────────────────────────
    for r in result.get("organic_results", []):
        if len(output["organic_urls"]) >= 5:
            break
        link = r.get("link", "")
        if link and not is_video_url(link):
            output["organic_urls"].append(link)
            # NEW — capture SerpAPI snippet for organic-results fallback
            output["snippet_by_url"][link] = r.get("snippet", "")

    # Wrappers were resolved in the raw response before parsing; this second
    # pass only catches anything that slipped through (cached, so no re-fetch).
    output["_raw_redirects"] = result.get("_google_redirects", {})
    _resolve_urls_in_place(output)

    return output

# ── Inspect helper ────────────────────────────────────────────────────────────

def inspect_result(result):
    print("=" * 65)
    print("TOP-LEVEL KEYS:")
    print("=" * 65)
    for k in result.keys():
        print(f"  {k}")

    # AI Overview
    print("\n" + "-" * 65)
    print("AI OVERVIEW")
    print("-" * 65)
    aio = result.get("ai_overview", {})
    if aio:
        print(f"  Raw keys: {list(aio.keys())}")
        if aio.get("text_blocks"):
            print(f"  ✓ text_blocks present ({len(aio['text_blocks'])} blocks) — pre-fetch worked")
        elif aio.get("page_token"):
            print(f"  ⚠ page_token still present — pre-fetch may have failed")

        parsed  = parse_serpapi_result(result)
        text    = parsed["aio_text"]
        sources = parsed["aio_sources"]
        stmts   = parsed["aio_statements"]

        print(f"\n  Text ({len(text)} chars):")
        print(f"  {text[:400]}" if text else "  (none)")
        print(f"\n  Sources ({len(sources)}) — videos filtered out:")
        for s in sources[:8]:
            print(f"    → {s.get('title','?')[:50]} | {s.get('url','?')}")

        # NEW — show statement + source pairs
        print(f"\n  ── Statement → Source pairs ({len(stmts)}) ──")
        if stmts:
            for i, stmt in enumerate(stmts, 1):
                print(f"  [{i}] Statement: {stmt['statement'][:120]}")
                if stmt['source_title'] or stmt['source_url']:
                    print(f"       Source:    {stmt['source_title'][:60]} | {stmt['source_url'][:70]}")
                else:
                    print(f"       Source:    (reference present but no URL resolved)")
        else:
            print("  (none — text_blocks may not carry inline references for this query)")

        # v2 — full block-unit segmentation preview (what Halloumi will score)
        units = parsed.get("aio_statements_grouped", [])
        print(f"\n  ── Block units / statements ({len(units)}) ──")
        for i, u in enumerate(units, 1):
            cites = ",".join(str(x) for x in u.get("citation_indexes", []))
            tag = f"[cites {cites}]" if cites else "(uncited)"
            print(f"  [{i}] {tag:<16} {u['statement'][:100]}")
    else:
        print("  ⚠ No AI Overview returned for this query.")
        alt_keys = [k for k in result if any(x in k.lower() for x in ["ai","overview","answer","featured"])]
        if alt_keys:
            print(f"  Possible alternatives: {alt_keys}")

    # Organic results
    print("\n" + "-" * 65)
    print("ORGANIC RESULTS (top 5, videos excluded)")
    print("-" * 65)
    organic = result.get("organic_results", [])
    forum_domains = ["reddit.com","quora.com","patient.info","healthunlocked.com",
                     "medhelp.org","inspire.com","webmd.com/answers","forums.webmd.com"]
    print(f"  Total returned by SerpAPI: {len(organic)}")
    shown = 0
    for r in organic:
        link = r.get("link", "?")
        if is_video_url(link):
            print(f"  ↷ Skipping video: {link[:60]}")
            continue
        tag = " 🗣 FORUM" if any(d in link for d in forum_domains) else ""
        print(f"  [{shown+1}]{tag} {r.get('title','?')[:55]}")
        print(f"       {link}")
        shown += 1
        if shown >= 5:
            break

    # People Also Ask
    print("\n" + "-" * 65)
    print("PEOPLE ALSO ASK — will be IGNORED")
    print("-" * 65)
    paa = result.get("related_questions", [])
    print(f"  Count: {len(paa)} (all skipped)")

    # Discussions & Forums
    print("\n" + "-" * 65)
    print("DISCUSSIONS & FORUMS — keep FIRST only")
    print("-" * 65)
    discussions = result.get("discussions_and_forums", [])
    if discussions:
        print(f"  Count: {len(discussions)}")
        d = discussions[0]
        print(f"  Keeping: {d.get('title','?')} → {d.get('link','?')}")
        print(f"  Discarding: {len(discussions)-1} others")
    else:
        print("  None found for this query.")

    print("\n" + "=" * 65)

# ── Run ───────────────────────────────────────────────────────────────────────
if 'QUESTIONS' not in dir() or not QUESTIONS:
    print("⚠️  Run Step 2 first to load questions.")
elif not SERPAPI_KEY:
    print("⚠️  Set your SERPAPI_KEY in Step 1 first.")
else:
    test_q = QUESTIONS[TEST_INDEX]
    print(f'Testing query [{TEST_INDEX}]: "{test_q}"\n')
    print("Calling SerpAPI...")

    RAW_RESULT = search_google(test_q, SERPAPI_KEY)
    inspect_result(RAW_RESULT)

    with open("step3_raw_result.json", "w") as f:
        json.dump(RAW_RESULT, f, indent=2)
    print("Raw JSON saved → step3_raw_result.json")
    print("\n✅ Step 3 complete.")


In [ ]:
# @title 🧩 STEP 4 — Parser: Extract URLs from SerpAPI Response { display-mode: "form" }
# @markdown Defines forum domains and previews the parser on your Step 3 test result.
# @markdown NOTE: parse_serpapi_result is defined in Step 3 — do not redefine it here.

FORUM_DOMAINS = [
    "reddit.com",
    "quora.com",
    "patient.info",
    "healthunlocked.com",
    "medhelp.org",
    "inspire.com",
    "drugs.com/comments",
    "webmd.com/answers",
    "forums.webmd.com",
    "community.patient.co.uk",
]

def is_forum_url(url: str) -> bool:
    return any(d in url for d in FORUM_DOMAINS)

# Preview on Step 3 result
if 'RAW_RESULT' not in dir():
    print("⚠️  Run Step 3 first.")
elif 'parse_serpapi_result' not in dir():
    print("⚠️  parse_serpapi_result not found — make sure Step 3 has been run.")
else:
    parsed = parse_serpapi_result(RAW_RESULT)

    print("PARSED OUTPUT")
    print("=" * 65)
    print(f"AIO Text ({len(parsed['aio_text'])} chars):")
    print(f"  {parsed['aio_text'][:300]}..." if parsed['aio_text'] else "  (none)")

    print(f"\nAIO Sources ({len(parsed['aio_sources'])}):")
    for s in parsed['aio_sources']:
        print(f"  → {s['title'][:50]} | {s['url']}")

    # NEW — show statement+source pairs
    stmts = parsed.get('aio_statements', [])
    print(f"\n── AIO Statement → Source pairs ({len(stmts)}) ──")
    if stmts:
        for i, stmt in enumerate(stmts, 1):
            print(f"  [{i}] {stmt['statement'][:110]}")
            if stmt['source_title'] or stmt['source_url']:
                print(f"       ↳ {stmt['source_title'][:60]} | {stmt['source_url'][:70]}")
            else:
                print(f"       ↳ (no URL resolved for this reference)")
    else:
        print("  (none — this AIO may not expose inline per-statement references)")

    print(f"\nOrganic URLs ({len(parsed['organic_urls'])}):")
    for i, url in enumerate(parsed['organic_urls']):
        tag = " 🗣 FORUM" if is_forum_url(url) else ""
        print(f"  [{i}]{tag} {url}")

    print(f"\nForum URL (Discussions block): {parsed['forum_url'] or '(none)'}")
    print("\n✅ Step 4 complete.")


In [ ]:
# @title 📄 STEP 5 — Page Scraper: Text, Authors, Dates, Disclosures, References { display-mode: "form" }
# @markdown Paste any article URL below to test the scraper live.

TEST_URL = "https://www.mayoclinic.org/diseases-conditions/heart-disease/in-depth/heart-disease-prevention/art-20046502"  # @param {type:"string"}

import requests
import trafilatura
import re
import json
import random
import io
import time
import asyncio
import threading
from collections import defaultdict
from bs4 import BeautifulSoup
from datetime import datetime
from urllib.parse import urljoin, urlparse

import pdfplumber
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from playwright.async_api import async_playwright

USER_AGENTS = [
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36 Edg/135.0.0.0",
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0.0.0 Safari/537.36",
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 14_4_1) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.4.1 Safari/605.1.15",
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:127.0) Gecko/20100101 Firefox/127.0",
]

# Accept-Encoding MUST advertise only what this interpreter can actually
# decode. This was hardcoded to "gzip, deflate, br", but urllib3 only ships
# gzip/deflate decoders -- Brotli needs the separate `brotli` package, which is
# NOT installed here. Servers honouring `br` therefore returned Brotli bytes
# that urllib3 handed back RAW, and `.decode(errors="replace")` turned them
# into mojibake that sailed through the quality gate as text_quality == "ok".
#
# Damage on the 2026-08-26 run: 30.2% of AIO sources and 35.7% of organic
# sources were binary garbage. JAMA/HON are scored ON that text, HallOumi
# entailed against it, and GisPy read it -- so this silently corrupted every
# downstream measure. Verified live: the same URL returns Content-Encoding
# 'br' with 42-48% replacement chars when `br` is advertised, and clean gzip
# HTML when it is not.
try:
    from urllib3.response import HTTPResponse as _U3Resp
    _DECODERS = list(getattr(_U3Resp, "CONTENT_DECODERS", ["gzip", "deflate"]))
except Exception:
    _DECODERS = ["gzip", "deflate"]
if "br" not in _DECODERS:
    try:                      # brotli present but not registered? then it IS safe
        import brotli         # noqa: F401
        _DECODERS.append("br")
    except ImportError:
        pass
_ACCEPT_ENCODING = ", ".join(dict.fromkeys(
    [e for e in _DECODERS if e != "x-gzip"] or ["gzip", "deflate"]))
print(f"Accept-Encoding set to: {_ACCEPT_ENCODING}")


def get_headers():
    return {
        "User-Agent":                random.choice(USER_AGENTS),
        "Accept":                    "text/html,application/xhtml+xml,application/xhtml;q=0.9,*/*;q=0.8",
        "Accept-Language":           "en-US,en;q=0.9",
        "Accept-Encoding":           _ACCEPT_ENCODING,
        "Referer":                   "https://www.google.com/",
        "DNT":                       "1",
        "Connection":                "keep-alive",
        "Upgrade-Insecure-Requests": "1",
    }

DISCLOSURE_KEYWORDS = [
    "conflict of interest", "conflicts of interest", "disclosure",
    "funding", "grant", "sponsor", "advertisement", "advertising",
    "supported by", "financial", "competing interest", "editorial independence",
    "video ownership", "affiliate", "paid", "compensation"
]

# ── CSV safety ────────────────────────────────────────────────────────────────

def clean_for_csv(value):
    if not isinstance(value, str):
        return value
    value = re.sub(r'[\r\n]+', ' ', value)
    value = value.replace('\t', ' ')
    value = value.replace('"', "'")
    value = re.sub(r'  +', ' ', value)
    return value.strip()

def as_text(value):
    """Coerce a scraped metadata value to a string.

    JSON-LD and meta attributes are NOT guaranteed to hold strings: some CMSs
    emit numbers where a name belongs (Healthline / Medical News Today ship
    "author": {"name": 0}), and bs4 returns a list for multi-valued attributes.
    Passing those straight into .strip() killed the whole question, so every
    metadata value goes through here first.
    """
    if value is None or isinstance(value, bool):
        return ""
    if isinstance(value, str):
        return value
    if isinstance(value, (int, float)):
        return str(value)
    if isinstance(value, (list, tuple)):
        return " ".join(as_text(v) for v in value).strip()
    return ""

# ── Fetchers ──────────────────────────────────────────────────────────────────

class FetchError(Exception):
    """Retryable fetch error (429/503/timeout/connection)."""
    pass

# Domains known to require JS rendering — skip requests/trafilatura entirely
_KNOWN_JS_DOMAINS = {
    "connect.mayoclinic.org",
    "www.mayoclinic.org",
    "mayoclinic.org",
    "www.uptodate.com",
    "uptodate.com",
    "world-heart-federation.org",
    "www.world-heart-federation.org",
}
_KNOWN_STATIC_OK = set()  # learned at runtime

# Per-domain rate limiter so concurrent fetches stay polite against the same host
_domain_lock = defaultdict(threading.Lock)
_domain_last = defaultdict(float)
_min_gap_per_host = 1.0

def _polite_wait(url):
    host = urlparse(url).netloc
    with _domain_lock[host]:
        gap = time.time() - _domain_last[host]
        if gap < _min_gap_per_host:
            time.sleep(_min_gap_per_host - gap)
        _domain_last[host] = time.time()

# Playwright worker thread — runs Playwright's ASYNC API on a private event
# loop owned by the worker thread. The old sync-API approach intermittently
# raised 'Sync API inside the asyncio loop' on Colab and, once broken, poisoned
# JS-fallback fetches for the rest of the run. The async API on a loop we
# create ourselves is immune to that, and the worker now auto-respawns if its
# thread died (e.g. after shutdown_playwright() at the end of a prior Step 8 run).
import queue as _queue

# R5 — PDF detection and Playwright click-to-expand helpers ────────────────────

def is_pdf_url(url):
    """True if URL points to a downloadable PDF (ignoring query/fragment)."""
    if not url:
        return False
    u = str(url).lower().split("?", 1)[0].split("#", 1)[0]
    return u.endswith(".pdf")

_EXPAND_KEYWORDS_RE = re.compile(
    r"(show\s*(more|references?|sources?|citations?)|expand|"
    r"references?|sources?|citations?|bibliography|further\s+reading)",
    re.I
)

async def _expand_collapsed_sections(page):
    """Best-effort: click common 'Show References' / 'Sources' / 'Citations'
    controls BEFORE reading page.content(). Filtered by keyword regex so we don't
    click unrelated controls. Matches controls WITHOUT aria-expanded
    (many sites, e.g. mayoclinic, use a plain button), dedups by button text, and
    waits longer so AJAX-loaded reference lists land. Each click is bounded and
    wrapped in try/except — failures never break the fetch."""
    selectors = [
        'button[aria-expanded="false"]',
        'a[aria-expanded="false"]',
        '[role="button"][aria-expanded="false"]',
        'summary',
        'button',
        'a',
        '[role="button"]',
    ]
    clicked = 0
    seen_txt = set()
    for sel in selectors:
        try:
            elements = await page.locator(sel).all()
        except Exception:
            continue
        for el in elements[:25]:   # cap scanned elements per selector
            try:
                txt = (await el.inner_text(timeout=500) or "").strip()
            except Exception:
                continue
            if not txt or len(txt) > 60:
                continue
            if not _EXPAND_KEYWORDS_RE.search(txt):
                continue
            if txt.lower() in seen_txt:
                continue
            seen_txt.add(txt.lower())
            try:
                await el.click(timeout=1500, no_wait_after=True)
                await page.wait_for_timeout(800)
                clicked += 1
            except Exception:
                pass
            if clicked >= 10:
                break
        if clicked >= 10:
            break
    if clicked:
        # Let any AJAX-loaded reference lists settle before content() is read.
        try:
            await page.wait_for_timeout(1200)
        except Exception:
            pass


def _is_pw_state_error(exc):
    """True when the error means the browser/driver state is corrupt (worth a
    teardown + one retry), as opposed to a page-level failure (404, timeout)."""
    s = f"{type(exc).__name__}: {exc}"
    return any(k in s for k in (
        "TargetClosed", "has been closed", "Connection closed",
        "Event loop is closed", "asyncio loop", "Sync API",
        "BrowserType.launch", "Playwright connection",
    ))

def _pw_worker(q):
    """Dedicated Playwright worker. Owns a private asyncio loop + browser state;
    never touched by other threads."""
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    state = {"pw": None, "browser": None, "context": None}

    async def ensure_browser():
        if state["browser"] is None:
            state["pw"]      = await async_playwright().start()
            state["browser"] = await state["pw"].chromium.launch(headless=True)
            state["context"] = await state["browser"].new_context(
                user_agent=get_headers()["User-Agent"]
            )

    async def teardown():
        for key in ("context", "browser"):
            if state[key]:
                try: await state[key].close()
                except Exception: pass
                state[key] = None
        if state["pw"]:
            try: await state["pw"].stop()
            except Exception: pass
            state["pw"] = None

    async def fetch_once(url, timeout):
        await ensure_browser()
        page = await state["context"].new_page()
        try:
            await page.goto(url, timeout=timeout * 1000, wait_until="domcontentloaded")
            await page.wait_for_timeout(1500)
            try:
                await _expand_collapsed_sections(page)
            except Exception:
                pass
            return await page.content()
        finally:
            try: await page.close()
            except Exception: pass

    while True:
        item = q.get()
        if item is None:  # poison pill — shut down
            try: loop.run_until_complete(teardown())
            except Exception: pass
            break

        url, timeout, holder = item
        holder["started"].set()
        try:
            try:
                holder["result"] = loop.run_until_complete(fetch_once(url, timeout))
            except Exception as e:
                if _is_pw_state_error(e):
                    # Corrupt browser/driver state — rebuild once and retry this URL
                    try: loop.run_until_complete(teardown())
                    except Exception: pass
                    holder["result"] = loop.run_until_complete(fetch_once(url, timeout))
                else:
                    raise
        except Exception as e:
            holder["error"] = e
        finally:
            holder["done"].set()

    try: loop.close()
    except Exception: pass

# A POOL of workers, each with its own thread, event loop and browser, all
# pulling from one queue. A single worker made every JS-rendered fetch in the
# run serial: at the 3,852-question scale (8.4 AIO sources/question, ~325
# Playwright fetches per 50 questions) it capped throughput no matter how many
# question workers ran. Workers share nothing but the queue, so the async-API
# isolation that fixed the Colab 'Sync API inside asyncio loop' bug still holds.
PLAYWRIGHT_WORKERS = int(os.environ.get("PLAYWRIGHT_WORKERS", "3"))

_pw_queue   = None
_pw_threads = []
_pw_lock    = threading.Lock()

def _ensure_pw_worker():
    """Spawn (or respawn) Playwright workers so PLAYWRIGHT_WORKERS are alive.
    Makes re-runs of Step 8 after shutdown_playwright() work without re-running
    this cell, and recovers if a worker thread ever died."""
    global _pw_queue, _pw_threads
    with _pw_lock:
        if _pw_queue is None:
            _pw_queue = _queue.Queue()
        _pw_threads = [t for t in _pw_threads if t.is_alive()]
        while len(_pw_threads) < max(1, PLAYWRIGHT_WORKERS):
            t = threading.Thread(target=_pw_worker, args=(_pw_queue,),
                                 name=f"playwright-worker-{len(_pw_threads)}",
                                 daemon=True)
            t.start()
            _pw_threads.append(t)
        return _pw_queue

def _playwright_fetch(url, timeout=20):
    """Submit a fetch request to the Playwright pool and wait for the result."""
    q = _ensure_pw_worker()
    holder = {"result": None, "error": None,
              "started": threading.Event(), "done": threading.Event()}
    q.put((url, timeout, holder))
    # Time spent waiting in the queue is not the page's fault: wait (bounded)
    # for a worker to pick the job up, THEN apply the per-page budget. A worker
    # may tear down + relaunch the browser once, hence twice the page timeout
    # plus launch overhead.
    holder["started"].wait(timeout=600)
    holder["done"].wait(timeout=timeout * 2 + 30)
    if holder["error"] is not None:
        raise holder["error"]
    if holder["result"] is None:
        raise RuntimeError(f"playwright timeout for {url}")
    return holder["result"]

def shutdown_playwright():
    """Stop every Playwright worker gracefully. Safe to call repeatedly; the
    next _playwright_fetch after this simply respawns a fresh pool."""
    global _pw_queue, _pw_threads
    with _pw_lock:
        if _pw_queue is not None:
            for t in _pw_threads:
                if t.is_alive():
                    _pw_queue.put(None)  # each worker exits on one pill
        _pw_threads = []
        _pw_queue = None

@retry(stop=stop_after_attempt(3),
       wait=wait_exponential(multiplier=1, min=2, max=10),
       retry=retry_if_exception_type((requests.Timeout, requests.ConnectionError, FetchError)),
       reraise=True)
def _requests_fetch(url, timeout=20):
    """Streamed fetch with a hard byte ceiling.

    The old version called r.content / r.text, which pulls the ENTIRE response
    into RAM at any size. With up to 16 concurrent fetches, two large journal
    PDFs in the same wave were enough to OOM a 12.7GB Colab runtime -- which is
    what killed the run ~17 sites in. Now we pre-check Content-Length, stream in
    64KB chunks, and stop dead at the cap.
    """
    with requests.get(url, headers=get_headers(), timeout=timeout,
                      stream=True) as r:
        if r.status_code in (429, 503):
            raise FetchError(f"status_{r.status_code}")
        r.raise_for_status()

        ctype  = r.headers.get("Content-Type", "").lower()
        is_pdf = "pdf" in ctype or url.lower().endswith(".pdf")
        cap    = MAX_PDF_BYTES if is_pdf else MAX_FETCH_BYTES

        declared = r.headers.get("Content-Length", "")
        if declared.isdigit() and int(declared) > cap:
            raise OversizedError(
                f"{int(declared) // (1024*1024)}MB > {cap // (1024*1024)}MB cap")

        data, truncated = _read_capped(r, cap)
        if truncated:
            if is_pdf:
                # A half-downloaded PDF is unparseable; don't hand pdfplumber junk.
                raise OversizedError(f"pdf exceeded {cap // (1024*1024)}MB cap")
            print(f"    ⚠ Truncated oversized page at "
                  f"{cap // (1024*1024)}MB: {url[:60]}")

        if is_pdf:
            return data
        return data.decode(r.encoding or "utf-8", errors="replace")

# ── Wayback Machine fallback ──────────────────────────────────────────────────
# Targets paywalls/bot-blocks (402/403): an archived copy usually serves the
# full article. The 'id_' flag requests the RAW archived page (no toolbar).

def _wayback_fetch(url, timeout=20):
    avail = requests.get(
        "http://archive.org/wayback/available",
        params={"url": url},
        headers={"User-Agent": get_headers()["User-Agent"]},
        timeout=timeout,
    )
    avail.raise_for_status()
    snap = (avail.json().get("archived_snapshots") or {}).get("closest") or {}
    snap_url = snap.get("url", "")
    if not snap_url or not snap.get("available"):
        raise FetchError("no_snapshot")
    # Insert id_ after the timestamp for the raw page without wayback chrome:
    # .../web/20240101000000/http://...  ->  .../web/20240101000000id_/http://...
    snap_url = re.sub(r"(/web/\d{4,14})/", r"\1id_/", snap_url, count=1)
    text = _capped_get(snap_url, timeout=timeout + 10)
    if not text or len(text) < 500:
        raise FetchError("snapshot_too_small")
    return text

# ── AMP variant fallback ──────────────────────────────────────────────────────
# Only tried on 401/402/403 (paywall/bot-block) responses. AMP pages are
# static, ad-light, and often skip the paywall JS entirely.

def _amp_fetch(url, timeout=15):
    base = url.split("#", 1)[0]
    candidates = []
    if "?" not in base:
        candidates.append(base.rstrip("/") + "/amp")
        candidates.append(base + ("" if base.endswith("/") else "/") + "amp/")
        candidates.append(base + "?amp=1")
    else:
        candidates.append(base + "&amp=1")
    for cand in candidates:
        try:
            amp_text = _capped_get(cand, timeout=timeout)
            if amp_text and len(amp_text) > 2000:
                head = amp_text.lower()[:2000]
                if "<html" in head or "<!doctype" in head:
                    return amp_text
        except Exception:
            continue
    raise FetchError("no_amp_variant")

_PAYWALL_STATUSES = ("401", "402", "403")

# ── Content quality validation ────────────────────────────────────────────────
# A fetch is only a success if the CONTENT is an article, not a bot-check /
# CAPTCHA / error interstitial served with HTTP 200. Validated against real
# failures: PMC "Checking your browser", Cloudflare "Just a moment...",
# Akamai "Access Denied", Jina-proxied CAPTCHA warnings.

_GARBAGE_TEXT_RE = re.compile(
    r"(checking your browser|just a moment|access denied|error 40\d|"
    r"\bforbidden\b|performing security verification|"
    r"not automatically redirected|please switch to a different browser|"
    r"enable javascript and cookies|requiring captcha|complete the captcha|"
    r"verify you are (?:a human|not a bot)|attention required|error code:|"
    r"security service to protect against malicious bots)",
    re.I)

def assess_text_quality(text):
    """Classify text: ok | garbage_binary | garbage_interstitial | too_short | empty.
    Garbage patterns only count on SHORT texts (<2500 chars) — a real article
    that merely mentions 'captcha' or 'forbidden foods' is long and stays ok."""
    if text is None or not str(text).strip():
        return "empty"
    t = str(text).strip()
    # Undecoded compressed bytes / wrong-codec text. The old gate never looked
    # for this, so 30%+ of sources passed as "ok" while holding binary junk.
    # U+FFFD is what .decode(errors="replace") emits for bytes it cannot map;
    # real prose essentially never contains it.
    _sample = t[:4000]
    if _sample:
        # Two unambiguous signals of undecoded bytes. Deliberately NOT an
        # ASCII-ratio test: that flags legitimate non-Latin pages (a Chinese
        # hospital page in the 8/26 corpus tripped it), and this study
        # includes global-health questions where those sources are real.
        _repl = _sample.count("\ufffd") / len(_sample)
        _ws = "\n" + "\r" + "\t"
        _ctrl = sum(1 for c in _sample
                    if ord(c) < 32 and c not in _ws) / len(_sample)
        if _repl > 0.02 or _ctrl > 0.02:
            return "garbage_binary"
    if len(t) < 2500 and _GARBAGE_TEXT_RE.search(t[:1500]):
        return "garbage_interstitial"
    if len(t) < 300:
        return "too_short"
    return "ok"

_HTML_TAG_STRIP_RE = re.compile(
    r"<script[\s\S]*?</script>|<style[\s\S]*?</style>|<[^>]+>")

def _fetched_content_ok(html):
    """Gate used inside fetch_html: reject fetches whose visible text is a
    bot-check/error interstitial so the cascade continues to the next method.
    Shortness alone does NOT reject here — full extraction judges that later."""
    if html is None:
        return False
    if isinstance(html, bytes):
        return True   # PDFs validated after extraction
    sample = _HTML_TAG_STRIP_RE.sub(" ", html[:60000])
    sample = re.sub(r"\s+", " ", sample).strip()
    return assess_text_quality(sample) != "garbage_interstitial"

# Jina prefixes its markdown with Title/Warning headers — reject its own
# error/CAPTCHA pages explicitly (they arrive as HTTP 200).
_JINA_BAD_RE = re.compile(
    r"^(?:Title:\s*(?:Just a moment|Access Denied|Attention Required).*|"
    r"Warning:\s*(?:Target URL returned error|This page maybe requiring CAPTCHA).*)$",
    re.I | re.M)

# ── PubMed Central / PubMed fetcher (Europe PMC — no bot-checks) ─────────────
# pmc.ncbi.nlm.nih.gov serves bot-check interstitials to plain requests; the
# Europe PMC REST API serves the same articles' full text (OA) or abstract
# freely. Returns extracted TEXT (not HTML) — extract_main_text passes
# non-HTML strings through unchanged.

_PMC_URL_RE    = re.compile(r"ncbi\.nlm\.nih\.gov/(?:pmc/)?articles/(PMC\d+)", re.I)
_PUBMED_URL_RE = re.compile(r"pubmed\.ncbi\.nlm\.nih\.gov/(\d+)", re.I)
_XML_TAG_RE    = re.compile(r"<[^>]+>")

def _europepmc_text(xml_or_html):
    return clean_for_csv(re.sub(r"\s+", " ", _XML_TAG_RE.sub(" ", xml_or_html)).strip())

def _europepmc_search(query, timeout=25):
    r = requests.get(
        "https://www.ebi.ac.uk/europepmc/webservices/rest/search",
        params={"query": query, "resultType": "core", "format": "json"},
        timeout=timeout)
    r.raise_for_status()
    return r.json().get("resultList", {}).get("result", [])

def _resolve_pmcid(url, timeout=25):
    """PMCID for a PMC/PubMed URL, or None if the URL isn't one."""
    m = _PMC_URL_RE.search(url)
    if m:
        return m.group(1)
    m2 = _PUBMED_URL_RE.search(url)
    if not m2:
        return None
    try:
        hits = _europepmc_search(f"EXT_ID:{m2.group(1)} AND SRC:MED", timeout)
        return hits[0].get("pmcid") if hits else None
    except Exception:
        return None

def _europepmc_fulltext(pmcid, timeout=25):
    """Open-access full text for a PMCID. Raises FetchError when not OA."""
    xml = _capped_get(
        f"https://www.ebi.ac.uk/europepmc/webservices/rest/{pmcid}/fullTextXML",
        timeout=timeout)
    if len(xml) > 3000:
        text = _europepmc_text(xml)
        if len(text) > 1000:
            return text
    raise FetchError("not_open_access")

def _europepmc_abstract(pmcid, timeout=25):
    """Abstract for a PMCID — last resort when even Wayback has no full text."""
    hits = _europepmc_search(f"PMCID:{pmcid}", timeout)
    abstract = hits[0].get("abstractText", "") if hits else ""
    if abstract and len(abstract) > 300:
        return _europepmc_text(abstract)
    raise FetchError("no_abstract")

def fetch_html(url, timeout=20):
    """
    Returns (html, method, error_msg). html may be None.
    Cascade: Europe PMC (PMC/PubMed URLs) -> requests (retries) -> trafilatura
             -> Playwright -> AMP variant -> Jina Reader -> Wayback Machine
             -> Europe PMC abstract (PMC last resort).
    EVERY stage's content is validated with _fetched_content_ok — a bot-check /
    CAPTCHA / error page served as HTTP 200 no longer counts as success, so the
    cascade keeps going (this was how PMC interstitials and Jina error pages
    ended up as source 'text' in the datasets).
    """
    _polite_wait(url)
    host = urlparse(url).netloc

    err_parts = []
    paywall_like = False

    # 0. PMC / PubMed — Europe PMC serves full text with no bot-checks
    pmcid = _resolve_pmcid(url) if "ncbi.nlm.nih.gov" in host else None
    if pmcid:
        try:
            text = _europepmc_fulltext(pmcid, timeout=timeout + 5)
            return text, "europepmc", ""
        except Exception as e:
            err_parts.append(f"europepmc:{type(e).__name__}:{str(e)[:60]}")

    if host in _KNOWN_JS_DOMAINS:
        try:
            html = _playwright_fetch(url, timeout=timeout)
            if html and _fetched_content_ok(html):
                return html, "playwright", ""
            if html:
                err_parts.append("playwright:garbage_interstitial")
        except Exception as e:
            err_parts.append(f"playwright:{type(e).__name__}:{str(e)[:80]}")

    # 1. requests (with retry)
    try:
        result = _requests_fetch(url, timeout=timeout)
        if _fetched_content_ok(result):
            _KNOWN_STATIC_OK.add(host)
            return result, "requests", ""
        err_parts.append("requests:garbage_interstitial")
        paywall_like = True   # bot-check page — treat like a block for AMP
    except OversizedError as e:
        # Too big is a property of the document, not of this fetch strategy.
        # Trying the same file six more ways just burns time and RAM.
        print(f"    ⚠ Skipping oversized document ({e}): {url[:60]}")
        return None, "", f"oversized:{e}"
    except Exception as e:
        err_msg = f"requests:{type(e).__name__}:{str(e)[:80]}"
        err_parts.append(err_msg)
        paywall_like = any(f"{s} Client Error" in err_msg for s in _PAYWALL_STATUSES)

    # 2. trafilatura fetcher
    try:
        html = trafilatura.fetch_url(url)
        if html and _fetched_content_ok(html):
            return html, "trafilatura", ""
        if html:
            err_parts.append("trafilatura:garbage_interstitial")
    except Exception as e:
        err_parts.append(f"trafilatura:{type(e).__name__}")

    # 3. Playwright (JS render) — skip if already tried via _KNOWN_JS_DOMAINS
    if host not in _KNOWN_JS_DOMAINS:
        try:
            html = _playwright_fetch(url, timeout=timeout)
            if html and _fetched_content_ok(html):
                # Remember this domain needs JS so future calls skip the slow path
                _KNOWN_JS_DOMAINS.add(host)
                return html, "playwright", ""
            if html:
                err_parts.append("playwright:garbage_interstitial")
        except Exception as e:
            err_parts.append(f"playwright:{type(e).__name__}:{str(e)[:80]}")

    # 4. AMP variant — cheap shot, only worth it on paywall/bot-block statuses
    if paywall_like:
        try:
            html = _amp_fetch(url, timeout=timeout)
            if html and _fetched_content_ok(html):
                return html, "amp", ""
            if html:
                err_parts.append("amp:garbage_interstitial")
        except Exception as e:
            err_parts.append(f"amp:{type(e).__name__}:{str(e)[:60]}")

    # 5. Jina AI Reader (free no-key proxy that renders JS)
    try:
        jina_url = "https://r.jina.ai/" + url
        jina_text = _capped_get(jina_url, timeout=timeout + 10)
        if jina_text and len(jina_text) > 200:
            # Reject Jina's own error/CAPTCHA pages (delivered as HTTP 200)
            if _JINA_BAD_RE.search(jina_text[:1500]) or not _fetched_content_ok(jina_text):
                err_parts.append("jina:garbage_interstitial")
            else:
                return jina_text, "jina", ""
    except Exception as e:
        err_parts.append(f"jina:{type(e).__name__}:{str(e)[:80]}")

    # 6. Wayback Machine — archived copy bypasses paywalls & bot-blocks
    try:
        html = _wayback_fetch(url, timeout=timeout)
        if html and _fetched_content_ok(html):
            return html, "wayback", ""
        if html:
            err_parts.append("wayback:garbage_interstitial")
    except Exception as e:
        err_parts.append(f"wayback:{type(e).__name__}:{str(e)[:60]}")

    # 7. Europe PMC abstract — PMC last resort when even Wayback lacks full text
    if pmcid:
        try:
            text = _europepmc_abstract(pmcid, timeout=timeout + 5)
            return text, "europepmc_abstract", ""
        except Exception as e:
            err_parts.append(f"europepmc_abstract:{type(e).__name__}:{str(e)[:40]}")

    return None, "none", " | ".join(err_parts)

# ── Download size guards ─────────────────────────────────────────────────────
# Nothing here used to bound how many bytes a single document could pull into
# memory. These caps are deliberately generous for prose and tighter for PDFs,
# which cost far more to parse than to download.
MAX_FETCH_BYTES = 25 * 1024 * 1024   # 25MB per HTML document
MAX_PDF_BYTES   = 15 * 1024 * 1024   # PDFs get pdfplumber'd; keep them smaller
MAX_PDF_PAGES   = 60                 # pages actually extracted from a PDF
MAX_CACHE_BYTES = 256 * 1024 * 1024  # total _fetch_cache payload budget


class OversizedError(Exception):
    """Document exceeds a size cap. Deliberately NOT in the tenacity retry
    tuple -- retrying a too-big file just downloads it again."""


def _read_capped(response, max_bytes):
    """Stream a response body up to max_bytes. -> (bytes, truncated_bool)."""
    chunks, total = [], 0
    for chunk in response.iter_content(chunk_size=65536):
        if not chunk:
            continue
        chunks.append(chunk)
        total += len(chunk)
        if total >= max_bytes:
            return b"".join(chunks)[:max_bytes], True
    return b"".join(chunks), False


def _capped_get(url, timeout=20, max_bytes=None):
    """Streamed GET returning decoded text, capped. For the fallback fetchers."""
    max_bytes = max_bytes or MAX_FETCH_BYTES
    with requests.get(url, headers=get_headers(), timeout=timeout,
                      stream=True) as r:
        r.raise_for_status()
        declared = r.headers.get("Content-Length", "")
        if declared.isdigit() and int(declared) > max_bytes:
            raise OversizedError(f"{int(declared) // (1024*1024)}MB")
        data, _ = _read_capped(r, max_bytes)
        return data.decode(r.encoding or "utf-8", errors="replace")


# Success-only fetch cache. The old @lru_cache also cached FAILURES, so a URL
# that failed once (e.g. transient Playwright crash) stayed failed for the
# whole session even after the worker recovered.
_fetch_cache = {}
_fetch_cache_lock = threading.Lock()
_fetch_cache_bytes = 0

def _cached_fetch(url):
    """Cache by URL so duplicate citation URLs across questions cost nothing.
    Only successful fetches (html is not None) are cached.

    The entry-count cap alone was misleading: entries hold RAW page bodies, so
    4096 of them is unbounded in bytes. Budget the payload instead.
    """
    global _fetch_cache_bytes
    with _fetch_cache_lock:
        if url in _fetch_cache:
            return _fetch_cache[url]
    result = fetch_html(url)
    if result[0] is not None:
        payload = result[0]
        size = len(payload) if isinstance(payload, (bytes, str)) else 0
        with _fetch_cache_lock:
            if (len(_fetch_cache) < 4096
                    and _fetch_cache_bytes + size <= MAX_CACHE_BYTES):
                _fetch_cache[url] = result
                _fetch_cache_bytes += size
    return result


def clear_fetch_cache():
    """Drop the whole page cache. Called between batches in Step 8."""
    global _fetch_cache_bytes
    with _fetch_cache_lock:
        n, mb = len(_fetch_cache), _fetch_cache_bytes / (1024 * 1024)
        _fetch_cache.clear()
        _fetch_cache_bytes = 0
    print(f"  🧹 Cleared fetch cache ({n} entries, {mb:.0f}MB)")

def extract_main_text(html, url):
    # PDF branch
    if isinstance(html, bytes) or url.lower().endswith(".pdf"):
        try:
            if isinstance(html, bytes):
                raw = html
            else:
                with requests.get(url, headers=get_headers(), timeout=20,
                                  stream=True) as _r:
                    _r.raise_for_status()
                    raw, _ = _read_capped(_r, MAX_PDF_BYTES)
            if raw[:4] == b"%PDF" or isinstance(html, bytes):
                with pdfplumber.open(io.BytesIO(raw)) as pdf:
                    n_pages = len(pdf.pages)
                    parts = []
                    # Extracting EVERY page was the biggest single allocator in
                    # the pipeline: pdfplumber holds a per-page object graph and
                    # a 300-page clinical PDF can balloon to multiple GB. Cap the
                    # pages and flush each one as we go.
                    for p in pdf.pages[:MAX_PDF_PAGES]:
                        try:
                            parts.append(p.extract_text() or "")
                        finally:
                            try:
                                p.flush_cache()
                            except Exception:
                                pass
                    if n_pages > MAX_PDF_PAGES:
                        print(f"    ℹ PDF truncated to first {MAX_PDF_PAGES} of "
                              f"{n_pages} pages: {url[:60]}")
                    return clean_for_csv("\n".join(parts))
        except Exception:
            pass
        if isinstance(html, bytes):
            # Couldn't decode as PDF and we have no HTML — give up
            return ""
    # NEW — Jina-style markdown short-circuit: if input doesn't look like HTML,
    # treat it as already-extracted text and skip trafilatura/BS entirely.
    if isinstance(html, str):
        head = html.lower()[:1000]
        if "<html" not in head and "<body" not in head and "<!doctype" not in head:
            return clean_for_csv(html)


    # 1. trafilatura recall-first (much more permissive than favor_precision)
    text = trafilatura.extract(
        html,
        include_comments=False,
        include_tables=True,
        no_fallback=False,
        favor_recall=True,
    ) or ""

    # 2. If short, try default settings — sometimes that surfaces more
    if len(text) < 500:
        alt = trafilatura.extract(
            html,
            include_comments=False,
            include_tables=True,
            no_fallback=False,
        ) or ""
        if len(alt) > len(text):
            text = alt

    # 3. BeautifulSoup body-text fallback for stubborn pages
    if len(text) < 500:
        try:
            soup = BeautifulSoup(html, "lxml")
            for t in soup(["script", "style", "nav", "footer", "header",
                           "aside", "form", "noscript", "iframe"]):
                t.decompose()
            # NEW — strip ad containers BEFORE pulling text
            for sel in ("ins.adsbygoogle", "[class*='advert']", "[class*='Advert']",
                        "[class*='sponsor']", "[class*='Sponsor']", "[class*='promo']",
                        "[id*='ad-']", "[id*='-ad']", "[id*='ads-']",
                        "[class*='ad-']", "[class*='-ad']", "[class*='ads-']",
                        "[data-ad-slot]", "[data-ad-client]", "[data-ad]",
                        "aside.ad", "div.ad", "section.ad"):
                try:
                    for el in soup.select(sel): el.decompose()
                except Exception: pass
            main = soup.find("main") or soup.find("article") or soup.body or soup
            candidate = "\n".join(s.strip() for s in main.stripped_strings if s.strip())
            if len(candidate) > len(text):
                text = candidate
        except Exception:
            pass

    return clean_for_csv(text)

# ── NEW: Editorial org-name resolver ─────────────────────────────────────────
# Maps clean domain fragments → human-readable org names.
# Add more entries here as needed.

DOMAIN_ORG_MAP = {
    "heart.org":          "American Heart Association",
    "cancer.org":         "American Cancer Society",
    "diabetes.org":       "American Diabetes Association",
    "cdc.gov":            "Centers for Disease Control and Prevention",
    "nih.gov":            "National Institutes of Health",
    "nlm.nih.gov":        "National Library of Medicine",
    "medlineplus.gov":    "MedlinePlus",
    "mayoclinic.org":     "Mayo Clinic",
    "clevelandclinic.org":"Cleveland Clinic",
    "hopkinsmedicine.org":"Johns Hopkins Medicine",
    "healthline.com":     "Healthline",
    "webmd.com":          "WebMD",
    "medicalnewstoday.com":"Medical News Today",
    "who.int":            "World Health Organization",
    "acog.org":           "American College of Obstetricians and Gynecologists",
    "aap.org":            "American Academy of Pediatrics",
    "lung.org":           "American Lung Association",
    "kidney.org":         "National Kidney Foundation",
    "alz.org":            "Alzheimer's Association",
    "nami.org":           "National Alliance on Mental Illness",
    "niddk.nih.gov":      "National Institute of Diabetes and Digestive and Kidney Diseases",
    "nhlbi.nih.gov":      "National Heart, Lung, and Blood Institute",
    "arthritis.org":      "Arthritis Foundation",
    "psoriasis.org":      "National Psoriasis Foundation",
}

def get_editorial_org_name(soup, url):
    """
    Identify the publishing organization of a page for editorial attribution.
    Tries (in order):
      1. Known domain → org name lookup
      2. og:site_name meta tag
      3. JSON-LD publisher / sourceOrganization
      4. Title tag site suffix (e.g. "Article Title | Site Name")
      5. Bare domain as last resort
    Returns a string like "American Heart Association editorial team".
    """
    domain = urlparse(url).netloc.lower()
    if domain.startswith("www."):
        domain = domain[4:]

    # 1. Known mapping
    for known, org_name in DOMAIN_ORG_MAP.items():
        if known in domain:
            return f"{org_name} editorial team"

    # 2. og:site_name
    og_tag = (
        soup.find("meta", attrs={"property": "og:site_name"}) or
        soup.find("meta", attrs={"name":     "og:site_name"})
    )
    if og_tag:
        site_name = as_text(og_tag.get("content", "")).strip()
        if site_name and len(site_name) < 80:
            return f"{site_name} editorial team"

    # 3. JSON-LD publisher / sourceOrganization
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "{}")
            items = data if isinstance(data, list) else [data]
            for item in items:
                for key in ("publisher", "sourceOrganization", "provider"):
                    pub = item.get(key, {})
                    if isinstance(pub, dict):
                        name = as_text(pub.get("name", "")).strip()
                        if name and len(name) < 80:
                            return f"{name} editorial team"
        except Exception:
            pass

    # 4. Title tag — take last segment after common separators
    title_tag = soup.find("title")
    if title_tag:
        title_text = title_tag.get_text(strip=True)
        for sep in [" | ", " - ", " – ", " — "]:
            if sep in title_text:
                candidate = title_text.split(sep)[-1].strip()
                if candidate and len(candidate) < 70:
                    return f"{candidate} editorial team"

    # 5. Bare domain fallback
    return f"{domain} editorial team"

# ── Author extractor (updated to accept URL for editorial fallback) ───────────

def extract_authors(soup, url=""):
    candidates = []

    # 1. JSON-LD structured data
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "{}")
            items = data if isinstance(data, list) else [data]
            for item in items:
                author = item.get("author", {})
                # An author entry can be a dict, a bare string, or a list mixing
                # both — and its "name" is not always a string (see as_text).
                for a in (author if isinstance(author, list) else [author]):
                    if isinstance(a, dict):
                        candidates.append(as_text(a.get("name", "")))
                    else:
                        candidates.append(as_text(a))
        except Exception:
            pass

    # 2. Meta tags
    for meta in soup.find_all("meta", attrs={"name": re.compile(r"author", re.I)}):
        val = as_text(meta.get("content", ""))
        if val:
            candidates.append(val)

    # 3. Class-based author tags
    for tag in soup.find_all(attrs={"class": re.compile(r"author|byline|writer", re.I)}):
        text = tag.get_text(separator=" ", strip=True)
        if text and len(text) < 100 and text.count("|") <= 2:
            candidates.append(text)

    # 4. rel="author" links
    for a in soup.find_all("a", rel="author"):
        candidates.append(a.get_text(strip=True))

    # 5. "By [Name]" plain-text pattern
    by_pattern = re.compile(r'\bBy\s+([A-Z][^\n<]{2,80})', re.IGNORECASE)
    for tag in soup.find_all(["p", "span", "div", "em", "small"]):
        text = tag.get_text(separator=" ", strip=True)
        match = by_pattern.match(text)
        if (
            match
            and len(text) < 80
            and text.count("|") <= 1
            and not text.rstrip().endswith(":")
            and not text.rstrip().endswith(",")
        ):
            candidates.append(text)

    # Filter out date-like strings and nav menu items
    date_like = re.compile(
        r'\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec|\d{4}|'
        r'last\s+reviewed|last\s+updated|reviewed\s+by|updated|published)\b',
        re.IGNORECASE
    )

    seen, clean = set(), []
    for c in candidates:
        c = clean_for_csv(as_text(c).strip())
        if not c or c in seen or len(c) <= 1:
            continue
        if not any(ch.isalpha() for ch in c):
            continue  # numeric/punctuation-only placeholders are never names
        if date_like.search(c) and len(c) < 60:
            continue
        if c.count("|") > 2:
            first = c.split("|")[0].strip()
            if first and len(first) < 80:
                c = first
            else:
                continue
        if c.rstrip().endswith(":") or c.rstrip().endswith(","):
            continue
        seen.add(c)
        clean.append(c)

    if clean:
        return " | ".join(clean)

    # ── NEW: Editorial fallback ───────────────────────────────────────────────
    # No author found after all methods — if we have a URL, attribute to the
    # editorial team of the publishing organization.
    if url:
        editorial = get_editorial_org_name(soup, url)
        # Tag as inferred (mirrors the '(inferred)' date convention) so scoring
        # can treat editorial-team fallbacks as non-authored.
        editorial = f"{editorial} (inferred)"
        print(f"    ℹ No author found — using editorial fallback: '{editorial}'")
        return editorial

    return ""

def extract_dates(soup):
    pub_date, upd_date = "", ""

    # 1. JSON-LD structured data
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "{}")
            items = data if isinstance(data, list) else [data]
            for item in items:
                if not pub_date:
                    pub_date = item.get("datePublished", "")
                if not upd_date:
                    upd_date = item.get("dateModified", "")
        except Exception:
            pass

    # 2. Meta tags
    if not pub_date:
        for name in ["article:published_time", "date", "pubdate", "DC.date",
                     "citation_publication_date", "citation_date",
                     "citation_online_date", "citation_cover_date"]:
            tag = soup.find("meta", attrs={"property": name}) or \
                  soup.find("meta", attrs={"name": name})
            if tag:
                pub_date = tag.get("content", "")
                break

    if not upd_date:
        for name in ["article:modified_time", "last-modified",
                     "citation_modification_date", "revised"]:
            tag = soup.find("meta", attrs={"property": name}) or \
                  soup.find("meta", attrs={"name": name})
            if tag:
                upd_date = tag.get("content", "")
                break

    # 3. <time> tag
    if not pub_date:
        t = soup.find("time", attrs={"datetime": True})
        if t:
            pub_date = t.get("datetime", "")

    # 4. Last-resort: scan ALL visible text for a date-like string.
    #    Handles JINA-markdown pages (one giant text block — the old per-tag
    #    150-char scan missed these) and journal citation formats like
    #    "2021 Nov-Dec". Prefers a date sitting next to a publish/update cue.
    if not pub_date:
        MON = (r"(?:Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|"
               r"Jul(?:y)?|Aug(?:ust)?|Sep(?:t(?:ember)?)?|Oct(?:ober)?|"
               r"Nov(?:ember)?|Dec(?:ember)?)")
        date_alts = (
            MON + r"\.?\s+\d{1,2},?\s+\d{4}"                       # February 21, 2022
            + r"|\d{1,2}\s+" + MON + r"\.?\s+\d{4}"                 # 21 February 2022
            + r"|\d{4}\s+" + MON + r"\.?(?:\s*[-\u2013/]\s*" + MON + r"\.?)?"  # 2021 Nov-Dec
            + r"|" + MON + r"\.?\s+\d{4}"                            # February 2022
            + r"|\d{1,2}/\d{1,2}/\d{4}"                              # 02/21/2022
            + r"|\d{4}-\d{2}-\d{2}"                                  # 2022-02-21
        )
        date_re = re.compile(date_alts, re.IGNORECASE)
        cue_re = re.compile(
            r"(?:published|posted|updated|last\s+(?:updated|revised|reviewed)|"
            r"reviewed|date[d]?)\b[^.\n]{0,40}?(" + date_alts + r")",
            re.IGNORECASE)
        full_text = soup.get_text(" ", strip=True)
        m = cue_re.search(full_text)          # 4a. date next to a publish/update cue
        if m:
            pub_date = m.group(1).strip()      # labeled date (e.g. "Published on ...") -> real
        else:
            m2 = date_re.search(full_text)    # 4b. any bare date anywhere -> mark inferred
            if m2:
                pub_date = f"{m2.group(0).strip()} (inferred)"

    return str(pub_date)[:35], str(upd_date)[:35]

def extract_disclosures(soup):
    found = []
    kw_pattern = re.compile("|".join(DISCLOSURE_KEYWORDS), re.I)

    for tag in soup.find_all(["p", "div", "section", "aside", "footer"]):
        text = tag.get_text(separator=" ", strip=True)
        if kw_pattern.search(text) and 20 < len(text) < 2000:
            found.append(clean_for_csv(text))

    found_sorted = sorted(set(found), key=len)
    deduped = []
    for i, t in enumerate(found_sorted):
        if not any(t in found_sorted[j] for j in range(i + 1, len(found_sorted))):
            deduped.append(t)

    return " || ".join(deduped[:5])

def extract_references(soup, base_url):
    """R5: Broader keyword detection, multi-section collection (no break-on-first),
    lower thresholds inside confirmed reference sections, false-positive guard."""
    base_domain = urlparse(base_url).netloc
    skip_patterns = re.compile(
        r"(facebook|twitter|instagram|youtube|linkedin|tiktok|"
        r"pinterest|mailto:|javascript:|google\.com/search|"
        r"amazon\.com/|ads\.|doubleclick|googlesyndication|"
        r"googleadservices|taboola|outbrain|criteo|mgid|adsense|"
        r"adnxs|adsrvr|adroll|adform|adsystem)",
        re.I
    )
    url_in_text_pattern = re.compile(r'https?://[^\s\)\]>,\"\']+', re.IGNORECASE)

    # R5 - broader keyword regex covers "Resources", "Further reading", etc.
    section_labels = re.compile(
        r"(reference|citation|source|bibliography|footnote|show.?reference|"
        r"\bresources?\b|further\s+(reading|information)|external\s+links?|"
        r"more\s+information|see\s+also|useful\s+links?|helpful\s+links?|"
        r"related\s+links?|additional\s+resources?)",
        re.I
    )

    # R5 - collect ALL candidate sections, don't break-on-first
    candidates = []
    for details in soup.find_all(["details", "div", "section", "ol", "ul", "nav"]):
        summary = details.find("summary")
        summary_text = summary.get_text(strip=True) if summary else ""
        id_class = (
            (details.get("id", "") or "") + " " +
            " ".join(details.get("class", []) or []) + " " +
            (details.get("aria-label", "") or "") + " " +
            summary_text
        )
        if section_labels.search(id_class):
            candidates.append(details)

    # Heading-anchored siblings (also collect all, don't break)
    for heading in soup.find_all(["h2", "h3", "h4", "button", "summary", "span"]):
        if section_labels.search(heading.get_text(strip=True)):
            sibling = heading.find_next_sibling()
            if sibling:
                candidates.append(sibling)

    # R5 - lower thresholds inside confirmed reference sections
    in_confirmed = bool(candidates)
    MIN_CITATION_LENGTH = 10 if in_confirmed else 40
    MIN_WORD_COUNT      = 1 if in_confirmed else 5
    search_scopes = candidates if candidates else [soup]

    references = []
    seen_urls = set()

    for scope in search_scopes:
        # R5 - false-positive guard
        if in_confirmed:
            has_outbound = False
            for li in scope.find_all("li"):
                for a in li.find_all("a", href=True):
                    full = urljoin(base_url, a["href"].strip())
                    p = urlparse(full)
                    if (p.scheme in ("http", "https")
                            and p.netloc and p.netloc != base_domain
                            and not skip_patterns.search(full)
                            and not is_video_url(full)):
                        has_outbound = True
                        break
                if has_outbound:
                    break
            if not has_outbound:
                continue

        for i, li in enumerate(scope.find_all("li")):
            text = li.get_text(separator=" ", strip=True)

            if len(text) < MIN_CITATION_LENGTH:
                continue
            if len(text.split()) < MIN_WORD_COUNT:
                continue

            url = None
            for a in li.find_all("a", href=True):
                href = a["href"].strip()
                full = urljoin(base_url, href)
                parsed = urlparse(full)
                if (
                    parsed.scheme in ["http", "https"]
                    and parsed.netloc != base_domain
                    and not skip_patterns.search(full)
                    and not is_video_url(full)
                ):
                    url = full
                    break

            if not url:
                for match in url_in_text_pattern.finditer(text):
                    candidate = match.group().rstrip(".,;)")
                    parsed = urlparse(candidate)
                    if (
                        parsed.scheme in ["http", "https"]
                        and parsed.netloc != base_domain
                        and not skip_patterns.search(candidate)
                        and not is_video_url(candidate)
                    ):
                        url = candidate
                        break

            # Require a resolvable URL only in the whole-page fallback (no
            # confirmed reference section): url-less <li> there are usually body
            # bullets that caused false-positive 'refs' (e.g. the FDA page).
            # Inside a confirmed reference section (guarded by has_outbound above)
            # we KEEP url-less references — real citations without links, common
            # on mayoclinic etc.
            if not in_confirmed and not url:
                continue

            clean_text = clean_for_csv(re.sub(r'^\d+[\.\ ]\s*', '', text).strip())
            if not clean_text:
                continue
            if url and url in seen_urls:
                continue
            if url:
                seen_urls.add(url)
            references.append({
                "number": len(references) + 1,
                "text":   clean_text,
                "url":    url,
            })

    # Dedup by text (preserve order)
    seen_texts = set()
    deduped = []
    for ref in references:
        if ref["text"] not in seen_texts:
            seen_texts.add(ref["text"])
            deduped.append(ref)

    # Renumber after dedup
    for i, ref in enumerate(deduped, 1):
        ref["number"] = i

    return deduped


# ── Academic URL detector ─────────────────────────────────────────────────────

ACADEMIC_DOMAINS_RE = re.compile(
    r"(pubmed\.ncbi\.nlm\.nih\.gov|ncbi\.nlm\.nih\.gov/pmc|"
    r"doi\.org|pmc\.ncbi|jamanetwork\.com|nejm\.org|thelancet\.com|"
    r"bmj\.com|nature\.com/articles|science\.org/doi|"
    r"sciencedirect\.com|springerlink\.com|springeropen\.com|"
    r"wiley\.com/doi|academic\.oup\.com|journals\.plos\.org|"
    r"tandfonline\.com|karger\.com|mdpi\.com|frontiersin\.org|"
    r"nih\.gov/|cdc\.gov/mmwr|annals\.org|cell\.com/|jama\.jamanetwork)",
    re.I
)

def is_academic_url(url: str) -> bool:
    return bool(ACADEMIC_DOMAINS_RE.search(url or ""))


# NEW — Inline-link fallback IS now used, but only when formal refs are empty.
# See extract_inline_links_ranked below; called from scrape_page as last-resort citation source.


# ── Academic reference extractor ──────────────────────────────────────────────

def extract_academic_references(soup, base_url):
    skip_re   = re.compile(
        r"(facebook|twitter|instagram|youtube|linkedin|tiktok|"
        r"pinterest|mailto:|javascript:|google\.com/search|amazon\.com|ads\.|doubleclick)",
        re.I
    )
    doi_re    = re.compile(r"\b10\.\d{4,}/\S+")
    pubmed_re = re.compile(r"pubmed\.ncbi\.nlm\.nih\.gov/(\d+)")

    section_label_re = re.compile(
        r"(reference|citation|bibliography|footnote|endnote)", re.I
    )

    ref_section = None
    for tag in soup.find_all(["section", "div", "ol", "ul"]):
        id_str = (
            tag.get("id", "") + " " +
            " ".join(tag.get("class", [])) + " " +
            tag.get("aria-label", "")
        )
        if section_label_re.search(id_str):
            ref_section = tag
            break

    if not ref_section:
        for hdr in soup.find_all(["h2", "h3", "h4", "h5"]):
            if section_label_re.search(hdr.get_text(strip=True)):
                nxt = hdr.find_next_sibling()
                if nxt:
                    ref_section = nxt
                break

    scope      = ref_section if ref_section else soup
    references = []

    for item in scope.find_all(["li", "p"]):
        if len(references) >= 5:
            break

        text = item.get_text(separator=" ", strip=True)
        if len(text) < 30 or len(text.split()) < 5:
            continue

        if not re.search(r"\b(19|20)\d{2}\b", text):
            continue

        url = None

        for a in item.find_all("a", href=True):
            href = a["href"].strip()
            full = urljoin(base_url, href)
            p    = urlparse(full)
            if p.scheme in ("http", "https") and not skip_re.search(full):
                url = full
                break

        if not url:
            m = doi_re.search(text)
            if m:
                url = "https://doi.org/" + m.group().lstrip("/")

        if not url:
            m = pubmed_re.search(text)
            if m:
                url = f"https://pubmed.ncbi.nlm.nih.gov/{m.group(1)}/"

        clean_text = clean_for_csv(re.sub(r"^\d+[.)]\s*", "", text).strip())
        if clean_text:
            references.append({
                "number": len(references) + 1,
                "text":   clean_text,
                "url":    url,
            })

    return references


# ── NEW: Inline-link fallback (used only when formal refs are empty) ─────────
# Pulls outbound <a href> from <main>/<article>/body, filters ads + social,
# ranks academic-leaning links FIRST so real citations beat marketing links.

_ACADEMIC_LIKE_RE = re.compile(
    r"(pubmed\.ncbi|ncbi\.nlm\.nih\.gov|doi\.org|"
    r"jamanetwork|nejm\.org|thelancet|bmj\.com|nature\.com|"
    r"science\.org|sciencedirect|springer|wiley\.com|"
    r"academic\.oup|plos\.org|frontiersin|"
    r"\.gov/|\.edu/|who\.int|cdc\.gov|nih\.gov)",
    re.I
)

_INLINE_SKIP_RE = re.compile(
    r"(facebook|twitter|instagram|youtube|linkedin|tiktok|"
    r"pinterest|mailto:|javascript:|google\.com/search|"
    r"amazon\.com/|ads\.|doubleclick|googlesyndication|"
    r"googleadservices|taboola|outbrain|criteo|mgid|adsense|"
    r"adnxs|adsrvr|adroll|adform|adsystem)",
    re.I
)

def extract_inline_links_ranked(soup, base_url, max_n=5):
    """Fallback when extract_references / extract_academic_references both return empty.
    Grabs hyperlinks embedded in the article PROSE (anchors inside <p>), keeping
    BOTH internal (same-domain) and external links — internal links are often
    legitimate body citations (e.g. an org linking to its own explainer pages).
    Navigation / menu / sidebar / header / footer regions are stripped first so
    menu links never reach the citation slots. Academic, then external, then
    internal links are ranked so real citations win the limited cite slots."""
    base_domain = urlparse(base_url).netloc
    main = soup.find("main") or soup.find("article") or soup.body or soup

    # Work on an isolated copy so stripping nav doesn't mutate the caller's soup.
    try:
        scope = BeautifulSoup(str(main), "lxml")
    except Exception:
        scope = main

    # Strip navigation / chrome regions — by tag, by role, and by id/class hint.
    for _t in scope.find_all(["nav", "header", "footer", "aside"]):
        _t.decompose()
    for _el in scope.find_all(attrs={"role": "navigation"}):
        _el.decompose()
    _navish = re.compile(
        r"(nav|menu|breadcrumb|sidebar|section-nav|\btoc\b|pagination|share|social)",
        re.I,
    )
    for _attr in ("class", "id"):
        for _el in scope.find_all(attrs={_attr: _navish}):
            try:
                _el.decompose()
            except Exception:
                pass

    out, seen = [], set()
    # Only anchors that live inside prose <p> blocks count as "embedded in the text".
    for _p in scope.find_all("p"):
        for a in _p.find_all("a", href=True):
            href = a["href"].strip()
            if not href or href.startswith("#"):
                continue
            full = urljoin(base_url, href)
            p = urlparse(full)
            if p.scheme not in ("http", "https"):
                continue
            if _INLINE_SKIP_RE.search(full):
                continue
            if is_video_url(full):
                continue
            if full in seen:
                continue
            seen.add(full)
            a_class = " ".join(a.get("class", []) or [])
            if re.search(r"\bbtn\b|button", a_class, re.I):
                continue       # CTA / language-toggle buttons aren't body citations
            anchor_text = a.get_text(separator=" ", strip=True)
            if len(anchor_text) < 3:
                continue       # skip bare-URL / icon links
            out.append({
                "number":    len(out) + 1,
                "text":      clean_for_csv(anchor_text),
                "url":       full,
                "_internal": (p.netloc == base_domain),
                "_academic": bool(_ACADEMIC_LIKE_RE.search(full)),
            })
    # Rank academic first, then external, then internal; preserve doc order within.
    out.sort(key=lambda r: (
        0 if r["_academic"] else 1,
        0 if not r["_internal"] else 1,
        r["number"],
    ))
    for r in out:
        r.pop("_academic", None)
        r.pop("_internal", None)
    for i, r in enumerate(out, 1):
        r["number"] = i
    return out[:max_n * 2]   # over-fetch so scrape_cited_sources can dedupe failed ones

# ── Page-quality signals for JAMA / HONcode scoring ──────────────────────────
# The human coders judged the RENDERED page: bylines, "Medically reviewed by"
# lines, "Last updated" labels, footer disclaimers, reference headings. None of
# that survives trafilatura's body-text extraction, so these are read from the
# FULL soup and stored as sig_* columns that health_scorers.py prefers over the
# body text. The regexes live in health_scorers.py (single source of truth).
import sys as _sys_sig
_sig_dir = globals().get("PROJECT_DIR") or os.getcwd()
if _sig_dir not in _sys_sig.path:
    _sys_sig.path.insert(0, _sig_dir)
from health_scorers import (SIG_COLS, DATE_LABEL_RE, BARE_DATE_RE, DISCLAIMER_RE,
                            disclosure_statement, has_credential)

# Cues that introduce an author / reviewer name. The window after the cue is
# scanned for credentials (MD, PA-C, Physician Assistant, ...).
_SIG_BYLINE_CUE_RE = re.compile(
    r"(?:(?<=\s)|^)(?:written\s+by|authored\s+by|author(?:s)?\s*:|contributor(?:s)?\s*:?|"
    r"by)\s*[:\-–—]?\s+(?=[A-Z])", re.I)
_SIG_REVIEWER_CUE_RE = re.compile(
    r"(?:medically|clinically|expert|editorially)?\s*(?:reviewed|verified|fact[- ]checked)\s+by\s*[:\-–—]?\s+(?=[A-Z])",
    re.I)
_SIG_NAME_RE = re.compile(r"^[A-Z][\w'.\-]+(?:\s+(?:[A-Z][\w'.\-]+|[A-Z]\.|de|van|von|del|la))+")
_SIG_ORG_BYLINE_RE = re.compile(
    r"^(?:region|state|google|u\.s\.|the|our|a|an|editorial|staff|team|admin|"
    r"category|topic|health|type|location|date|clicking|using|submitting|"
    r"continuing|checking|phone|email|mail|appointment|visiting)\b", re.I)

_SIG_REF_HEADING_RE = re.compile(
    r"^(?:references?|sources?|citations?|bibliography|works\s+cited|footnotes|"
    r"(?:view|show|see)\s+(?:all\s+)?(?:sources?|references?|citations?)|"
    r"\d+\s+sources?|sources?\s+(?:and|&)\s+references?|references?\s+(?:and|&)\s+sources?|"
    r"article\s+(?:sources?|references?)|scientific\s+references?|"
    r"(?:medical|clinical)\s+references?)\s*[:.]?$", re.I)
_SIG_NUMBERED_ANCHOR_RE = re.compile(r"^\[?\d{1,3}\]?$")
_SIG_NUMBERED_HREF_RE = re.compile(r"#(?:ref|cite|fn|foot|source|note|bib)", re.I)
_SIG_PORTAL_HOST_RE = re.compile(
    r"(login|signin|sign-in|portal|mychart|followmyhealth|patientnotebook|"
    r"appointment|booking|schedul|billpay|bill-pay|payment|careers|"
    r"apps\.apple|play\.google|maps\.google|goo\.gl|bit\.ly|linktr|"
    r"paypal|shopify|eventbrite|surveymonkey|typeform|zocdoc|solvhealth|"
    r"healow|clockwisemd|nextgen|athenahealth|epic\.com)", re.I)
_SIG_TWO_LEVEL_TLDS = {"co.uk", "org.uk", "ac.uk", "com.au", "org.au", "co.nz",
                       "co.jp", "com.br", "co.in", "gov.uk", "gov.au", "edu.au"}


def _sig_registered_domain(host):
    host = (host or "").lower().split(":")[0]
    if host.startswith("www."):
        host = host[4:]
    parts = host.split(".")
    if len(parts) >= 3 and ".".join(parts[-2:]) in _SIG_TWO_LEVEL_TLDS:
        return ".".join(parts[-3:])
    return ".".join(parts[-2:]) if len(parts) >= 2 else host


def _sig_visible_text(soup):
    """Whole-page visible text (scripts/styles dropped, NOTHING else stripped:
    footers and byline blocks are exactly what we want)."""
    try:
        s = BeautifulSoup(str(soup), "lxml")
    except Exception:
        s = soup
    for t in s(["script", "style", "noscript", "svg", "template"]):
        t.decompose()
    return re.sub(r"\s+", " ", s.get_text(" ")).strip(), s


def _sig_jsonld_credentialed(soup):
    """author / reviewedBy in JSON-LD carrying jobTitle / honorificSuffix /
    a credential in the name."""
    for script in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(script.string or "{}")
        except Exception:
            continue
        items = data if isinstance(data, list) else [data]
        stack = list(items)
        while stack:
            it = stack.pop()
            if isinstance(it, list):
                stack.extend(it); continue
            if not isinstance(it, dict):
                continue
            if "@graph" in it and isinstance(it["@graph"], list):
                stack.extend(it["@graph"])
            for key in ("author", "reviewedBy", "medicalReviewer", "contributor", "editor"):
                val = it.get(key)
                for p in (val if isinstance(val, list) else [val]):
                    if isinstance(p, dict):
                        blob = " ".join(as_text(p.get(k, "")) for k in
                                        ("name", "jobTitle", "honorificSuffix",
                                         "honorificPrefix", "description"))
                        if p.get("@type") == "Person" and has_credential(blob):
                            return True
                        if as_text(p.get("honorificSuffix")) or \
                           re.search(r"\b(MD|DO|PhD|RN|NP|PA-C)\b", as_text(p.get("jobTitle"))):
                            return True
                    elif isinstance(p, str) and has_credential(p):
                        return True
    return False


def extract_quality_signals(soup, url):
    """Page-level cues the JAMA/HON coders relied on. Every value is a CSV-safe
    string or int; all keys in SIG_COLS are always present."""
    out = {k: "" for k in SIG_COLS}
    out["sig_credentialed"] = 0
    out["sig_ref_section"] = 0
    out["sig_n_numbered_cites"] = 0
    out["sig_n_external_links"] = 0
    try:
        full, clean = _sig_visible_text(soup)
    except Exception:
        return out
    if not full:
        return out

    # ── byline / reviewer + credentials ──────────────────────────────────
    credentialed = False
    byline, reviewer = "", ""
    for m in _SIG_BYLINE_CUE_RE.finditer(full):
        window = full[m.end():m.end() + 90]
        nm = _SIG_NAME_RE.match(window)
        if not nm or _SIG_ORG_BYLINE_RE.match(window):
            continue
        cand = full[m.start():m.end()] + window
        if not byline:
            byline = cand[:120]
        if has_credential(window):
            credentialed = True
            byline = cand[:120]
            break
    for m in _SIG_REVIEWER_CUE_RE.finditer(full):
        window = full[m.end():m.end() + 90]
        if not _SIG_NAME_RE.match(window):
            continue
        cand = full[m.start():m.end()] + window
        if not reviewer:
            reviewer = cand[:120]
        if has_credential(window):
            credentialed = True
            reviewer = cand[:120]
            break
    if not credentialed:
        # "Contributor Ronak Shah, MD" (Mass General) — cue without a colon.
        for m in re.finditer(r"\bcontributors?\b", full, re.I):
            if has_credential(full[m.end():m.end() + 90]):
                credentialed = True
                byline = byline or full[m.start():m.start() + 120]
                break
    if not credentialed:
        try:
            credentialed = _sig_jsonld_credentialed(soup)
        except Exception:
            pass
    out["sig_byline"] = clean_for_csv(byline)
    out["sig_reviewer"] = clean_for_csv(reviewer)
    out["sig_credentialed"] = int(credentialed)

    # ── visible labeled date ─────────────────────────────────────────────
    dm = DATE_LABEL_RE.search(full)
    if dm:
        out["sig_date_label"] = clean_for_csv(dm.group(0)[:80])
    # Bare date the reader can SEE at the top of the article (byline zone).
    # Dates that exist only in JSON-LD / meta tags scored 0 with the coders.
    _main0 = clean.find("article") or clean.find("main") or clean.body or clean
    _top = re.sub(r"\s+", " ", _main0.get_text(" "))[:1200]
    for bm in BARE_DATE_RE.finditer(_top):
        if re.search(r"(©|copyright|\(c\))\s*$", _top[max(0, bm.start() - 15):bm.start()], re.I):
            continue
        out["sig_date_visible"] = clean_for_csv(bm.group(0)[:40])
        break

    # ── complementarity disclaimer (body or footer) ──────────────────────
    cm = DISCLAIMER_RE.search(full)
    if cm:
        s, e = max(0, cm.start() - 60), min(len(full), cm.end() + 100)
        out["sig_disclaimer"] = clean_for_csv(full[s:e])

    # ── disclosure STATEMENT (>= 6-word sentence, not a nav link) ────────
    out["sig_disclosure"] = clean_for_csv(disclosure_statement(full))

    # ── reference section heading ────────────────────────────────────────
    ref_section = 0
    for tag in clean.find_all(["h1", "h2", "h3", "h4", "h5", "h6", "summary",
                               "button", "strong", "b", "span", "p", "div", "a"]):
        # own text only (skip containers that hold whole sections)
        own = tag.get_text(" ", strip=True)
        if not own or len(own) > 40 or len(own.split()) > 4:
            continue
        if _SIG_REF_HEADING_RE.match(own):
            ref_section = 1
            break
    out["sig_ref_section"] = ref_section

    # ── numbered citation links ([1], superscripts, #ref anchors) ────────
    n_num = 0
    for a in clean.find_all("a", href=True):
        txt = a.get_text("", strip=True)
        if _SIG_NUMBERED_ANCHOR_RE.match(txt) or \
           (a.parent is not None and a.parent.name == "sup") or \
           _SIG_NUMBERED_HREF_RE.search(a["href"]):
            n_num += 1
    out["sig_n_numbered_cites"] = n_num

    # ── external source links inside the article prose ───────────────────
    base_dom = _sig_registered_domain(urlparse(url).netloc)
    main = clean.find("article") or clean.find("main") or clean.body or clean
    try:
        scope = BeautifulSoup(str(main), "lxml")
    except Exception:
        scope = main
    for _t in scope.find_all(["nav", "header", "footer", "aside", "form"]):
        _t.decompose()
    for _el in scope.find_all(attrs={"role": "navigation"}):
        _el.decompose()
    _navish_sig = re.compile(
        r"(nav|menu|breadcrumb|sidebar|section-nav|\btoc\b|pagination|share|social|"
        r"related|recommended|promo|footer|header|comment)", re.I)
    for _attr in ("class", "id"):
        for _el in scope.find_all(attrs={_attr: _navish_sig}):
            try:
                _el.decompose()
            except Exception:
                pass
    ext = set()
    for blk in scope.find_all(["p", "li", "td", "blockquote"]):
        for a in blk.find_all("a", href=True):
            href = a["href"].strip()
            if not href or href.startswith(("#", "mailto:", "tel:", "javascript:")):
                continue
            full_u = urljoin(url, href)
            pu = urlparse(full_u)
            if pu.scheme not in ("http", "https") or not pu.netloc:
                continue
            dom = _sig_registered_domain(pu.netloc)
            if not dom or dom == base_dom:
                continue
            if _INLINE_SKIP_RE.search(full_u) or is_video_url(full_u) or \
               _SIG_PORTAL_HOST_RE.search(full_u):
                continue
            ext.add(full_u.split("#", 1)[0])
    out["sig_n_external_links"] = len(ext)
    return out


def scrape_page(url):
    result = {
        "url":          url,
        "text":         "",
        "authors":      "",
        "pub_date":     "",
        "updated_date": "",
        "disclosures":  "",
        "references":   [],
        "error":        "",
        "fetch_method": "",
        "text_quality": "",
    }
    # Page-quality signals (see extract_quality_signals); empty for PDFs.
    result.update({k: "" for k in SIG_COLS})
    for _k in ("sig_credentialed", "sig_ref_section",
               "sig_n_numbered_cites", "sig_n_external_links"):
        result[_k] = 0

    html, method, fetch_err = _cached_fetch(url)
    result["fetch_method"] = method
    if not html:
        result["error"] = f"fetch_failed:{fetch_err}" if fetch_err else "fetch_failed"
        result["text_quality"] = "empty"
        return result

    # PDFs return bytes — no soup-based metadata is possible, just text
    if isinstance(html, bytes):
        result["text"] = extract_main_text(html, url)
        result["text_quality"] = assess_text_quality(result["text"])
        if not result["text"]:
            result["error"] = "pdf_extract_empty"
        else:
            # Editorial-team author fallback still works without a soup
            try:
                empty_soup = BeautifulSoup("<html><body></body></html>", "lxml")
                result["authors"] = extract_authors(empty_soup, url=url)
            except Exception:
                pass
        # NEW — updated_date fallback for PDF branch (no soup available)
        if not result["updated_date"]:
            result["updated_date"] = f"{datetime.now().strftime('%Y-%m-%d')}_current"
        return result

    soup = BeautifulSoup(html, "lxml")
    result["text"]         = extract_main_text(html, url)
    # Pass URL so editorial fallback can identify the org name
    result["authors"]      = extract_authors(soup, url=url)
    result["pub_date"], \
    result["updated_date"] = extract_dates(soup)
    # NEW — updated_date fallback: use today's access date when none on page
    if not result["updated_date"]:
        result["updated_date"] = f"{datetime.now().strftime('%Y-%m-%d')}_current"
    result["disclosures"]  = extract_disclosures(soup)
    # Full-page cues for JAMA/HON scoring (bylines, reviewer lines, date
    # labels, disclaimers, disclosure statements, reference headings, links).
    try:
        result.update(extract_quality_signals(soup, url))
    except Exception as _sig_err:
        print(f"    ⚠ quality-signal extraction failed: {type(_sig_err).__name__}: {str(_sig_err)[:60]}")

    if is_academic_url(url):
        print(f"    ℹ Academic URL detected — using academic reference extractor")
        refs = extract_academic_references(soup, url) or extract_references(soup, url)
    else:
        refs = extract_references(soup, url)

    # NEW — when formal refs are empty, fall back to ranked inline links
    if not refs:
        refs = extract_inline_links_ranked(soup, url)
        if refs:
            print(f"    ℹ Formal refs empty — using {len(refs)} inline-link fallback citations")

    result["references"] = refs

    # Flag empty/short extractions so the CSV explains the NaN instead of hiding it
    if not result["text"]:
        result["error"] = "extraction_empty"
    elif len(result["text"]) < 300:
        result["error"] = "extraction_short"

    # Final content-quality verdict on the extracted text — downstream scoring
    # (Halloumi) only trusts sources whose text_quality is 'ok'.
    result["text_quality"] = assess_text_quality(result["text"])

    return result

# ── Live test ─────────────────────────────────────────────────────────────────
if TEST_URL:
    print(f"Scraping: {TEST_URL}\n")
    r = scrape_page(TEST_URL)

    print(f"Text ({len(r['text'])} chars):")
    print(f"  {r['text'][:500]}..." if r['text'] else "  (none)")
    print(f"\nAuthors:     {r['authors'] or '(none)'}")
    # NEW — indicate if editorial fallback was used
    if r['authors'] and 'editorial team' in r['authors']:
        print(f"  ℹ (editorial fallback — no named author found on page)")
    print(f"Published:   {r['pub_date'] or '(none)'}")
    print(f"Updated:     {r['updated_date'] or '(none)'}")
    print(f"Disclosures: {r['disclosures'][:200] or '(none)'}")
    print("Page signals:")
    for _k in SIG_COLS:
        print(f"  {_k:22s} {str(r.get(_k, ''))[:90]}")
    print(f"\nReferences ({len(r['references'])}):")
    for ref in r['references']:
        url_str = f" → {ref['url']}" if ref['url'] else " (no link)"
        print(f"  [{ref['number']}] {ref['text'][:100]}{url_str}")
    print("\n✅ Step 5 complete.")

    # ── NEW: Quick editorial-fallback test ────────────────────────────────────
    print("\n── Editorial author fallback test (force no-author scenario) ──")
    from bs4 import BeautifulSoup as _BS
    _empty_soup = _BS("<html><body></body></html>", "lxml")
    _test_urls = [
        "https://www.heart.org/en/health-topics/heart-attack",
        "https://www.cdc.gov/heartdisease/index.htm",
        "https://www.healthline.com/health/heart-disease",
        "https://www.unknownsite.org/articles/test",
    ]
    for _u in _test_urls:
        _result = extract_authors(_empty_soup, url=_u)
        print(f"  {_u[:55]}")
        print(f"    → '{_result}'")


In [ ]:
# @title 🔗 STEP 6 — Cited Source Scraper (1 Level Deep, keep at 5 max citations) { display-mode: "form" }
# @markdown Scrapes every reference URL found on a page and pulls the same fields.
# @markdown Goes ONE level deep only — does not follow citations of citations.

MAX_CITATIONS_PER_PAGE = 5  # @param {type:"slider", min:1, max:20, step:1}

import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Non-article URL filter ───────────────────────────────────────────────────

_SKIP_CITATION_PATTERNS = re.compile(
    r"(/search(/|$)|/contents/search|uptodate\.com/contents/search"
    r"|wikipedia\.org/wiki/Wikipedia:|/index\.html?$"
    r"|amazon\.com|ebay\.|/shop/?$|/store/?$)",
    re.I,
)
_HOMEPAGE_RE = re.compile(r"^https?://[^/]+/?$")

def _is_skippable_citation(url):
    """Return True for search pages, homepages, and other clearly non-article URLs."""
    if not url:
        return True
    if _HOMEPAGE_RE.match(url):
        return True
    if _SKIP_CITATION_PATTERNS.search(url):
        return True
    return False

def scrape_cited_sources(references, max_n=5, workers=4):
    """
    Concurrently scrape up to `max_n` citation URLs through the full scrape_page
    pipeline so cite{i}_text gets the FULL body text (not a snippet) and
    cite{i}_authors / pub_date / updated_date / disclosures get populated.

    Over-fetches up to 2x to compensate for fetch failures, then trims to max_n.

    Accepts either:
      - new format: list of dicts {"number", "text", "url"}
      - old format: list of plain URL strings (backwards compatible)
    """
    # Build deduped queue of candidate (url, ref_dict) pairs
    queue = []
    seen = set()
    pdf_skipped_count = 0   # R5 - track PDFs filtered from cite slots
    for ref in references:
        if isinstance(ref, dict):
            url = ref.get("url")
            ref_text = ref.get("text", "")
        else:
            url, ref_text = ref, ""
        if not url or url in seen:
            continue
        if _is_skippable_citation(url):
            print(f"    ↷ Skipping (non-article URL): {url[:70]}")
            continue
        # R5 - filter PDF citation URLs (PDFs still allowed at source level)
        if is_pdf_url(url):
            pdf_skipped_count += 1
            print(f"    - Skipping PDF citation: {url[:70]}")
            continue
        seen.add(url)
        queue.append((url, ref_text))
        if len(queue) >= max_n * 2:
            break

    if not queue:
        # R5 - even with no scraped results, expose skipped count via a marker
        if pdf_skipped_count:
            return [{"_pdf_skipped_count": pdf_skipped_count}]
        return []

    print(f"    Scraping {len(queue)} citation URL(s) concurrently (max_n={max_n})")
    results_by_url = {}
    with ThreadPoolExecutor(max_workers=min(workers, len(queue))) as ex:
        futures = {ex.submit(scrape_page, url): (url, ref_text)
                   for url, ref_text in queue}
        for fut in as_completed(futures):
            url, ref_text = futures[fut]
            try:
                data = fut.result()
            except Exception as e:
                data = {"url": url, "text": "", "authors": "", "pub_date": "",
                        "updated_date": "", "disclosures": "", "references": [],
                        "error": f"exception:{type(e).__name__}:{str(e)[:80]}",
                        "fetch_method": ""}
            data["ref_text"] = ref_text
            results_by_url[url] = data

    # Preserve original reference order, then trim to max_n
    ordered = [results_by_url[url] for url, _ in queue if url in results_by_url]
    trimmed = ordered[:max_n]
    # R5 - stash pdf_skipped_count on the first result so flatten_citations can emit cite1_is_pdf_skipped
    if trimmed and pdf_skipped_count:
        trimmed[0]["_pdf_skipped_count"] = pdf_skipped_count
    return trimmed

# Preview: scrape first citations from Step 5's test page
if 'r' not in dir():
    print("⚠️  Run Step 5 first.")
else:
    # Support both old cited_urls and new references key
    refs = r.get('references') or r.get('cited_urls') or []

    if not refs:
        print("⚠️  No references found on the Step 5 test page.")
    else:
        # Count how many actually have links
        has_link = [ref for ref in refs if (ref.get('url') if isinstance(ref, dict) else ref)]
        print(f"References found:   {len(refs)}")
        print(f"With links:         {len(has_link)}")
        print(f"Scraping first {min(MAX_CITATIONS_PER_PAGE, len(has_link))} linked references...\n")

        citation_results = scrape_cited_sources(refs, max_n=MAX_CITATIONS_PER_PAGE)

        print("\nRESULTS:")
        for i, cr in enumerate(citation_results):
            # scrape_cited_sources can return a marker dict when every candidate
            # citation was filtered out (e.g. all PDFs) — it has no scrape fields.
            if "_pdf_skipped_count" in cr:
                print(f"\n  (no scrapable citations — {cr['_pdf_skipped_count']} PDF citation(s) skipped)")
                continue
            print(f"\n  Citation [{i+1}]: {cr.get('url', '(no url)')}")
            if cr.get('ref_text'):
                print(f"    Ref text:  {cr['ref_text'][:80]}")
            print(f"    Authors:   {(cr.get('authors') or '')[:60] or '(none)'}")
            print(f"    Published: {cr.get('pub_date') or '(none)'}")
            print(f"    Text len:  {len(cr.get('text') or '')} chars")
            print(f"    Error:     {cr.get('error') or 'none'}")

        print("\n✅ Step 6 complete.")

In [ ]:
# @title 🗣️ STEP 7 — Forum & Blog Handler (Reddit, Quora, etc.) { display-mode: "form" }
# @markdown Grabs: first/top post text + top 3 HEAD-level comments only (no reply threads).
# @markdown Reddit uses Arctic Shift archive API (free, no auth).
# @markdown Quora is flagged for optional SerpAPI retry — link is always saved.
# @markdown Posts too new to be archived are flagged for optional SerpAPI retry.

TEST_FORUM_URL = "https://www.reddit.com/r/NoStupidQuestions/comments/1ltvg6n/how_do_white_people_avoid_sunburn_how_do_they/"  # @param {type:"string"}

import requests
import trafilatura
import re
import json
import random
import time
from bs4 import BeautifulSoup

# ── Forum domain list — health-relevant only ──────────────────────────────────
FORUM_DOMAINS = [
    "reddit.com",
    "quora.com",
    "patient.info",
    "healthunlocked.com",
    "medhelp.org",
    "inspire.com",
    "drugs.com/comments",
    "webmd.com/answers",
    "forums.webmd.com",
    "community.patient.co.uk",
]

def is_forum_url(url):
    return any(d in url for d in FORUM_DOMAINS)

# ── Arctic Shift Reddit fetcher ───────────────────────────────────────────────

ARCTIC_BASE = "https://arctic-shift.photon-reddit.com"

def extract_reddit_post_id(url):
    match = re.search(r'/comments/([a-z0-9]+)/', url, re.IGNORECASE)
    return match.group(1) if match else None

def fetch_reddit_via_arctic(url, timeout=15):
    """
    Fetches a Reddit post + top comments via Arctic Shift.
    Uses the /comments/tree endpoint which returns all comments
    in tree structure — we take top-level only and sort by score.
    """
    post_id = extract_reddit_post_id(url)
    if not post_id:
        print(f"    ⚠ Could not extract Reddit post ID from URL")
        return None

    headers = {
        "User-Agent": "research-scraper/1.0 (academic use)",
        "Accept":     "application/json",
    }

    try:
        # Fetch post
        post_resp = requests.get(
            f"{ARCTIC_BASE}/api/posts/ids",
            params={"ids": post_id},
            headers=headers,
            timeout=timeout
        )
        post_resp.raise_for_status()
        posts = post_resp.json().get("data", [])

        if not posts:
            print(f"    ⚠ Arctic Shift: post not found (may be too recent)")
            return {"not_found": True}

        # Fetch comment tree — top-level only, sorted by score client-side
        comments_resp = requests.get(
            f"{ARCTIC_BASE}/api/comments/tree",
            params={
                "link_id": f"t3_{post_id}",
                "limit":   9999,
            },
            headers=headers,
            timeout=timeout
        )
        comments_resp.raise_for_status()
        tree_data = comments_resp.json().get("data", [])

        # Each item is {"kind": "t1", "data": {...}} — unwrap the data envelope
        # Filter to top-level only: parent_id starts with t3_ (direct reply to post)
        top_level = []
        for item in tree_data:
            if item.get("kind") != "t1":
                continue
            c = item.get("data", {})
            parent_id = c.get("parent_id", "")
            body      = c.get("body", "").strip()
            if (
                parent_id.startswith("t3_")
                and not c.get("stickied")
                and body not in ("", "[removed]", "[deleted]")
            ):
                top_level.append(c)

        print(f"    Top-level comments found: {len(top_level)}")

        # Sort by score descending so most upvoted come first
        top_level.sort(key=lambda c: int(c.get("score", 0) or 0), reverse=True)

        return {
            "post":     posts[0],
            "comments": top_level,
        }

    except Exception as e:
        print(f"    ⚠ Arctic Shift fetch failed: {e}")
        return None

# ── Main forum scraper ────────────────────────────────────────────────────────

def scrape_forum(url):
    result = {
        "url":           url,    # always saved regardless of scrape success
        "post_text":     "",
        "comments":      [],     # list of {"author", "score", "text"} dicts
        "authors":       "",
        "pub_date":      "",
        "error":         "",
        "serpapi_retry": False,
    }

    # ── REDDIT ────────────────────────────────────────────────────────────────
    if "reddit.com" in url:
        data = fetch_reddit_via_arctic(url)

        if data and data.get("not_found"):
            print(f"    ⚠ Post not in Arctic Shift archive (too recent). Flagged for SerpAPI retry.")
            result["error"]         = "not_in_archive"
            result["serpapi_retry"] = True
            return result

        if not data:
            result["error"]         = "fetch_failed"
            result["serpapi_retry"] = True
            return result

        try:
            post = data["post"]

            selftext = (post.get("selftext") or "").strip()
            title    = (post.get("title") or "").strip()
            result["post_text"] = (
                selftext if selftext and selftext not in ("[removed]", "[deleted]")
                else title
            )
            result["authors"]  = post.get("author", "")

            # Fix 6: Convert Unix timestamp to ISO date string
            raw_ts = post.get("created_utc", "")
            if raw_ts:
                try:
                    from datetime import datetime, timezone as _tz
                    result["pub_date"] = datetime.fromtimestamp(
                        int(raw_ts), tz=_tz.utc
                    ).strftime("%Y-%m-%dT%H:%M:%SZ")
                except (ValueError, TypeError):
                    result["pub_date"] = str(raw_ts)
            else:
                result["pub_date"] = ""

            # Fix 3: Store comments as list of dicts (matches flatten_forum expectation)
            result["comments"] = [
                {
                    "author": c.get("author", ""),
                    "score":  c.get("score", 0),
                    "text":   c.get("body", "").strip(),
                }
                for c in data["comments"][:5]
            ]

        except Exception as e:
            print(f"    ⚠ Reddit parse error: {e}")
            result["error"]         = f"parse_error: {e}"
            result["serpapi_retry"] = True

        return result

    # ── QUORA ─────────────────────────────────────────────────────────────────
    # Quora is JS-rendered and blocks scrapers — always flag for retry
    # but the URL is already saved in result so it's never lost
    elif "quora.com" in url:
        print(f"    ⚠ Quora blocks scraping — URL saved, flagged for SerpAPI retry.")
        result["error"]         = "quora_blocked"
        result["serpapi_retry"] = True
        return result

    # ── GENERIC HEALTH FORUM ──────────────────────────────────────────────────
    else:
        # fetch_html returns (html, method, error) — unpacking the tuple here.
        # Passing the whole tuple into trafilatura raised an exception that
        # killed the ENTIRE question row upstream.
        html, fetch_method, fetch_err = fetch_html(url)
        if not html or not isinstance(html, str):
            # bytes (PDF) can't be parsed as a forum page either
            result["error"]         = f"fetch_failed:{fetch_err}" if fetch_err else "fetch_failed"
            result["serpapi_retry"] = True
            return result

        text   = trafilatura.extract(html, favor_precision=False) or ""
        blocks = [b.strip() for b in text.split("\n\n") if len(b.strip()) > 30]

        if blocks:
            result["post_text"] = blocks[0]
        # Fix 3: Store comments as list of dicts (matches flatten_forum expectation)
        result["comments"] = [
            {"author": "", "score": "", "text": block}
            for block in blocks[1:4]
        ]

        soup = BeautifulSoup(html, "lxml")
        result["authors"]  = extract_authors(soup)
        result["pub_date"], _ = extract_dates(soup)

    return result

# ── Live test ─────────────────────────────────────────────────────────────────
if TEST_FORUM_URL:
    print(f"Scraping forum: {TEST_FORUM_URL}\n")
    fr = scrape_forum(TEST_FORUM_URL)

    print(f"URL saved:  {fr['url']}")
    print(f"\nPost text ({len(fr['post_text'])} chars):")
    print(f"  {fr['post_text'][:300]}..." if fr['post_text'] else "  (none)")
    print(f"\nAuthor:    {fr['authors'] or '(none)'}")
    print(f"Date:      {fr['pub_date'] or '(none)'}")
    for _ci, _c in enumerate(fr.get('comments', [])[:3], 1):
        _txt = _c.get('text', '') if isinstance(_c, dict) else str(_c)
        _auth = _c.get('author', '') if isinstance(_c, dict) else ''
        print(f"Comment {_ci}: {_txt[:150] or '(none)'}" + (f" [by {_auth}]" if _auth else ""))
    if fr["error"]:
        print(f"\n⚠ Error: {fr['error']}")
    if fr["serpapi_retry"]:
        print(f"🔁 Flagged for optional SerpAPI retry")
    print("\n✅ Step 7 complete.")

In [ ]:
#Step 8 runs Steps 3-7 which test/preview

In [ ]:
# @title 🚀 STEP 8 — Full Pipeline: All Questions { display-mode: "form" }
# @markdown Runs all questions through SerpAPI → scrape → citations → forum.
# @markdown Results saved after every question. Downloads CSV at the end.

PROJECT_DIR = r"C:/Users/SAgno/Documents/AIO_Workspace/LocalRun"  # @param {type:"string"}
OUTPUT_FILE = "research_output.csv"  # @param {type:"string"}

import pandas as pd
import csv
import json
import re
from datetime import date

# --- Persist output to Drive so the pipeline survives a runtime restart ---
import os
os.makedirs(PROJECT_DIR, exist_ok=True)
# Each batch gets its own file (research_output_batch01.csv, _batch02.csv, ...)
# so a crash mid-batch never touches earlier batches' saved work, and Step 2.5
# can tell which batches are already done just by looking at what exists.
_out_base, _out_ext = os.path.splitext(OUTPUT_FILE)
_batch_num_for_file = globals().get("CURRENT_BATCH_NUM")
OUTPUT_FILE = os.path.join(
    PROJECT_DIR,
    f"{_out_base}_batch{_batch_num_for_file:02d}{_out_ext}"
    if _batch_num_for_file is not None else OUTPUT_FILE)
print(f"Output will be written to: {OUTPUT_FILE}")

# ── Canonical forum/video domain lists ───────────────────────────────────────

FORUM_DOMAINS = [
    "reddit.com",
    "quora.com",
    "patient.info",
    "healthunlocked.com",
    "medhelp.org",
    "inspire.com",
    "drugs.com/comments",
    "webmd.com/answers",
    "forums.webmd.com",
    "community.patient.co.uk",
    "connect.mayoclinic.org",
    "inspire.com",
]

VIDEO_DOMAINS = [
    "youtube.com", "youtu.be", "vimeo.com",
    "dailymotion.com", "tiktok.com", "instagram.com/reel",
]

def is_forum_url(url: str) -> bool:
    return any(d in url for d in FORUM_DOMAINS)

def is_video_url(url: str) -> bool:
    return any(d in url for d in VIDEO_DOMAINS)

# ── Page-quality signal columns (sig_*) ───────────────────────────────────────
# Defined in health_scorers.py and populated by Step 5's scrape_page; carried
# here per source slot and per citation slot so Step 9 / Steps 11-12 see them.
try:
    from health_scorers import SIG_COLS
except ImportError:
    SIG_COLS = globals().get("SIG_COLS") or [
        "sig_byline", "sig_reviewer", "sig_credentialed", "sig_date_label", "sig_date_visible",
        "sig_disclaimer", "sig_disclosure", "sig_ref_section",
        "sig_n_numbered_cites", "sig_n_external_links"]

# ── Fragment stripper ─────────────────────────────────────────────────────────

def _strip_fragment(url: str) -> str:
    return url.split('#')[0] if url else url

# ── CSV-safe helpers ──────────────────────────────────────────────────────────

def _csv_clean(val):
    if not isinstance(val, str):
        return val
    val = re.sub(r'[\r\n]+', ' ', val)
    val = val.replace('\t', ' ')
    val = val.replace('"', "'")
    val = re.sub(r'  +', ' ', val)
    return val.strip()

def _clean_dict(d):
    out = {}
    for k, v in d.items():
        if isinstance(v, str):
            out[k] = _csv_clean(v)
        elif isinstance(v, list):
            out[k] = [_clean_dict(i) if isinstance(i, dict) else (_csv_clean(i) if isinstance(i, str) else i) for i in v]
        elif isinstance(v, dict):
            out[k] = _clean_dict(v)
        else:
            out[k] = v
    return out

# ── Save helper ───────────────────────────────────────────────────────────────

def save_results(rows, filename):
    if not rows:
        print("  ⚠ No rows to save.")
        return
    df = pd.DataFrame(rows)
    df.to_csv(
        filename,
        index=False,
        encoding='utf-8-sig',
        quoting=csv.QUOTE_ALL,
        lineterminator='\n',
    )
    print(f"  💾 Saved {len(df)} rows × {len(df.columns)} cols → {filename}")

# ── Forum flattening ──────────────────────────────────────────────────────────

def flatten_forum(forum_data):
    if not forum_data:
        return {}
    flat = {}
    for k, v in forum_data.items():
        if k == "comments":
            for i, c in enumerate(v[:5], 1):
                flat[f"forum_comment{i}_author"] = _csv_clean(str(c.get("author", "")))
                flat[f"forum_comment{i}_score"]  = c.get("score", "")
                flat[f"forum_comment{i}_text"]   = _csv_clean(str(c.get("text", ""))[:500])
        elif k == "post_text":
            flat["forum_post_text"] = _csv_clean(str(v)[:1000])
        elif k == "text":
            flat["forum_text"] = _csv_clean(str(v)[:1000])
        elif isinstance(v, str):
            flat[f"forum_{k}"] = _csv_clean(v)
        else:
            flat[f"forum_{k}"] = v
    return flat

# ── Citation flattening ───────────────────────────────────────────────────────

def flatten_citations(cited_sources, prefix, max_n=5):
    """
    Emit columns for up to max_n citations. Always emits the full set of fields
    for every slot (even if empty) so the downstream reshape in Step 9 finds
    consistent column names across rows.

    Previous version omitted pub_date/updated_date/disclosures and named the
    pub_date column `_date` — Step 9 reads `_pub_date`, which is why those
    columns were 100% NaN in the final datasets.

    R5: reads _pdf_skipped_count stashed by scrape_cited_sources; emits
    {prefix}_cite1_is_pdf_skipped flag. Also falls back to ref_text when
    scraped text is empty (snippet fallback for cite slots).
    """
    flat = {}
    # R5 - read pdf_skipped_count stashed on first result by scrape_cited_sources
    first = cited_sources[0] if cited_sources else {}
    pdf_skipped_count = first.get("_pdf_skipped_count", 0)
    for i in range(1, max_n + 1):
        p = f"{prefix}_cite{i}"
        src = cited_sources[i - 1] if i - 1 < len(cited_sources) else {}
        # R5 - sentinel dicts (no URL, only internal _keys) are treated as empty slots
        if src and not src.get("url") and all(k.startswith("_") for k in src):
            src = {}
        cite_text = _csv_clean(str(src.get("text", "")))
        cite_quality = _csv_clean(str(src.get("text_quality", "")))
        # R5 - snippet fallback: use ref_text when full scrape produced nothing
        if not cite_text and src.get("ref_text"):
            cite_text = _csv_clean(str(src.get("ref_text", ""))[:1000])
            cite_quality = "snippet_only"
        flat[f"{p}_url"]          = _csv_clean(str(src.get("url", "")))
        flat[f"{p}_ref_text"]     = _csv_clean(str(src.get("ref_text", ""))[:300])
        flat[f"{p}_text"]         = cite_text
        flat[f"{p}_authors"]      = _csv_clean(str(src.get("authors", "")))
        flat[f"{p}_pub_date"]     = _csv_clean(str(src.get("pub_date", "")))
        flat[f"{p}_updated_date"] = _csv_clean(str(src.get("updated_date", "")))
        flat[f"{p}_disclosures"]  = _csv_clean(str(src.get("disclosures", "")))
        flat[f"{p}_error"]        = _csv_clean(str(src.get("error", "")))
        flat[f"{p}_fetch_method"] = _csv_clean(str(src.get("fetch_method", "")))
        flat[f"{p}_text_quality"] = cite_quality
        for _k in SIG_COLS:                      # page-quality signals per cite
            flat[f"{p}_{_k}"] = _csv_clean(str(src.get(_k, "")))
    # R5 - PDF skip flag: True when >=1 PDF was shifted past in cite slots
    flat[f"{prefix}_cite1_is_pdf_skipped"] = bool(pdf_skipped_count)
    return flat

# ── NEW: AIO statement flattening ─────────────────────────────────────────────

def flatten_aio_statements(aio_statements):
    """
    Convert the list of {statement, source_title, source_url} dicts from
    parse_serpapi_result into flat column pairs for the output CSV.

    Produces columns:
        aio_statement_1          — text of the 1st cited statement
        aio_statement_1_source   — source title for that statement
        aio_statement_1_source_url — source URL for that statement
        aio_statement_2          — ...
        aio_statement_2_source
        aio_statement_2_source_url
        ... (up to however many statements are present)

    Only statements that have at least a non-empty snippet are included.
    Duplicate (statement, source_url) pairs are deduplicated so the same
    statement does not appear twice if SerpAPI returns duplicate references.
    """
    flat = {}
    seen = set()
    col_num = 0

    for stmt in aio_statements:
        statement   = _csv_clean(stmt.get("statement", ""))
        source_title = _csv_clean(stmt.get("source_title", ""))
        source_url   = _csv_clean(stmt.get("source_url", ""))

        if not statement:
            continue

        # Deduplicate on (statement text, source URL) pair
        dedup_key = (statement[:200], source_url)
        if dedup_key in seen:
            continue
        seen.add(dedup_key)

        col_num += 1
        flat[f"aio_statement_{col_num}"]            = statement
        flat[f"aio_statement_{col_num}_source"]     = source_title
        flat[f"aio_statement_{col_num}_source_url"] = source_url

    return flat


# ── NEW: Grouped statement flattening with citation-index ranges ─────────────

def flatten_aio_statements_grouped(grouped, max_n=15):
    """Emit columns:
        aio_statement{N}_text       — the statement snippet
        aio_statement{N}_citations  — citation indexes formatted as '1:4' or '1,3,5'
                                       (1-based; matches aio_src1, aio_src2, ...)"""
    flat = {}
    seen = set()
    col = 0
    for item in grouped:
        stmt = _csv_clean(item.get("statement", ""))
        idxs = item.get("citation_indexes", [])
        if not stmt or not idxs:
            continue
        key = (stmt[:200], tuple(idxs))
        if key in seen:
            continue
        seen.add(key)
        col += 1
        if col > max_n:
            break
        flat[f"aio_statement{col}_text"]      = stmt
        flat[f"aio_statement{col}_citations"] = _format_citation_range(idxs)
    return flat

# ── Statement→source matcher (token containment) ─────────────────────────────
# Fallback attribution when SerpAPI exposes no inline references. The previous
# rapidfuzz partial_ratio>=75 over the first 3,000 chars never matched Google's
# paraphrased AIO text (0 hits on real runs); vocabulary containment over the
# FULL source text does.

_STMT_STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "to", "in", "for", "on", "with",
    "as", "by", "at", "is", "are", "be", "was", "were", "that", "this",
    "it", "its", "your", "you", "can", "may", "from", "not", "no", "if",
    "but", "have", "has", "such", "also", "these", "those", "their", "they",
}

def _stmt_content_tokens(s):
    """Lowercased alphanumeric tokens minus stopwords."""
    return {t for t in re.findall(r"[a-z0-9]+", s.lower())
            if t not in _STMT_STOPWORDS and len(t) > 1}

def _split_aio_chunks(aio_text, min_words=8):
    """Split AIO text into statement chunks. AIO text arrives newline-free
    (both _parse_text_blocks and CSV cleaning join on spaces), so paragraphs
    don't exist — split on bullet markers and sentence boundaries instead."""
    chunks = []
    for seg in re.split(r"\s+•\s+|\n{2,}", aio_text):
        for sent in re.split(r"(?<=[.!?])\s+", seg.strip()):
            sent = sent.strip(" •")
            if len(sent.split()) >= min_words:
                chunks.append(sent)
    return chunks

def match_statements_to_sources(aio_text, src_texts,
                                threshold=0.6, top_k=3, max_statements=15):
    """
    Attribute AIO statements to scraped sources by token containment:
        containment = |statement_tokens ∩ source_tokens| / |statement_tokens|
    over the FULL source text. Returns [{statement, citation_indexes}, ...]
    with citation_indexes as 1-based aio_src indexes (matching aio_src{i}).
    """
    src_tokens = {idx: _stmt_content_tokens(t)
                  for idx, t in src_texts.items() if t}
    if not src_tokens:
        return []
    grouped = []
    for chunk in _split_aio_chunks(aio_text)[:30]:
        ct = _stmt_content_tokens(chunk)
        if len(ct) < 4:
            continue
        scored = []
        for idx, st in src_tokens.items():
            if not st:
                continue
            containment = len(ct & st) / len(ct)
            if containment >= threshold:
                scored.append((containment, idx))
        scored.sort(reverse=True)
        idxs = sorted(idx for _, idx in scored[:top_k])
        if idxs:
            grouped.append({"statement": chunk, "citation_indexes": idxs})
        if len(grouped) >= max_statements:
            break
    return grouped

def annotate_units_by_containment(units, src_texts, threshold=0.6, top_k=3):
    """Attach citation_indexes to EXISTING statement units by token containment.

    Why this exists: the old code did `grouped = match_statements_to_sources(...)`,
    which REPLACED the full unit list with only the units the matcher could tie
    to a source. Every uncited paragraph, heading and bullet was silently
    discarded. On the 2026-08-26 run that lost >50% of the overview text on 41
    of 377 questions (one 548-word overview came through as a single 10-word
    fragment) and quietly shrank the Halloumi denominator.

    It also re-split the text into its own chunks, which breaks the rule that a
    statement is Google's RENDERED unit and never a sentence split.

    This annotates in place instead: same units, same order, same granularity,
    with citations added where the evidence supports them.
    """
    src_tokens = {idx: _stmt_content_tokens(t)
                  for idx, t in src_texts.items() if t}
    if not src_tokens:
        return units, 0
    hits = 0
    for u in units:
        if u.get("citation_indexes"):
            continue
        ct = _stmt_content_tokens(u.get("statement", ""))
        if len(ct) < 4:
            continue
        scored = []
        for idx, st in src_tokens.items():
            if st and len(ct & st) / len(ct) >= threshold:
                scored.append((len(ct & st) / len(ct), idx))
        scored.sort(reverse=True)
        idxs = sorted(idx for _, idx in scored[:top_k])
        if idxs:
            u["citation_indexes"] = idxs
            hits += 1
    return units, hits


# ── Per-question processor ────────────────────────────────────────────────────

def process_question(question, api_key, search_date):
    print(f"\n{'='*60}")
    print(f"Q: {question}")
    print(f"{'='*60}")

    row = {"question": _csv_clean(question)}
    # questions.xlsx metadata, immediately to the right of the question.
    _meta_cols = globals().get("QUESTION_META_COLS", [])
    _meta      = globals().get("QUESTION_META", {}).get(question, {})
    for _c in _meta_cols:
        row[_c] = _meta.get(_c, "")
    row.update({
        "search_date": search_date,
        "aio_present": False,
        "aio_text":    "",
    })

    # 1. SerpAPI
    try:
        result  = search_google(question, api_key)
        parsed  = parse_serpapi_result(result)
    except Exception as e:
        print(f"  ✗ SerpAPI error: {e}")
        row["error"] = _csv_clean(f"serpapi_error: {e}")
        return row

    # Google /goto wrappers are resolved inside parse_serpapi_result; keep the
    # tally so any that survive are visible in the data, not just the log.
    row["google_redirects_resolved"]   = parsed.get("google_redirects_resolved", 0)
    row["google_redirects_unresolved"] = parsed.get("google_redirects_unresolved", 0)

    # Gate: AI Overview must be present. If Google didn't return one for this
    # question, we skip organic / forum scraping entirely and emit a stub row.
    aio_text_raw = (parsed.get("aio_text") or "").strip()
    aio_sources  = parsed.get("aio_sources", []) or []
    if not aio_text_raw and not aio_sources:
        print(f"  ⚠ No AI Overview returned by Google — skipping question (no organic scrape).")
        row["aio_present"] = False
        row["aio_text"]    = ""
        # aio_token_empty: Google showed an overview placeholder whose content
        # SerpAPI could not load. Final (not retried), but kept distinct from a
        # genuine no-overview result.
        row["error"]       = "aio_token_empty" if result.get("_aio_token_empty") else "no_aio"
        return row

    row["aio_present"] = True
    row["aio_text"]    = _csv_clean(aio_text_raw)
    print(f"  AIO text: {len(row['aio_text'])} chars")

    # NEW — flatten per-statement citation columns right after aio_text
    aio_statements = parsed.get("aio_statements", [])
    if aio_statements:
        stmt_cols = flatten_aio_statements(aio_statements)
        row.update(stmt_cols)
        print(f"  AIO statements with sources: {len(stmt_cols) // 3}")
    else:
        print(f"  AIO statements: none (AIO may not expose inline references)")
    # R5 - defer grouped emit until after AIO scraping (fuzzy fallback needs source texts)
    grouped_serpapi = parsed.get("aio_statements_grouped", [])

    # 2. AIO sources
    scraped_aio  = 0
    aio_src_texts = {}  # R5 - collect scraped texts for fuzzy fallback
    for src in aio_sources:
        url = src.get("url", "")
        if not url or is_video_url(url):
            continue
        scraped_aio += 1
        prefix = f"aio_src{scraped_aio}"
        print(f"  Scraping AIO src {scraped_aio}: {url[:60]}")

        url = _strip_fragment(url)
        scraped = _clean_dict(scrape_page(url))
        row[f"{prefix}_title"]          = _csv_clean(src.get("title", ""))
        row[f"{prefix}_url"]            = _csv_clean(url)
        row[f"{prefix}_filtered_index"] = scraped_aio   # R5 - stable index for Cell 9
        row[f"{prefix}_text"]           = scraped.get("text", "")
        row[f"{prefix}_authors"]        = scraped.get("authors", "")
        row[f"{prefix}_pub_date"]       = scraped.get("pub_date", "")
        row[f"{prefix}_updated_date"]   = scraped.get("updated_date", "")
        row[f"{prefix}_disclosures"]    = scraped.get("disclosures", "")
        row[f"{prefix}_error"]          = scraped.get("error", "")
        row[f"{prefix}_fetch_method"]   = scraped.get("fetch_method", "")
        row[f"{prefix}_text_quality"]   = scraped.get("text_quality", "")
        for _k in SIG_COLS:                  # page-quality signals (JAMA/HON)
            row[f"{prefix}_{_k}"]       = scraped.get(_k, "")
        # NEW — fall back to SerpAPI snippet if scrape produced no text
        if not row[f"{prefix}_text"]:
            snippet = src.get("snippet", "") or parsed.get("snippet_by_url", {}).get(url, "")
            if snippet:
                row[f"{prefix}_text"]         = _csv_clean(snippet)
                row[f"{prefix}_fetch_method"] = "snippet_fallback"
                row[f"{prefix}_text_quality"] = "snippet_only"
                row[f"{prefix}_error"]        = (row.get(f"{prefix}_error", "") + "|filled_from_snippet").strip("|")

        aio_src_texts[scraped_aio] = row[f"{prefix}_text"]  # R5 - for fuzzy fallback

        refs = scraped.get("references", [])
        cited = scrape_cited_sources(refs, max_n=5)
        row.update(flatten_citations(cited, prefix, max_n=5))

    # grouped now contains EVERY block unit (cited AND uncited) in document
    # order — the containment fallback only fires when NO unit carries
    # citations (i.e. Google exposed no refs at all for this AIO).
    grouped = list(grouped_serpapi)
    _has_cited = any(g.get("citation_indexes") for g in grouped)
    _cont_used = False
    if not _has_cited and aio_text_raw and aio_src_texts:
        try:
            if grouped:
                # Preferred path: keep Google's rendered units, add citations.
                grouped, _hits = annotate_units_by_containment(grouped, aio_src_texts)
                _cont_used = _hits > 0
                if _hits:
                    print(f"  AIO statements (containment): annotated {_hits}"
                          f"/{len(grouped)} units, none dropped")
            else:
                # Only when the block parser produced nothing at all is there no
                # unit list to annotate; chunk the raw text as a last resort.
                _cont = match_statements_to_sources(aio_text_raw, aio_src_texts)
                if _cont:
                    grouped = _cont
                    _cont_used = True
                    print(f"  AIO statements (containment, no blocks): {len(grouped)}")
        except Exception as _cont_err:
            print(f"  ⚠ Containment fallback error: {_cont_err}")

    # Loud guard: statements must account for the overview. If they cover far
    # less, text is being dropped somewhere and the run should say so rather
    # than quietly under-report.
    if aio_text_raw and grouped:
        _rw = len(str(aio_text_raw).split())
        _sw = sum(len(str(g.get("statement", "")).split()) for g in grouped)
        if _rw and _sw / _rw < 0.5:
            print(f"  ⚠️  STATEMENT COVERAGE {_sw}/{_rw} words ({_sw/_rw:.0%}) — "
                  f"overview text is being lost; check the block parser.")

    # Emit grouped statement columns; store JSON + source marker for Cell 9
    if grouped:
        row.update(flatten_aio_statements_grouped(grouped))
        _n_cited = sum(1 for g in grouped if g.get("citation_indexes"))
        print(f"  AIO statements (grouped): {len(grouped)} units, {_n_cited} cited")
    row["aio_statements_grouped_json"] = json.dumps(grouped, ensure_ascii=False) if grouped else ""
    row["aio_statements_source"] = (
        "serpapi" if _has_cited
        else ("containment" if _cont_used else "none")
    )

    # 3. Organic results
    organic_urls = parsed.get("organic_urls", [])
    scraped_org  = 0
    forum_done   = False

    for url in organic_urls:
        if not url or is_video_url(url):
            continue
        scraped_org += 1
        prefix = f"org{scraped_org}"
        print(f"  Scraping organic {scraped_org}: {url[:60]}")

        if is_forum_url(url) and not forum_done:
            # Guard: a forum-scrape exception must never discard the whole
            # question row — record the error on this slot and keep going.
            try:
                forum_data = _clean_dict(scrape_forum(url))
            except Exception as _fe:
                forum_data = {"error": f"forum_exception:{type(_fe).__name__}:{str(_fe)[:80]}",
                              "serpapi_retry": True}
            row[f"{prefix}_url"]           = _csv_clean(url)
            row[f"{prefix}_type"]          = "forum"
            row[f"{prefix}_error"]         = _csv_clean(str(forum_data.get("error", "")))
            row[f"{prefix}_serpapi_retry"] = forum_data.get("serpapi_retry", False)
            row.update({f"{prefix}_{k}": v for k, v in flatten_forum(forum_data).items()})
            forum_done = True
        else:
            url = _strip_fragment(url)
            scraped = _clean_dict(scrape_page(url))
            row[f"{prefix}_url"]           = _csv_clean(url)
            row[f"{prefix}_type"]          = "article"
            row[f"{prefix}_text"]          = scraped.get("text", "")
            row[f"{prefix}_authors"]       = scraped.get("authors", "")
            row[f"{prefix}_pub_date"]      = scraped.get("pub_date", "")
            row[f"{prefix}_updated_date"]  = scraped.get("updated_date", "")
            row[f"{prefix}_disclosures"]   = scraped.get("disclosures", "")
            row[f"{prefix}_error"]         = scraped.get("error", "")
            row[f"{prefix}_fetch_method"]  = scraped.get("fetch_method", "")
            row[f"{prefix}_text_quality"]  = scraped.get("text_quality", "")
            for _k in SIG_COLS:              # page-quality signals (JAMA/HON)
                row[f"{prefix}_{_k}"]      = scraped.get(_k, "")
            # NEW — fall back to SerpAPI snippet if scrape produced no text
            if not row[f"{prefix}_text"]:
                snippet = parsed.get("snippet_by_url", {}).get(url, "")
                if snippet:
                    row[f"{prefix}_text"]         = _csv_clean(snippet)
                    row[f"{prefix}_fetch_method"] = "snippet_fallback"
                    row[f"{prefix}_text_quality"] = "snippet_only"
                    row[f"{prefix}_error"]        = (row.get(f"{prefix}_error", "") + "|filled_from_snippet").strip("|")


            refs  = scraped.get("references", [])
            cited = scrape_cited_sources(refs, max_n=5)
            row.update(flatten_citations(cited, prefix, max_n=5))

    # 4. Forum URL (discussions_and_forums)
    #
    # Google's "Discussions and forums" block is a SEPARATE SERP feature, not a
    # sixth ranked organic result, and it only appears for some questions. When
    # it was appended into the next org{N} slot it became a phantom "rank 6"
    # present for 96 of 377 questions on the 2026-08-26 run: every row a forum,
    # 57 of them google.com/goto wrappers, none passing the text-quality gate,
    # and all scoring exactly 0 on all four JAMA criteria. Averaging that into
    # the organic arm depressed it by 0.22 JAMA / 0.15 HON on a quarter of the
    # corpus and made the AIO deficit look smaller than it is.
    #
    # Set INCLUDE_FORUM_BLOCK = True to restore the old behaviour (e.g. to
    # analyse the forums block deliberately as its own arm). Left off, the
    # organic arm is exactly the top 5 the design calls for. Forums that
    # genuinely RANK inside the top 5 are unaffected -- they arrive through the
    # normal organic loop above.
    INCLUDE_FORUM_BLOCK = False

    forum_url = parsed.get("forum_url")
    if forum_url and not INCLUDE_FORUM_BLOCK:
        row["forum_block_url"] = _csv_clean(forum_url)   # recorded, not scored
        forum_url = None
    if forum_url and not forum_done:
        scraped_org += 1
        prefix = f"org{scraped_org}"
        print(f"  Scraping forum: {forum_url[:60]}")
        try:
            forum_data = _clean_dict(scrape_forum(forum_url))
        except Exception as _fe:
            forum_data = {"error": f"forum_exception:{type(_fe).__name__}:{str(_fe)[:80]}",
                          "serpapi_retry": True}
        row[f"{prefix}_url"]           = _csv_clean(forum_url)
        row[f"{prefix}_type"]          = "forum"
        row[f"{prefix}_error"]         = _csv_clean(str(forum_data.get("error", "")))
        row[f"{prefix}_serpapi_retry"] = forum_data.get("serpapi_retry", False)
        row.update({f"{prefix}_{k}": v for k, v in flatten_forum(forum_data).items()})

    print(f"  ✓ AIO sources scraped: {scraped_aio} | Organic scraped: {scraped_org}")
    return row

# ── Main pipeline ─────────────────────────────────────────────────────────────

_required_fns = [
    'search_google', 'parse_serpapi_result',
    'scrape_page', 'scrape_cited_sources', 'scrape_forum',
]
# NOTE: dir() must be captured OUTSIDE the comprehension. A comprehension gets
# its own scope, so a bare `dir()` inside it returns only the comprehension's
# locals -- which made every function look missing and silently skipped the
# entire pipeline, leaving Step 9 to reuse a stale research_output.csv.
_defined_names = set(dir())
_missing_fns = [f for f in _required_fns if f not in _defined_names]
if _missing_fns:
    print(f"⚠️  Run Steps 3–7 first. Missing: {', '.join(_missing_fns)}")
elif 'QUESTIONS' not in dir() or not QUESTIONS:
    print("⚠️  Run Step 2 first to load questions.")
elif 'CURRENT_BATCH_NUM' not in dir():
    print("⚠️  Run Step 2.5 (Batch Plan) first.")
elif CURRENT_BATCH_NUM is None:
    print("✅ All batches already complete — nothing to do.")
    print("   Proceed to Step 8.5 (Merge Batch Files).")
elif not SERPAPI_KEY:
    print("⚠️  Set your SERPAPI_KEY in Step 1 first.")
else:
    from concurrent.futures import ThreadPoolExecutor, as_completed
    import threading

    QUESTION_WORKERS = 2  # @param {type:"slider", min:1, max:4, step:1}
    # Lowered from 4 -> each question already runs its own 8-worker citation
    # pool, so QUESTION_WORKERS=4 meant up to 32 pages fetched concurrently.
    # That plus an unbounded-until-4096 raw-HTML cache is what ran a free
    # Colab runtime out of RAM on the full 408-question set; batching (Step
    # 2.5) plus this bounds peak memory per run instead.
    all_rows     = []
    rows_lock    = threading.Lock()
    search_date  = str(date.today())
    total        = len(CURRENT_BATCH_QUESTIONS)

    def _run_one(idx_question):
        idx, question = idx_question
        print(f"\n[{idx}/{total}] Processing...")
        try:
            return process_question(question, SERPAPI_KEY, search_date)
        except Exception as e:
            import traceback as _tb
            print(f"  ✗ Fatal error on question {idx}: {type(e).__name__}: {e}")
            print(_tb.format_exc())   # the 8/26 'int'.strip crash had no traceback to go on
            _stub = {"question": _csv_clean(question)}
            for _c in globals().get("QUESTION_META_COLS", []):
                _stub[_c] = globals().get("QUESTION_META", {}).get(question, {}).get(_c, "")
            _stub["search_date"] = search_date
            # Prefixed so Step 2.5 re-queues the question (up to MAX_ATTEMPTS).
            _stub["error"]       = _csv_clean(f"question_exception: {type(e).__name__}: {e}")
            return _stub

    try:
        with ThreadPoolExecutor(max_workers=QUESTION_WORKERS) as ex:
            futures = {ex.submit(_run_one, (i, q)): i
                       for i, q in enumerate(CURRENT_BATCH_QUESTIONS, 1)}
            done_count = 0
            for fut in as_completed(futures):
                row = fut.result()
                with rows_lock:
                    all_rows.append(row)
                    done_count += 1
                    save_results(all_rows, OUTPUT_FILE)
                print(f"  ✓ Progress saved ({done_count}/{total} questions done)")

    except KeyboardInterrupt:
        print(f"\n⚠️  Interrupted after {len(all_rows)} question(s). Saving...")
        save_results(all_rows, OUTPUT_FILE)
        print("✅ Partial results saved. Re-run Step 8 to resume from scratch.")
        raise

    print(f"\n{'='*60}")
    print(f"✅ Batch {CURRENT_BATCH_NUM}/{_n_batches} complete! "
          f"{len(all_rows)} question(s) processed.")
    print(f"{'='*60}")

    if CURRENT_BATCH_NUM < _n_batches:
        print("\n🔁 Next: free this session's RAM before the next batch.")
        print("   1. Runtime → Restart session")
        print("   2. Re-run top to bottom: Step 1 → Step 2 → Step 2.5 → "
              "Steps 3-7 → Step 8 (this cell)")
        print("      Step 2.5 auto-detects this batch is done and queues the next one.")
    else:
        print("\n🎉 All batches complete!")
        print("   Proceed to Step 8.5 (Merge Batch Files), then Step 9 as usual.")

    # Free the Playwright browser/contexts after the run completes
    try:
        shutdown_playwright()
        print("  ✓ Playwright browser closed")
    except Exception as e:
        print(f"  ⚠ Playwright shutdown: {e}")

    # Release cached page bodies so a same-session re-run starts clean.
    try:
        clear_fetch_cache()
    except Exception as e:
        print(f"  ⚠ Cache clear: {e}")

    print(f"\n📁 Saved locally: {OUTPUT_FILE}")


In [ ]:
# @title 🔗 STEP 8.5 — Merge Batch Files { display-mode: "form" }
# @markdown Concatenates every `research_output_batchNN.csv` on Drive into a
# @markdown single `research_output.csv` — the exact filename Step 9 already
# @markdown reads, so Step 9 needs no changes. Safe to re-run any time; always
# @markdown rebuilds from whatever batch files currently exist.

import glob, os, re, csv
import pandas as pd

_batch_files = sorted(
    glob.glob(os.path.join(PROJECT_DIR, "research_output_batch*.csv")),
    key=lambda p: int(re.search(r"batch(\d+)", p).group(1)))

if not _batch_files:
    print("⚠️  No batch files found — run Step 8 at least once first.")
else:
    print(f"Merging {len(_batch_files)} batch file(s):")
    _frames = []
    for _f in _batch_files:
        _df = pd.read_csv(_f, encoding='utf-8-sig', low_memory=False)
        print(f"  {os.path.basename(_f)}: {len(_df)} rows")
        _frames.append(_df)

    merged = pd.concat(_frames, ignore_index=True, sort=False)
    _before = len(merged)
    # A question can appear in several batches (a failed attempt, then a
    # retry). Keep its latest SUCCESSFUL row; fall back to the latest failure
    # only when it never succeeded. Plain keep="last" could let a later quota
    # stub overwrite real data.
    _err = merged["error"].fillna("").astype(str) if "error" in merged.columns \
        else pd.Series("", index=merged.index)
    _retry_re = globals().get("RETRYABLE_ERROR_RE",
                              r"serpapi_error|question_exception|aio_prefetch_failed")
    merged["_failed"] = _err.str.contains(_retry_re, case=False, regex=True).astype(int)
    merged["_order"] = range(len(merged))
    merged = (merged.sort_values(["_failed", "_order"], ascending=[True, False])
                    .drop_duplicates(subset="question", keep="first")
                    .sort_values("_order")
                    .drop(columns=["_failed", "_order"]))
    if len(merged) < _before:
        print(f"  ⚠️  Collapsed {_before - len(merged)} duplicate question row(s) "
              f"(kept the latest successful attempt).")

    _final_path = os.path.join(PROJECT_DIR, "research_output.csv")
    merged.to_csv(_final_path, index=False, encoding='utf-8-sig',
                  quoting=csv.QUOTE_ALL, lineterminator='\n')
    print(f"\n✅ Merged {len(merged)} question(s) → {_final_path}")
    print("   Proceed to Step 9.")


In [ ]:
# @title 📊 STEP 9 — Reshape Raw CSV into Two Clean Datasets { display-mode: "form" }
# @markdown Reshapes research_output.csv into:
# @markdown   • aio_dataset.csv   — one row per AIO source per question
# @markdown   • organic_dataset.csv — one row per organic result per question

PROJECT_DIR   = r"C:/Users/SAgno/Documents/AIO_Workspace/LocalRun"  # @param {type:"string"}
OUTPUT_SUBDIR = "Output"                 # @param {type:"string"}
RAW_FILE      = "research_output.csv"    # @param {type:"string"}
AIO_OUT       = "aio_dataset.csv"        # @param {type:"string"}
ORG_OUT       = "organic_dataset.csv"    # @param {type:"string"}
# NOTE: source/citation slots are detected dynamically from the raw CSV's
# column names — ALL aio_src{N} slots are kept (every AIO citation survives).

import pandas as pd
import re
import csv
import json

# Carried through from questions.xlsx by Steps 2 and 8. Declared here too so
# Step 9 can be re-run on its own without Step 2 in the namespace.
QUESTION_META_COLS = ["risk_level", "rubric_complexity", "requires_hedging"]

# Page-quality signal columns written by Step 5 / Step 8 (sig_*). Read from
# health_scorers.py so this list can never drift from the scorer's.
import sys as _sys_step9
if PROJECT_DIR not in _sys_step9.path:
    _sys_step9.path.insert(0, PROJECT_DIR)
try:
    from health_scorers import SIG_COLS
except ImportError:
    SIG_COLS = ["sig_byline", "sig_reviewer", "sig_credentialed", "sig_date_label", "sig_date_visible",
                "sig_disclaimer", "sig_disclosure", "sig_ref_section",
                "sig_n_numbered_cites", "sig_n_external_links"]

def _sig_val(qrow, col):
    """Signal cell -> clean string ('' for NaN / missing column)."""
    v = qrow.get(col, '')
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return ''
    return clean_text(str(v))

# --- Output to Drive: datasets live ONLY in the Output subfolder ---
import os as _os_step9
_OUT_DIR = _os_step9.path.join(PROJECT_DIR, OUTPUT_SUBDIR)
_os_step9.makedirs(_OUT_DIR, exist_ok=True)
AIO_OUT = _os_step9.path.join(_OUT_DIR, AIO_OUT)
ORG_OUT = _os_step9.path.join(_OUT_DIR, ORG_OUT)
RAW_FILE = _os_step9.path.join(PROJECT_DIR, RAW_FILE)   # raw stays in project root
print(f"Datasets will be written to: {_OUT_DIR}")

# ── Helpers ───────────────────────────────────────────────────────────────────

def clean_text(val):
    """Strip newlines/tabs from any string value."""
    if not isinstance(val, str):
        return val
    val = re.sub(r'[\r\n\t]+', ' ', val)
    val = re.sub(r'  +', ' ', val)
    return val.strip()

def clean_authors(val):
    """Post-process author strings to remove nav menu garbage."""
    if not isinstance(val, str) or not val:
        return val
    if val.count('|') > 2:
        first = val.split('|')[0].strip()
        return first if len(first) < 80 else ''
    if val.rstrip().endswith(':') or val.rstrip().endswith(','):
        return ''
    if len(val) > 150:
        return ''
    return val

def strip_fragment(url):
    """
    Remove #:~:text= highlight fragments that Google adds to AIO source URLs.
    e.g. https://cdc.gov/page#:~:text=foo  →  https://cdc.gov/page
    Also strips plain #anchor fragments.
    """
    if not isinstance(url, str):
        return url
    # Remove everything from # onward
    return re.sub(r'#.*$', '', url).strip()

def dedup_urls(urls):
    """
    Deduplicate a list of URLs after stripping fragments.
    Preserves first occurrence.
    """
    seen = set()
    result = []
    for u in urls:
        clean = strip_fragment(u) if u else u
        if clean and clean not in seen:
            seen.add(clean)
            result.append(clean)
        elif not clean:
            result.append(u)
    return result

def save_df(df, filename):
    df.to_csv(
        filename,
        index=False,
        encoding='utf-8-sig',
        quoting=csv.QUOTE_ALL,
        lineterminator='\n',
    )
    print(f"  💾 Saved {len(df)} rows × {len(df.columns)} cols → {filename}")

# ── Load raw CSV ──────────────────────────────────────────────────────────────

# Fix 13: Warn if the CSV is potentially stale
import os as _os, time as _time
if _os.path.exists(RAW_FILE):
    _age = _time.time() - _os.path.getmtime(RAW_FILE)
    if _age > 3600:
        print(f"\u26a0\ufe0f  Warning: '{RAW_FILE}' is {_age/3600:.1f}h old \u2014 re-run Step 8 if needed.")
    else:
        print(f"  \u2713 File age: {_age/60:.0f} min")
else:
    print(f"\u26a0\ufe0f  '{RAW_FILE}' not found \u2014 run Step 8 first.")

print(f"Loading {RAW_FILE}...")
try:
    raw = pd.read_csv(RAW_FILE, encoding='utf-8-sig', low_memory=False)
except UnicodeDecodeError:
    raw = pd.read_csv(RAW_FILE, encoding='utf-8', low_memory=False)

print(f"  Raw shape: {raw.shape}")
print(f"  Questions: {raw['question'].nunique()}")

# Loud check: Step 9 silently reshaping a stale CSV from a previous run is the
# single easiest way to analyse the wrong dataset without noticing.
_loaded_qs = globals().get("QUESTIONS") or []
if _loaded_qs:
    _n_raw, _n_loaded = raw['question'].nunique(), len(set(map(str, _loaded_qs)))
    if _n_raw != _n_loaded:
        print("\n" + "!" * 68)
        print(f"!! MISMATCH: research_output.csv holds {_n_raw} question(s) but Step 2")
        print(f"!! loaded {_n_loaded}. Step 8 did not run (or did not finish) on the")
        print(f"!! current questions file. Everything below describes the OLD run.")
        print("!" * 68 + "\n")
    else:
        print(f"  ✓ Matches the {_n_loaded} questions loaded in Step 2.")
else:
    print("  ℹ Step 2 not in namespace — cannot verify this CSV is current.")

_meta_in_raw = [c for c in QUESTION_META_COLS if c in raw.columns]
print(f"  Question metadata present: {_meta_in_raw if _meta_in_raw else '(none)'}")

# Clean all string columns in place
for col in raw.columns:
    raw[col] = raw[col].apply(lambda x: clean_text(x) if isinstance(x, str) else x)

# ── Detect structure dynamically ──────────────────────────────────────────────

def get_aio_src_indices(cols):
    """Return sorted list of AIO source indices found in column names."""
    indices = set()
    for c in cols:
        m = re.match(r'^aio_src(\d+)_', c)
        if m:
            indices.add(int(m.group(1)))
    return sorted(indices)

def get_org_indices(cols):
    """Return sorted list of organic result indices found in column names."""
    indices = set()
    for c in cols:
        m = re.match(r'^org(\d+)_', c)
        if m:
            indices.add(int(m.group(1)))
    return sorted(indices)

def get_cite_indices(cols, prefix):
    """Return sorted list of citation indices for a given prefix."""
    indices = set()
    for c in cols:
        m = re.match(rf'^{re.escape(prefix)}_cite(\d+)_', c)
        if m:
            indices.add(int(m.group(1)))
    return sorted(indices)

aio_src_indices = get_aio_src_indices(raw.columns)
org_indices     = get_org_indices(raw.columns)
print(f"  AIO src slots detected: {aio_src_indices}")
print(f"  Organic slots detected: {org_indices}")

# ── Build AIO dataset ─────────────────────────────────────────────────────────

print("\nBuilding AIO dataset...")
aio_rows = []

for _, qrow in raw.iterrows():
    question    = qrow.get('question', '')
    search_date = qrow.get('search_date', '')
    aio_text    = clean_text(str(qrow.get('aio_text', '') or ''))

    for i in aio_src_indices:
        prefix = f"aio_src{i}"
        url_raw = qrow.get(f"{prefix}_url", '')

        # Skip empty slots
        if pd.isna(url_raw) or str(url_raw).strip() == '':
            continue

        # Strip fragment from URL — deduplicate within question handled below
        url = strip_fragment(str(url_raw).strip())

        row = {
            'question':     question,
            **{c: qrow.get(c, '') for c in QUESTION_META_COLS},
            'search_date':  search_date,
            'type':         'AIO',
            'rank':         i,
            'source_index': i,   # stable aio_src index; survives dedup, used by Halloumi to map statement citations -> source text
            'aio_text':     aio_text,
            'title':        clean_text(str(qrow.get(f"{prefix}_title", '') or '')),
            'url':          url,
            'authors':      clean_authors(clean_text(str(qrow.get(f"{prefix}_authors", '') or ''))),
            'pub_date':     clean_text(str(qrow.get(f"{prefix}_pub_date", '') or '')),
            'updated_date': clean_text(str(qrow.get(f"{prefix}_updated_date", '') or '')),
            'text':         clean_text(str(qrow.get(f"{prefix}_text", '') or '')),
            'disclosures':  clean_text(str(qrow.get(f"{prefix}_disclosures", '') or '')),
            'error':        clean_text(str(qrow.get(f"{prefix}_error", '') or '')),
            'fetch_method': clean_text(str(qrow.get(f"{prefix}_fetch_method", '') or '')),
            'text_quality': clean_text(str(qrow.get(f"{prefix}_text_quality", '') or '')),
            **{k: _sig_val(qrow, f"{prefix}_{k}") for k in SIG_COLS},
        }

        # Citation columns
        cite_indices = get_cite_indices(raw.columns, prefix)
        for j in cite_indices:
            cp = f"{prefix}_cite{j}"
            row[f"cite{j}_url"]         = strip_fragment(clean_text(str(qrow.get(f"{cp}_url", '') or '')))
            row[f"cite{j}_ref_text"]    = clean_text(str(qrow.get(f"{cp}_ref_text", '') or ''))
            row[f"cite{j}_authors"]     = clean_authors(clean_text(str(qrow.get(f"{cp}_authors", '') or '')))
            row[f"cite{j}_pub_date"]    = clean_text(str(qrow.get(f"{cp}_pub_date", '') or ''))
            row[f"cite{j}_updated_date"]= clean_text(str(qrow.get(f"{cp}_updated_date", '') or ''))
            row[f"cite{j}_text"]         = clean_text(str(qrow.get(f"{cp}_text", '') or ''))
            row[f"cite{j}_disclosures"]  = clean_text(str(qrow.get(f"{cp}_disclosures", '') or ''))
            row[f"cite{j}_error"]        = clean_text(str(qrow.get(f"{cp}_error", '') or ''))
            row[f"cite{j}_fetch_method"] = clean_text(str(qrow.get(f"{cp}_fetch_method", '') or ''))
            row[f"cite{j}_text_quality"] = clean_text(str(qrow.get(f"{cp}_text_quality", '') or ''))
            for k in SIG_COLS:
                row[f"cite{j}_{k}"] = _sig_val(qrow, f"{cp}_{k}")


        # Statement columns are PER-AIO (identical on every row of a question):
        # statement N = the Nth block unit of the overview, with its citations.
        # `cited_by_statement_indexes` stays per-source (which statements cite
        # THIS source). The Halloumi notebook fills the verdict columns later.
        grouped_json = qrow.get("aio_statements_grouped_json", "")
        try:
            grouped_all = json.loads(grouped_json) if grouped_json else []
        except Exception:
            grouped_all = []
        this_idx = i  # matches aio_src{i}_filtered_index stored by Cell 8
        row["cited_by_statement_indexes"] = ",".join(
            str(j) for j, g in enumerate(grouped_all, 1)
            if this_idx in (g.get("citation_indexes") or [])
        )
        for k, g in enumerate(grouped_all, 1):
            if k > 15:
                break
            row[f"aio_statement{k}_text"]      = clean_text(g.get("statement", ""))
            row[f"aio_statement{k}_citations"] = ",".join(
                str(x) for x in sorted(set(int(v) for v in (g.get("citation_indexes") or [])))
            )
        row["aio_statements_source"] = str(qrow.get("aio_statements_source", "") or "")
        aio_rows.append(row)

# Deduplicate AIO rows by (question, url) — fragment stripping may have created dupes
seen_aio = set()
aio_rows_deduped = []
rank_counters = {}
for row in aio_rows:
    key = (row['question'], row['url'])
    if key not in seen_aio:
        seen_aio.add(key)
        # Re-number rank sequentially after dedup
        q = row['question']
        rank_counters[q] = rank_counters.get(q, 0) + 1
        row['rank'] = rank_counters[q]
        aio_rows_deduped.append(row)

aio_df = pd.DataFrame(aio_rows_deduped)
print(f"  AIO rows before dedup: {len(aio_rows)}")
print(f"  AIO rows after dedup:  {len(aio_rows_deduped)}")
print(f"  Questions in AIO:      {aio_df['question'].nunique() if len(aio_df) else 0}")
if len(aio_df):
    print(f"  Per-question counts:   {aio_df.groupby('question').size().to_dict()}")

# ── Build Organic dataset ─────────────────────────────────────────────────────

print("\nBuilding Organic dataset...")
org_rows = []

for _, qrow in raw.iterrows():
    question    = qrow.get('question', '')
    search_date = qrow.get('search_date', '')

    for i in org_indices:
        prefix = f"org{i}"
        url_raw = qrow.get(f"{prefix}_url", '')

        # Skip empty slots — this is the critical fix for missing questions
        # A question with no AIO data still has organic data; we must NOT skip
        # the question just because aio_text is empty
        if pd.isna(url_raw) or str(url_raw).strip() == '':
            continue

        url      = strip_fragment(str(url_raw).strip())
        src_type = clean_text(str(qrow.get(f"{prefix}_type", 'Organic') or 'Organic'))
        is_forum = src_type == 'forum'

        row = {
            'question':    question,
            **{c: qrow.get(c, '') for c in QUESTION_META_COLS},
            'search_date': search_date,
            'type':        'Organic',
            'rank':        i,
            'url':         url,
            'error':       clean_text(str(qrow.get(f"{prefix}_error", '') or '')),
        }

        if is_forum:
            row['is_forum'] = True
            row['forum_serpapi_retry'] = qrow.get(f"{prefix}_serpapi_retry", False)
            # Pull all forum_* columns
            for col in raw.columns:
                if col.startswith(f"{prefix}_forum_"):
                    short = col[len(f"{prefix}_"):]
                    row[short] = clean_text(str(qrow.get(col, '') or ''))
            for col in raw.columns:
                if col.startswith(f"{prefix}_forum_comment"):
                    short = col[len(f"{prefix}_"):]
                    row[short] = clean_text(str(qrow.get(col, '') or ''))
        else:
            row['is_forum']   = False
            row['authors']    = clean_authors(clean_text(str(qrow.get(f"{prefix}_authors", '') or '')))
            row['pub_date']   = clean_text(str(qrow.get(f"{prefix}_pub_date", '') or ''))
            row['updated_date'] = clean_text(str(qrow.get(f"{prefix}_updated_date", '') or ''))
            row['text']       = clean_text(str(qrow.get(f"{prefix}_text", '') or ''))
            row['disclosures']= clean_text(str(qrow.get(f"{prefix}_disclosures", '') or ''))
            row['fetch_method'] = clean_text(str(qrow.get(f"{prefix}_fetch_method", '') or ''))
            row['text_quality'] = clean_text(str(qrow.get(f"{prefix}_text_quality", '') or ''))
            for k in SIG_COLS:
                row[k] = _sig_val(qrow, f"{prefix}_{k}")

            # Citations
            cite_indices = get_cite_indices(raw.columns, prefix)
            for j in cite_indices:
                cp = f"{prefix}_cite{j}"
                row[f"cite{j}_url"]         = strip_fragment(clean_text(str(qrow.get(f"{cp}_url", '') or '')))
                row[f"cite{j}_ref_text"]    = clean_text(str(qrow.get(f"{cp}_ref_text", '') or ''))
                row[f"cite{j}_authors"]     = clean_authors(clean_text(str(qrow.get(f"{cp}_authors", '') or '')))
                row[f"cite{j}_pub_date"]    = clean_text(str(qrow.get(f"{cp}_pub_date", '') or ''))
                row[f"cite{j}_updated_date"]= clean_text(str(qrow.get(f"{cp}_updated_date", '') or ''))
                row[f"cite{j}_text"]         = clean_text(str(qrow.get(f"{cp}_text", '') or ''))
                row[f"cite{j}_disclosures"]  = clean_text(str(qrow.get(f"{cp}_disclosures", '') or ''))
                row[f"cite{j}_error"]        = clean_text(str(qrow.get(f"{cp}_error", '') or ''))
                row[f"cite{j}_fetch_method"] = clean_text(str(qrow.get(f"{cp}_fetch_method", '') or ''))
                row[f"cite{j}_text_quality"] = clean_text(str(qrow.get(f"{cp}_text_quality", '') or ''))
                for k in SIG_COLS:
                    row[f"cite{j}_{k}"] = _sig_val(qrow, f"{cp}_{k}")

        org_rows.append(row)

org_df = pd.DataFrame(org_rows)
print(f"  Organic rows: {len(org_df)}")
print(f"  Questions in Organic: {org_df['question'].nunique() if len(org_df) else 0}")
if len(org_df):
    print(f"  Per-question counts: {org_df.groupby('question').size().to_dict()}")

# ── Save ──────────────────────────────────────────────────────────────────────

print()
if len(aio_df):
    save_df(aio_df, AIO_OUT)
else:
    print("  ⚠ No AIO rows — skipping AIO output.")

if len(org_df):
    save_df(org_df, ORG_OUT)
else:
    print("  ⚠ No Organic rows — skipping Organic output.")

# ── statements_raw.csv — the canonical statement list per question ───────────
# One row per block unit (paragraph / list item) in document order, cited or
# not. The Halloumi notebook consumes this directly, so statements are scored
# at Google's native granularity instead of being re-segmented from aio_text.
stmt_raw_rows = []
for _, qrow in raw.iterrows():
    question = qrow.get('question', '')
    gj = qrow.get('aio_statements_grouped_json', '')
    try:
        units = json.loads(gj) if isinstance(gj, str) and gj.strip() else []
    except Exception:
        units = []
    for i, u in enumerate(units, 1):
        stmt_raw_rows.append({
            'question':         question,
            'statement_index':  i,
            'statement_text':   clean_text(str(u.get('statement', '') or '')),
            'citation_indexes': ",".join(str(x) for x in (u.get('citation_indexes') or [])),
        })

if stmt_raw_rows:
    stmt_raw_df = pd.DataFrame(stmt_raw_rows)
    save_df(stmt_raw_df, _os_step9.path.join(_OUT_DIR, "statements_raw.csv"))
    n_cited_units = (stmt_raw_df['citation_indexes'] != '').sum()
    print(f"  statements_raw: {len(stmt_raw_df)} units, {n_cited_units} cited")
else:
    print("  ⚠ No statement units found in aio_statements_grouped_json — "
          "statements_raw.csv not written (Halloumi will reconstruct from aio_text).")

# Saved to Drive only — no PC download. Files live in Output/.
print(f"📂 Files saved to: {_OUT_DIR}")

print("\n✅ Step 9 complete.")

In [ ]:
# @title 🛠️ STEP 9R — Repair Bad Source Texts in Existing Datasets { display-mode: "form" }
# @markdown Fixes garbage/blank/snippet-only source texts in ALREADY-GENERATED
# @markdown `aio_dataset.csv` / `organic_dataset.csv` **without re-running SerpAPI**
# @markdown (no API cost). Re-fetches each bad URL through the improved cascade
# @markdown (Europe PMC → requests → Playwright → AMP → Jina → Wayback, all
# @markdown content-validated) and patches the row in place.
# @markdown
# @markdown **Requires Steps 1 and 5 to have been run first** (uses their fetch
# @markdown machinery). Edits the datasets in `Output/` in place — unfixable rows
# @markdown keep their old text but get `text_quality` flagged so Halloumi knows
# @markdown to exclude them.

PROJECT_DIR   = r"C:/Users/SAgno/Documents/AIO_Workspace/LocalRun"  # @param {type:"string"}
OUTPUT_SUBDIR = "Output"              # @param {type:"string"}
AIO_FILE     = "aio_dataset.csv"      # @param {type:"string"}
ORGANIC_FILE = "organic_dataset.csv"  # @param {type:"string"}
ALSO_RETRY_SNIPPETS = True            # @param {type:"boolean"}

import os, csv
import pandas as pd

_OUT_DIR = os.path.join(PROJECT_DIR, OUTPUT_SUBDIR)

_needed = ['scrape_page', 'assess_text_quality', 'fetch_html']
_missing = [f for f in _needed if f not in globals()]
if _missing:
    raise RuntimeError(f"Run Steps 1 and 5 first — missing: {', '.join(_missing)}")

def _fieldstr(val):
    if val is None:
        return ""
    s = str(val).strip()
    return "" if s.lower() in ("nan", "none", "nat", "null") else s

def _classify_row(row):
    """Current quality of a dataset row's text. snippet_fallback counts as
    repairable when ALSO_RETRY_SNIPPETS (snippets aren't full text)."""
    q = assess_text_quality(_fieldstr(row.get("text")))
    if q == "ok" and ALSO_RETRY_SNIPPETS and \
       _fieldstr(row.get("fetch_method")) == "snippet_fallback":
        return "snippet_only"
    return q

def repair_dataset(name):
    path = os.path.join(_OUT_DIR, name)
    if not os.path.exists(path):
        print(f"⚠ {OUTPUT_SUBDIR}/{name} not found — run Step 9 first.")
        return
    df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    print(f"\n=== {name}: {len(df)} rows ===")

    if "text_quality" not in df.columns:
        df["text_quality"] = ""

    bad_idx = []
    for i, row in df.iterrows():
        # Forum rows have no scrapeable 'text' column semantics — skip them
        if str(row.get("is_forum", "")).lower() == "true":
            continue
        q = _classify_row(row)
        df.at[i, "text_quality"] = q
        if q != "ok" and _fieldstr(row.get("url")):
            bad_idx.append(i)

    print(f"  bad/snippet rows to repair: {len(bad_idx)}")
    if not bad_idx:
        # still persist the text_quality column for downstream steps
        df.to_csv(path, index=False, encoding="utf-8-sig",
                  quoting=csv.QUOTE_ALL, lineterminator="\n")
        print("  ✓ no repairs needed — text_quality column written")
        return

    repaired, unfixable = 0, 0
    for i in bad_idx:
        url = _fieldstr(df.at[i, "url"])
        old_q = df.at[i, "text_quality"]
        print(f"  ↻ [{old_q}] {url[:70]}")
        try:
            scraped = scrape_page(url)
        except Exception as e:
            print(f"     ✗ exception: {type(e).__name__}: {str(e)[:60]}")
            unfixable += 1
            continue
        new_q = scraped.get("text_quality", "")
        if new_q == "ok" and scraped.get("text"):
            df.at[i, "text"]         = scraped["text"]
            df.at[i, "text_quality"] = "ok"
            df.at[i, "fetch_method"] = scraped.get("fetch_method", "")
            df.at[i, "error"]        = scraped.get("error", "")
            # Only overwrite metadata when the re-scrape found something real
            for col in ("authors", "pub_date", "updated_date", "disclosures"):
                if col in df.columns and _fieldstr(scraped.get(col)):
                    df.at[i, col] = scraped[col]
            # Page-quality signals come from the fresh full page — always overwrite.
            for col in globals().get("SIG_COLS", []):
                if col not in df.columns:
                    df[col] = ""
                df.at[i, col] = scraped.get(col, "")
            repaired += 1
            print(f"     ✓ repaired via {scraped.get('fetch_method','?')} "
                  f"({len(scraped['text']):,} chars)")
        else:
            unfixable += 1
            print(f"     ✗ still {new_q or 'failed'} — flagged for exclusion")

    df.to_csv(path, index=False, encoding="utf-8-sig",
              quoting=csv.QUOTE_ALL, lineterminator="\n")
    print(f"  💾 saved {name}: {repaired} repaired, {unfixable} unfixable "
          f"(flagged via text_quality)")

repair_dataset(AIO_FILE)
repair_dataset(ORGANIC_FILE)

try:
    shutdown_playwright()
except Exception:
    pass

print("\n✅ Step 9R complete. Re-run Steps 11–13 (and Halloumi) on the repaired data.")


In [ ]:
# @title 🔍 STEP 10 — Verify Data Quality (NaN rates & text lengths) { display-mode: "form" }
# @markdown Run AFTER Step 9 has produced aio_dataset.csv and organic_dataset.csv.
# @markdown Reports NaN percentages and average text lengths so you can confirm
# @markdown the fixes worked (text should be < 5% NaN, citation text > 5,000 avg chars).

PROJECT_DIR   = r"C:/Users/SAgno/Documents/AIO_Workspace/LocalRun"  # @param {type:"string"}
OUTPUT_SUBDIR = "Output"          # @param {type:"string"}
AIO_FILE = "aio_dataset.csv"      # @param {type:"string"}
ORG_FILE = "organic_dataset.csv"  # @param {type:"string"}

import os
import pandas as pd

AIO_FILE = os.path.join(PROJECT_DIR, OUTPUT_SUBDIR, AIO_FILE)
ORG_FILE = os.path.join(PROJECT_DIR, OUTPUT_SUBDIR, ORG_FILE)

def report(path, name):
    if not os.path.exists(path):
        print(f"⚠️  {path} not found — run Steps 8 & 9 first.")
        return
    df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    print(f"\n=== {name}: {len(df)} rows × {len(df.columns)} cols ===")

    # Headline columns: main text + each citation slot's text/author/date/disclosure
    text_cols = ["text"] + [f"cite{i}_text"        for i in range(1, 6)]
    auth_cols =            [f"cite{i}_authors"     for i in range(1, 6)]
    date_cols =            [f"cite{i}_pub_date"    for i in range(1, 6)]
    upd_cols  =            [f"cite{i}_updated_date" for i in range(1, 6)]
    disc_cols =            [f"cite{i}_disclosures" for i in range(1, 6)]

    def summarize(cols, label):
        print(f"  -- {label} --")
        for c in cols:
            if c not in df.columns:
                continue
            s = df[c]
            nan_pct = s.isna().mean() * 100
            non_empty = s.dropna().astype(str).str.strip()
            non_empty = non_empty[non_empty != ""]
            avg_len = non_empty.str.len().mean() if len(non_empty) else 0
            print(f"    {c:25s} NaN={nan_pct:5.1f}%  populated={len(non_empty):3d}  avg_len={avg_len:7.0f}")

    summarize(text_cols, "TEXT (target: < 5% NaN, avg_len > 5,000)")
    summarize(auth_cols, "AUTHORS (target: mostly populated)")
    summarize(date_cols, "PUB_DATE (target: mostly populated)")
    summarize(upd_cols,  "UPDATED_DATE (sparse is OK)")
    summarize(disc_cols, "DISCLOSURES (sparse is OK)")

    if "error" in df.columns:
        print("  -- error breakdown --")
        for val, cnt in df["error"].fillna("(none)").value_counts().head(10).items():
            print(f"    {str(val)[:60]:60s} {cnt}")

    # Show fetch_method distribution if present (tells us how many JS pages we recovered)
    method_cols = [c for c in df.columns if c.endswith("_fetch_method") or c == "fetch_method"]
    if method_cols:
        print("  -- fetch_method distribution (across all *_fetch_method cols) --")
        all_methods = pd.concat([df[c] for c in method_cols], ignore_index=True)
        for val, cnt in all_methods.dropna().value_counts().head(10).items():
            print(f"    {str(val)[:30]:30s} {cnt}")

report(AIO_FILE, "AIO")
report(ORG_FILE, "Organic")

print("\n✅ Step 10 complete. If text NaN > 10% or cite*_text avg_len < 3,000,")
print("   inspect rows where error != 'nan' to see why those fetches failed.")


In [ ]:
# @title 🧪 STEP 11 — JAMA Scoring { display-mode: "form" }
# @markdown Run AFTER Step 9. Adds JAMA_* keyword scores for every source — the
# @markdown primary AIO source, its 5 sub-citations, and every organic result —
# @markdown reading the page-quality `sig_*` columns captured by Step 5 (bylines,
# @markdown reviewer credentials, date labels, disclosure statements, reference
# @markdown sections, external links) with the structured columns as fallback.
# @markdown Rules calibrated against the human coding worksheet. NaN-robust. Saves to `PROJECT_DIR`:
# @markdown   * `aio_dataset.csv`
# @markdown   * `organic_dataset.csv`
# @markdown
# @markdown CPU-only — no GPU runtime needed.

PROJECT_DIR   = r"C:/Users/SAgno/Documents/AIO_Workspace/LocalRun"  # @param {type:"string"}
OUTPUT_SUBDIR = "Output"               # @param {type:"string"}
AIO_IN      = "aio_dataset.csv"            # @param {type:"string"}
ORGANIC_IN  = "organic_dataset.csv"        # @param {type:"string"}
AIO_OUT     = "aio_dataset.csv"       # @param {type:"string"}
ORGANIC_OUT = "organic_dataset.csv"   # @param {type:"string"}


import sys, os, csv, re
import pandas as pd
from tqdm.auto import tqdm

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)
from health_scorers import JAMAAnalyzer, _field, SIG_COLS

jama = JAMAAnalyzer()

from comparison_builder import registered_domain as _reg_dom

# Hosts that are never a cited SOURCE even when they sit off the page's domain
# (patient portals, booking, billing). Mirrors Step 5's _SIG_PORTAL_HOST_RE.
_PORTAL_RE = re.compile(
    r"(login|signin|sign-in|portal|mychart|followmyhealth|patientnotebook|"
    r"appointment|booking|schedul|billpay|bill-pay|payment|careers|"
    r"apps\.apple|play\.google|maps\.google|facebook|twitter|instagram|"
    r"linkedin|youtube|tiktok|pinterest)", re.I)

def _external_citation(row, prefix=""):
    """Fallback attribution signal for rows scraped BEFORE the sig_* columns
    existed: a scraped sub-citation whose registered domain differs from the
    source's own. Only primary sources carry sub-citations."""
    if prefix != "":
        return False
    src_dom = _reg_dom(_field(row.get("url")))
    for j in range(1, 6):
        cu = _field(row.get(f"cite{j}_url"))
        if cu and not _PORTAL_RE.search(cu) and _reg_dom(cu) != src_dom:
            return True
    return False

def _row_signals(row, prefix=""):
    """Pull a source's structured fields + page-quality signals. prefix=''
    for the primary source, 'cite{j}_' for a sub-citation. NaN-robust."""
    g = lambda k: _field(row.get(f"{prefix}{k}"))
    sig = {k: g(k) for k in SIG_COLS}
    return {
        "text":              g("text"),
        "authors":           g("authors"),
        "pub_date":          g("pub_date"),
        "updated_date":      g("updated_date"),
        "disclosures":       g("disclosures"),
        "url":               g("url"),
        "external_citation": _external_citation(row, prefix),
        **sig,
    }

def _has_source(row, prefix=""):
    """A source/citation slot has data only if it has a URL. Empty slots
    (no data collected during the scrape) are left BLANK, not scored 0."""
    return bool(_field(row.get(f"{prefix}url")))

_SCORE_KEYS = list(jama.score().keys())   # sub-scores + Total

def _enrich(df, score_citations):
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="JAMA"):
        out = row.to_dict()
        if _has_source(row, ""):
            out.update(jama.score(**_row_signals(row, "")))
        else:
            for k in _SCORE_KEYS:
                out[k] = None          # no data collected -> blank, not 0
        if score_citations:
            for j in range(1, 6):
                if _has_source(row, f"cite{j}_"):
                    for k, v in jama.score(**_row_signals(row, f"cite{j}_")).items():
                        out[f"cite{j}_{k}"] = v
                else:
                    for k in _SCORE_KEYS:
                        out[f"cite{j}_{k}"] = None   # empty slot -> blank (clears stale 0)
        records.append(out)
    return pd.DataFrame(records)

def _load(name):
    return pd.read_csv(os.path.join(PROJECT_DIR, OUTPUT_SUBDIR, name))

def _save(df, name):
    path = os.path.join(PROJECT_DIR, OUTPUT_SUBDIR, name)
    df.to_csv(path, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL)
    print(f"💾 Saved {len(df)} rows × {len(df.columns)} cols → {path}")

aio_df = _load(AIO_IN)
org_df = _load(ORGANIC_IN)
print(f"Loaded {len(aio_df)} AIO + {len(org_df)} organic rows.")

_save(_enrich(aio_df, score_citations=True),  AIO_OUT)
_save(_enrich(org_df, score_citations=False), ORGANIC_OUT)
print("✅ Step 11 (JAMA) complete — outputs saved to Drive (no PC download).")


In [ ]:
# @title 🧪 STEP 12 — HONCODE Scoring { display-mode: "form" }
# @markdown Run AFTER Step 11 (JAMA). Adds HON_Authority + HON_Complementarity
# @markdown (HON_Total 0-2) for every source. HON_Attribution (duplicated
# @markdown JAMA_Attribution) and HON_Justifiability (not codeable) were dropped;
# @markdown stale copies of those columns are removed from the datasets here.
# @markdown Reads the JAMA outputs by default (so both score sets end up together);
# @markdown falls back to the plain Step 9 files if the JAMA outputs are missing.
# @markdown NaN-robust. Saves to `PROJECT_DIR`:
# @markdown   * `aio_dataset.csv`
# @markdown   * `organic_dataset.csv`
# @markdown
# @markdown CPU-only — no GPU runtime needed.

PROJECT_DIR   = r"C:/Users/SAgno/Documents/AIO_Workspace/LocalRun"  # @param {type:"string"}
OUTPUT_SUBDIR = "Output"               # @param {type:"string"}
AIO_IN      = "aio_dataset.csv"        # @param {type:"string"}
ORGANIC_IN  = "organic_dataset.csv"    # @param {type:"string"}
AIO_OUT     = "aio_dataset.csv"      # @param {type:"string"}
ORGANIC_OUT = "organic_dataset.csv"  # @param {type:"string"}


import sys, os, csv, re
import pandas as pd
from tqdm.auto import tqdm

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)
from health_scorers import HONcodeAnalyzer, _field, SIG_COLS

hon = HONcodeAnalyzer()

from comparison_builder import registered_domain as _reg_dom

# Hosts that are never a cited SOURCE even when they sit off the page's domain
# (patient portals, booking, billing). Mirrors Step 5's _SIG_PORTAL_HOST_RE.
_PORTAL_RE = re.compile(
    r"(login|signin|sign-in|portal|mychart|followmyhealth|patientnotebook|"
    r"appointment|booking|schedul|billpay|bill-pay|payment|careers|"
    r"apps\.apple|play\.google|maps\.google|facebook|twitter|instagram|"
    r"linkedin|youtube|tiktok|pinterest)", re.I)

def _external_citation(row, prefix=""):
    """Fallback attribution signal for rows scraped BEFORE the sig_* columns
    existed: a scraped sub-citation whose registered domain differs from the
    source's own. Only primary sources carry sub-citations."""
    if prefix != "":
        return False
    src_dom = _reg_dom(_field(row.get("url")))
    for j in range(1, 6):
        cu = _field(row.get(f"cite{j}_url"))
        if cu and not _PORTAL_RE.search(cu) and _reg_dom(cu) != src_dom:
            return True
    return False

def _row_signals(row, prefix=""):
    """Pull a source's structured fields + page-quality signals. prefix=''
    for the primary source, 'cite{j}_' for a sub-citation. NaN-robust."""
    g = lambda k: _field(row.get(f"{prefix}{k}"))
    sig = {k: g(k) for k in SIG_COLS}
    return {
        "text":              g("text"),
        "authors":           g("authors"),
        "pub_date":          g("pub_date"),
        "updated_date":      g("updated_date"),
        "disclosures":       g("disclosures"),
        "url":               g("url"),
        "external_citation": _external_citation(row, prefix),
        **sig,
    }

def _has_source(row, prefix=""):
    """A source/citation slot has data only if it has a URL. Empty slots
    (no data collected during the scrape) are left BLANK, not scored 0."""
    return bool(_field(row.get(f"{prefix}url")))

_SCORE_KEYS = list(hon.score().keys())   # sub-scores + Total

def _enrich(df, score_citations):
    records = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="HONcode"):
        out = row.to_dict()
        if _has_source(row, ""):
            out.update(hon.score(**_row_signals(row, "")))
        else:
            for k in _SCORE_KEYS:
                out[k] = None          # no data collected -> blank, not 0
        if score_citations:
            for j in range(1, 6):
                if _has_source(row, f"cite{j}_"):
                    for k, v in hon.score(**_row_signals(row, f"cite{j}_")).items():
                        out[f"cite{j}_{k}"] = v
                else:
                    for k in _SCORE_KEYS:
                        out[f"cite{j}_{k}"] = None   # empty slot -> blank (clears stale 0)
        records.append(out)
    return pd.DataFrame(records)

def _load_or_fallback(name, fallback):
    p = os.path.join(PROJECT_DIR, OUTPUT_SUBDIR, name)
    if os.path.exists(p):
        return pd.read_csv(p)
    fp = os.path.join(PROJECT_DIR, OUTPUT_SUBDIR, fallback)
    print(f"⚠ '{name}' not found — falling back to '{fallback}' (run Step 11 first to include JAMA columns).")
    return pd.read_csv(fp)

def _save(df, name):
    path = os.path.join(PROJECT_DIR, OUTPUT_SUBDIR, name)
    df.to_csv(path, index=False, encoding="utf-8-sig", quoting=csv.QUOTE_ALL)
    print(f"💾 Saved {len(df)} rows × {len(df.columns)} cols → {path}")

aio_df = _load_or_fallback(AIO_IN, "aio_dataset.csv")
org_df = _load_or_fallback(ORGANIC_IN, "organic_dataset.csv")
print(f"Loaded {len(aio_df)} AIO + {len(org_df)} organic rows.")

# Dropped variables: remove stale columns left by earlier runs so they cannot
# linger next to the new scores (primary + cite{j}_ copies).
_DROPPED_RE = re.compile(r"^(cite\d+_)?HON_(Attribution|Justifiability)$")
for _name, _df in (("AIO", aio_df), ("organic", org_df)):
    _stale = [c for c in _df.columns if _DROPPED_RE.match(c)]
    if _stale:
        _df.drop(columns=_stale, inplace=True)
        print(f"  Dropped {len(_stale)} stale column(s) from {_name}: {_stale[:4]}{' ...' if len(_stale) > 4 else ''}")

_save(_enrich(aio_df, score_citations=True),  AIO_OUT)
_save(_enrich(org_df, score_citations=False), ORGANIC_OUT)
print("✅ Step 12 (HONCODE) complete — outputs saved to Drive (no PC download).")


In [ ]:
# @title 📈 STEP 13 — AIO vs Organic Comparison CSVs { display-mode: "form" }
# @markdown Run AFTER Step 9 (and ideally Steps 11–12 so JAMA/HON scores are
# @markdown included). Builds the comparison CSVs in `PROJECT_DIR/Output/`:
# @markdown   • `comparison_sources.csv`   — one row per source, overlap flags
# @markdown   • `comparison_questions.csv` — one row per question (counts only)
# @markdown
# @markdown **The Excel workbook is NOT built here** — it is built by the two
# @markdown scoring notebooks (`Halloumi_Scoring.ipynb` and
# @markdown `GisPy_Scoring.ipynb`), each of which rebuilds it when it finishes.
# @markdown They can be run in **either order**: each adds only its own columns
# @markdown and preserves the other's, so whichever runs last leaves a workbook
# @markdown containing everything computed so far. The logic lives in
# @markdown `comparison_builder.py` (in PROJECT_DIR on Drive).

PROJECT_DIR   = r"C:/Users/SAgno/Documents/AIO_Workspace/LocalRun"  # @param {type:"string"}
OUTPUT_SUBDIR = "Output"                   # @param {type:"string"}
AIO_IN        = "aio_dataset.csv"          # @param {type:"string"}
ORGANIC_IN    = "organic_dataset.csv"      # @param {type:"string"}

import os, sys, importlib

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

import comparison_builder
importlib.reload(comparison_builder)   # pick up an updated module without a restart

summary = comparison_builder.build_comparison_outputs(
    PROJECT_DIR,
    output_subdir=OUTPUT_SUBDIR,
    aio_file=AIO_IN,
    organic_file=ORGANIC_IN,
    include_workbook=False,   # workbook is built by the Halloumi / GisPy notebooks
)

print(f"\n✅ Step 13 complete — comparison CSVs written to {summary['output_dir']}")
print("   Next: run Halloumi_Scoring.ipynb and/or GisPy_Scoring.ipynb "
      "(either order) — each rebuilds the workbook with all scores so far.")
